# Lumpy Collision Spare Parts Forecasting

This notebook uses the same internal and external handoff files as the external data check workbook, then focuses only on `lumpy` collision spare-parts demand.

The goal is to test whether feature engineering plus intermittent-demand models can improve WMAPE for many-zero monthly demand series.

Core setup:

- Target: monthly `demand`
- Scope: collision spare parts where `demand_type == "lumpy"`
- Forecast window: 18 months
- Operational lag: 3 months
- Backtest: rolling-origin windows
- Tracking metric: WMAPE and 3-month rolling WMAPE
- Target bands: below 70 percent, with 50 percent as the stretch goal

Models compared:

1. SBA/Croston intermittent-demand baseline
2. TSB intermittent-demand baseline
3. Hurdle Random Forest regression using engineered internal and external features


## Libraries


In [ ]:
from __future__ import annotations

import importlib.util
import math
import os
import subprocess
import sys
import warnings
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib-cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

required_packages = {
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
}
missing_packages = [
    package_name
    for import_name, package_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        *missing_packages,
    ])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


## Colab Mount

Run this cell in Colab. In VS Code/local Jupyter it will simply continue.


In [ ]:
# Local run: repository data folders are used.

print("Using repository-local data and results folders.")


## Configuration

These paths come from the existing `External_data_check` notebook.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
for candidate in [
    PROJECT_ROOT,
    PROJECT_ROOT.parent,
    PROJECT_ROOT.parent.parent,
    PROJECT_ROOT / "Inchscape Repo",
    PROJECT_ROOT.parent / "Inchscape Repo",
]:
    if (candidate / "data").exists() and (candidate / "results").exists():
        PROJECT_ROOT = candidate.resolve()
        break

DATA_DIRECTORY = PROJECT_ROOT / "data"
EXTERNAL_DATA_DIRECTORY = DATA_DIRECTORY / "external"
PROCESSED_DEMAND_DIRECTORY = DATA_DIRECTORY / "processed" / "all_sku_history"
RESULTS_DIRECTORY = PROJECT_ROOT / "results"

INTERNAL_SALES_PATH = PROCESSED_DEMAND_DIRECTORY / "collision_sales_lumpy.csv"
EXTERNAL_FEATURES_PATH = EXTERNAL_DATA_DIRECTORY / "monthly_external_features.csv"
OUTPUT_DIRECTORY = RESULTS_DIRECTORY / "lumpy_outputs"
OUTPUT_TABLE_DIRECTORY = OUTPUT_DIRECTORY / "tables"
OUTPUT_FIGURE_DIRECTORY = OUTPUT_DIRECTORY / "figures"

for output_directory in [
    OUTPUT_DIRECTORY,
    OUTPUT_TABLE_DIRECTORY,
    OUTPUT_FIGURE_DIRECTORY,
]:
    output_directory.mkdir(
        parents=True,
        exist_ok=True
    )

SKU_COLUMN = "sku_id"
DATE_COLUMN = "date"
TARGET_COLUMN = "demand"
DEMAND_TYPE_COLUMN = "demand_type"
COLLISION_COLUMN = "is_collision"

LUMPY_DEMAND_TYPE = "lumpy"

FORECAST_LAG_MONTHS = 3

# Keep the requested 3-month lag as the benchmark, but test alternatives.
# These are operational lead-time gaps between the last training month and
# the first forecast month. Invalid combinations are skipped automatically
# when history is too short.
FORECAST_LAG_MONTH_OPTIONS = [1, 3, 6]

TEST_MONTHS = 18
RECOMMENDED_TEST_WINDOWS = [9]
BACKTEST_TEST_WINDOWS = [TEST_MONTHS] + RECOMMENDED_TEST_WINDOWS

# Prefer the full 48-month training window. If the loaded history cannot support
# 48 train months + lag months + test months, the split builder reduces the train
# window only as much as needed so the notebook can still run.
PREFERRED_TRAIN_MONTHS = 48
MIN_TRAIN_MONTHS = 24

# The current real lumpy file may only have around 28 monthly periods. Keeping
# an 18-month forecast window and 3-month lag leaves only 7 train months. This
# mode allows a first-pass run, but the split table will flag it as short history.
ALLOW_SHORT_HISTORY_BACKTEST = True
ABSOLUTE_MIN_TRAIN_MONTHS = 6

# External feature mode options:
# - "selected": only the external features listed below
# - "calendar_only": only known-ahead calendar external features
# - "all": all numeric external features
# - "off": no external features
EXTERNAL_FEATURE_MODE = "selected"

SELECTED_EXTERNAL_FEATURE_MATCHES = [
    "min_temperature",
    "max_wind_gust",
    "lag_1yr_national_annual_motorization_rate_context",
    "lag_1yr_national_annual_fatalities_per_10k_veh",
    "working_days",
    "lag_1yr_national_annual_fatalities_per_100_col",
    "weekend_days",
    "days_in_month",
]

BACKTEST_STEP_MONTHS = 3
MAX_BACKTEST_FOLDS = None  # Set to an integer such as 8 for a quicker run.

RANDOM_STATE = 42


# Tuning setup. Each fold tunes only inside the training window. If the training
# window is too short, the notebook uses conservative defaults and labels that
# fold as short-history/no-validation.
TUNING_VALIDATION_MONTHS = 3
MIN_TUNING_FIT_MONTHS = 6

SBA_ALPHA_GRID = [0.05, 0.10, 0.20, 0.30]
TSB_ALPHA_GRID = [0.05, 0.10, 0.20, 0.30]
TSB_BETA_GRID = [0.05, 0.10, 0.20, 0.30]

HURDLE_RF_PARAM_GRID = [
    {
        "n_estimators": 200,
        "classifier_min_samples_leaf": 2,
        "regressor_min_samples_leaf": 2,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 300,
        "classifier_min_samples_leaf": 3,
        "regressor_min_samples_leaf": 2,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 200,
        "classifier_min_samples_leaf": 4,
        "regressor_min_samples_leaf": 3,
        "max_features": 0.75,
    },
]

HURDLE_RF_OCCURRENCE_THRESHOLD_GRID = [0.00, 0.20, 0.35, 0.50, 0.65, 0.80]
HURDLE_RF_DEFAULT_OCCURRENCE_THRESHOLD = 0.50

DIRECT_RF_PARAM_GRID = [
    {
        "n_estimators": 200,
        "regressor_min_samples_leaf": 2,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 300,
        "regressor_min_samples_leaf": 3,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 200,
        "regressor_min_samples_leaf": 4,
        "max_features": 0.75,
    },
]
DIRECT_RF_FEATURE_PROFILES = [
    "rf_history_only",
    "rf_history_calendar",
    "rf_history_calendar_external",
]
DIRECT_RF_DEFAULT_FEATURE_PROFILE = "rf_history_calendar"
DIRECT_RF_FORECAST_FLOOR_GRID = [0.00, 0.25, 0.50, 1.00, 2.00]
DIRECT_RF_DEFAULT_FORECAST_FLOOR = 1.00


# Hurdle RF feature gate. External features are available again, but the model
# can only use them when they improve validation WMAPE inside the training
# window. Short-history/no-validation folds fall back to internal history plus
# calendar features.
HURDLE_RF_FEATURE_PROFILES = [
    "rf_history_only",
    "rf_history_calendar",
    "rf_history_calendar_external",
    "positive_impact_auto",
]
HURDLE_RF_DEFAULT_FEATURE_PROFILE = "rf_history_calendar"
FEATURE_GATE_MIN_WMAPE_IMPROVEMENT_PERCENT = 0.50
FEATURE_GATE_MAX_EXTRA_FEATURES = 12
FEATURE_OVERFIT_WARNING_GAP_PERCENT = 25.0



# Validation-only calibration checks whether each forecast should be dampened or
# lifted before scoring. It is learned inside the training window, never on the
# test window.
CALIBRATION_MULTIPLIER_GRID = [0.0, 0.25, 0.50, 0.75, 1.00, 1.25, 1.50]

# The Benchmark Agent also reports an operational-floor view so zero-calibration
# does not hide the best usable non-zero forecast.
OPERATIONAL_CALIBRATION_MIN_MULTIPLIER = 0.25


# Lumpy-demand decision metrics. These relative weights are not accounting
# costs; they are a planning sensitivity view. Increase stockout weight when
# service level matters more than overstock.
MASE_NAIVE_LAG_MONTHS = 1
INVENTORY_HOLDING_COST_WEIGHT = 1.0
INVENTORY_STOCKOUT_COST_WEIGHT = 5.0

# Descriptor-aware allocation lets the aggregate model assign demand to new SKUs
# that share useful family/subfamily traits with historical demand.
AGGREGATE_ALLOCATION_DESCRIPTOR_COLUMNS = [
    "SUBFAMILY_DESCRIPTION",
    "FAMILY_DESCRIPTION",
    "MATERIAL_DESCRIPTION",
]
AGGREGATE_DESCRIPTOR_MAX_UNIQUE = 50
AGGREGATE_NEW_SKU_EQUAL_WEIGHT = 0.25


# Hybrid allocation experiments. These are focused on the individual SKU-month
# problem: keep the cleaner aggregate monthly total forecast, then decide which
# SKU-month rows should receive that volume.
HYBRID_TOTAL_STRATEGIES = [
    "recent_3m_total",
    "recent_6m_total",
    "same_month_mean_total",
]
HYBRID_GATE_TOP_SHARE_GRID = [0.10, 0.20, 0.35, 0.50, 1.00]
HYBRID_GATE_MIN_PROBABILITY_GRID = [0.00, 0.10, 0.20, 0.35, 0.50]
HYBRID_DEFAULT_TOP_SHARE = 0.50
HYBRID_DEFAULT_MIN_PROBABILITY = 0.10

EMPIRICAL_BAYES_PROBABILITY_PRIOR_GRID = [3.0, 8.0, 16.0]
EMPIRICAL_BAYES_SIZE_PRIOR_GRID = [2.0, 6.0, 12.0]
EMPIRICAL_BAYES_SEASONALITY_WEIGHT_GRID = [0.00, 0.25]
EMPIRICAL_BAYES_DEFAULT_PROBABILITY_PRIOR = 8.0
EMPIRICAL_BAYES_DEFAULT_SIZE_PRIOR = 6.0
EMPIRICAL_BAYES_DEFAULT_SEASONALITY_WEIGHT = 0.25

OCCURRENCE_GATED_ALLOCATION_FEATURE_PROFILES = [
    "rf_history_calendar",
    "rf_history_calendar_external",
]
OCCURRENCE_GATED_ALLOCATION_SCORE_MODES = [
    "rf_expected",
    "rf_probability",
    "rf_probability_eb_size",
]
OCCURRENCE_GATED_ALLOCATION_DEFAULT_SCORE_MODE = "rf_expected"

LIFECYCLE_SIGNAL_NAME_MATCHES = [
    "stockout",
    "stock_out",
    "out_of_stock",
    "availability",
    "available",
    "supersession",
    "superseded",
    "replacement",
    "launch",
    "introduced",
    "introduction",
    "discontinued",
    "inactive",
    "active",
    "lifecycle",
]


## Load Internal And External Data


In [ ]:
def to_month_start(values):
    return pd.to_datetime(values).dt.to_period("M").dt.to_timestamp()


def resolve_input_path(configured_path):
    configured_path = Path(configured_path)

    if configured_path.exists():
        return configured_path

    file_name = configured_path.name
    candidate_paths = [
        configured_path,
        PROJECT_ROOT / "data" / "external" / file_name,
        PROJECT_ROOT / "data" / "processed" / "all_sku_history" / file_name,
        PROJECT_ROOT / "data" / "processed" / "collision_flag_only" / file_name,
        PROJECT_ROOT / "data" / "processed" / file_name,
        Path.cwd() / file_name,
    ]

    workspace_matches = sorted(Path.cwd().rglob(file_name))
    for candidate_path in candidate_paths + workspace_matches:
        if candidate_path.exists():
            print(f"Using local input file for {file_name}: {candidate_path}")
            return candidate_path

    searched_locations = "\n".join(str(path) for path in candidate_paths)
    raise FileNotFoundError(
        f"Could not find {file_name}. Place it in the repo data folders or update the path in the configuration cell.\n"
        f"Original path: {configured_path}\n"
        f"Searched:\n{searched_locations}"
    )


COLLISION_TRUE_VALUES = {"true", "1", "yes", "y"}


def collision_flag_true_mask(data):
    if COLLISION_COLUMN not in data.columns:
        return pd.Series(True, index=data.index)
    return (
        data[COLLISION_COLUMN]
        .astype(str)
        .str.lower()
        .str.strip()
        .isin(COLLISION_TRUE_VALUES)
    )


def build_collision_sku_lookup(sales):
    if "collision_flag_observed" not in sales.columns:
        return pd.DataFrame(
            columns=[
                SKU_COLUMN,
                "first_collision_flag_month",
                "last_collision_flag_month",
                "collision_flagged_rows",
            ]
        )

    flagged_rows = sales.loc[sales["collision_flag_observed"]].copy()
    if flagged_rows.empty:
        return pd.DataFrame(
            columns=[
                SKU_COLUMN,
                "first_collision_flag_month",
                "last_collision_flag_month",
                "collision_flagged_rows",
            ]
        )

    return (
        flagged_rows
        .groupby(SKU_COLUMN, as_index=False)
        .agg(
            first_collision_flag_month=(DATE_COLUMN, "min"),
            last_collision_flag_month=(DATE_COLUMN, "max"),
            collision_flagged_rows=(SKU_COLUMN, "size"),
        )
        .sort_values(["first_collision_flag_month", SKU_COLUMN])
    )


def summarize_collision_sku_filter(sales, collision_sales):
    lookup = build_collision_sku_lookup(sales)

    if lookup.empty:
        return lookup, pd.DataFrame(
            [
                {
                    "collision_flagged_skus": 0,
                    "collision_flagged_rows": 0,
                    "collision_history_rows_kept": len(collision_sales),
                    "pre_first_flag_rows_kept": 0,
                    "first_collision_flag_month": pd.NaT,
                    "earliest_kept_collision_sku_month": collision_sales[DATE_COLUMN].min()
                    if len(collision_sales)
                    else pd.NaT,
                }
            ]
        )

    selected_with_first_flag = collision_sales.merge(
        lookup[[SKU_COLUMN, "first_collision_flag_month"]],
        on=SKU_COLUMN,
        how="left",
    )
    pre_first_flag_rows = selected_with_first_flag[DATE_COLUMN].lt(
        selected_with_first_flag["first_collision_flag_month"]
    ).sum()

    summary = pd.DataFrame(
        [
            {
                "collision_flagged_skus": lookup[SKU_COLUMN].nunique(),
                "collision_flagged_rows": int(sales["collision_flag_observed"].sum()),
                "collision_history_rows_kept": len(collision_sales),
                "pre_first_flag_rows_kept": int(pre_first_flag_rows),
                "first_collision_flag_month": lookup["first_collision_flag_month"].min(),
                "earliest_kept_collision_sku_month": collision_sales[DATE_COLUMN].min(),
            }
        ]
    )
    return lookup, summary


def load_internal_external_data(
    internal_sales_path=INTERNAL_SALES_PATH,
    external_features_path=EXTERNAL_FEATURES_PATH,
):
    internal_sales_path = resolve_input_path(internal_sales_path)
    external_features_path = resolve_input_path(external_features_path)

    print(f"Loading internal sales from: {internal_sales_path}")
    print(f"Loading external features from: {external_features_path}")

    sales = pd.read_csv(internal_sales_path)
    external = pd.read_csv(external_features_path)

    if DATE_COLUMN not in sales.columns and "month" in sales.columns:
        sales[DATE_COLUMN] = sales["month"]

    if DATE_COLUMN not in sales.columns:
        raise ValueError(f"Internal sales must contain a {DATE_COLUMN!r} or 'month' column.")
    if SKU_COLUMN not in sales.columns:
        raise ValueError(f"Internal sales must contain a {SKU_COLUMN!r} column.")

    if COLLISION_COLUMN not in sales.columns:
        if "row_is_collision" in sales.columns:
            sales[COLLISION_COLUMN] = sales["row_is_collision"]
        elif "collision_flag" in sales.columns:
            sales[COLLISION_COLUMN] = sales["collision_flag"]

    if DATE_COLUMN not in external.columns:
        if {"year", "month"}.issubset(external.columns):
            external[DATE_COLUMN] = pd.to_datetime(
                external["year"].astype(int).astype(str)
                + "-"
                + external["month"].astype(int).astype(str).str.zfill(2)
                + "-01"
            )
        else:
            raise ValueError("External features need either date or year/month columns.")

    sales[DATE_COLUMN] = to_month_start(sales[DATE_COLUMN])
    external[DATE_COLUMN] = to_month_start(external[DATE_COLUMN])

    sales[TARGET_COLUMN] = pd.to_numeric(sales[TARGET_COLUMN], errors="coerce").fillna(0)

    if "year" not in sales.columns:
        sales["year"] = sales[DATE_COLUMN].dt.year
    if "month" not in sales.columns:
        sales["month"] = sales[DATE_COLUMN].dt.month

    if COLLISION_COLUMN in sales.columns:
        collision_mask = collision_flag_true_mask(sales)
        collision_skus = sales.loc[collision_mask, SKU_COLUMN].dropna().unique()
        if len(collision_skus) == 0:
            raise ValueError(
                f"{COLLISION_COLUMN!r} exists, but no collision SKUs were flagged true."
            )
        sales["collision_flag_observed"] = collision_mask
        sales["collision_sku_selected"] = sales[SKU_COLUMN].isin(collision_skus)
        collision_sales = sales.loc[sales["collision_sku_selected"]].copy()
        print(
            "Collision SKU filter:",
            f"{len(collision_skus):,} flagged SKUs;",
            f"{len(collision_sales):,} rows kept across full SKU history.",
        )
    else:
        sales["collision_flag_observed"] = True
        sales["collision_sku_selected"] = True
        collision_sales = sales.copy()

    if DEMAND_TYPE_COLUMN not in collision_sales.columns:
        raise ValueError(f"Internal sales must contain {DEMAND_TYPE_COLUMN!r}.")

    collision_sales["demand_type_clean"] = (
        collision_sales[DEMAND_TYPE_COLUMN].astype(str).str.lower().str.strip()
    )

    lumpy_sales = collision_sales.loc[
        collision_sales["demand_type_clean"].eq(LUMPY_DEMAND_TYPE)
    ].copy()

    if lumpy_sales.empty:
        raise ValueError("No lumpy collision rows found.")

    return sales, external, collision_sales, lumpy_sales


sales_raw, external_raw, collision_sales, lumpy_sales_raw = load_internal_external_data()
collision_sku_lookup, collision_filter_summary = summarize_collision_sku_filter(
    sales_raw,
    collision_sales,
)

print("Raw sales rows:", len(sales_raw))
print("Collision SKU-history rows:", len(collision_sales))
print("Raw lumpy rows:", len(lumpy_sales_raw))
print("External monthly rows:", len(external_raw))

display(collision_filter_summary)
display(collision_sku_lookup.head(50))
display(lumpy_sales_raw.head())
display(external_raw.head())


## Demand Type Check

This mirrors the split from the external data check workbook, then narrows the modelling data to lumpy demand.


In [ ]:
demand_type_distribution = (
    collision_sales
    .drop_duplicates([SKU_COLUMN, "demand_type_clean"])
    .groupby("demand_type_clean", as_index=False)
    .agg(sku_count=(SKU_COLUMN, "nunique"))
    .sort_values("sku_count", ascending=False)
)

lumpy_overview = pd.DataFrame(
    [
        {
            "rows": len(lumpy_sales_raw),
            "skus": lumpy_sales_raw[SKU_COLUMN].nunique(),
            "first_month": lumpy_sales_raw[DATE_COLUMN].min(),
            "last_month": lumpy_sales_raw[DATE_COLUMN].max(),
            "zero_month_share": (lumpy_sales_raw[TARGET_COLUMN].fillna(0).eq(0)).mean(),
            "total_demand": lumpy_sales_raw[TARGET_COLUMN].sum(),
            "mean_monthly_row_demand": lumpy_sales_raw[TARGET_COLUMN].mean(),
        }
    ]
)

display(demand_type_distribution)
display(lumpy_overview)


## Complete Monthly SKU Grid

If the source already contains zero-demand months, this mostly keeps the data as-is. If some missing SKU-months represent no demand, this fills them as zero from the SKU's first observed month onward.


In [ ]:
def complete_sku_month_grid(data):
    frames = []
    max_date = data[DATE_COLUMN].max()

    for sku, sku_data in data.groupby(SKU_COLUMN, sort=False):
        sku_data = sku_data.sort_values(DATE_COLUMN)
        sku_dates = pd.date_range(
            sku_data[DATE_COLUMN].min(),
            max_date,
            freq="MS",
        )
        sku_grid = pd.DataFrame(
            {
                SKU_COLUMN: sku,
                DATE_COLUMN: sku_dates,
            }
        )
        frames.append(sku_grid)

    grid = pd.concat(frames, ignore_index=True)
    completed = grid.merge(
        data,
        on=[SKU_COLUMN, DATE_COLUMN],
        how="left",
        suffixes=("", "_source"),
    )

    completed[TARGET_COLUMN] = completed[TARGET_COLUMN].fillna(0)

    if DEMAND_TYPE_COLUMN not in completed.columns:
        completed[DEMAND_TYPE_COLUMN] = LUMPY_DEMAND_TYPE
    else:
        completed[DEMAND_TYPE_COLUMN] = completed[DEMAND_TYPE_COLUMN].fillna(LUMPY_DEMAND_TYPE)

    if "demand_type_clean" not in completed.columns:
        completed["demand_type_clean"] = LUMPY_DEMAND_TYPE
    else:
        completed["demand_type_clean"] = completed["demand_type_clean"].fillna(LUMPY_DEMAND_TYPE)

    if "collision_sku_selected" in completed.columns:
        completed["collision_sku_selected"] = completed["collision_sku_selected"].fillna(True)
    if "collision_flag_observed" in completed.columns:
        completed["collision_flag_observed"] = completed["collision_flag_observed"].fillna(False)
    if COLLISION_COLUMN in completed.columns:
        completed[COLLISION_COLUMN] = completed[COLLISION_COLUMN].fillna(
            completed["collision_sku_selected"]
            if "collision_sku_selected" in completed.columns
            else True
        )

    completed["year"] = completed[DATE_COLUMN].dt.year
    completed["month"] = completed[DATE_COLUMN].dt.month

    # Carry mostly static descriptors within SKU when they exist.
    descriptor_columns = [
        column
        for column in completed.columns
        if column not in [TARGET_COLUMN, DATE_COLUMN, "year", "month"]
        and completed[column].dtype == "object"
    ]

    for column in descriptor_columns:
        completed[column] = (
            completed
            .sort_values([SKU_COLUMN, DATE_COLUMN])
            .groupby(SKU_COLUMN)[column]
            .transform(lambda series: series.ffill().bfill())
        )

    return completed.sort_values([SKU_COLUMN, DATE_COLUMN]).reset_index(drop=True)


lumpy_sales = complete_sku_month_grid(lumpy_sales_raw)

grid_summary = pd.DataFrame(
    [
        {
            "rows_before_grid": len(lumpy_sales_raw),
            "rows_after_grid": len(lumpy_sales),
            "skus": lumpy_sales[SKU_COLUMN].nunique(),
            "first_month": lumpy_sales[DATE_COLUMN].min(),
            "last_month": lumpy_sales[DATE_COLUMN].max(),
            "zero_month_share_after_grid": lumpy_sales[TARGET_COLUMN].eq(0).mean(),
        }
    ]
)

display(grid_summary)


## Feature Engineering

This run uses a reduced external feature set from the previous feature-importance view, instead of passing every external signal into the model.

Current external mode: `selected`.

Selected external source features:

- `min_temperature`
- `max_wind_gust`
- `lag_1yr_national_annual_motorization_rate_context`
- `lag_1yr_national_annual_fatalities_per_10k_veh...`
- `working_days`
- `lag_1yr_national_annual_fatalities_per_100_col...`
- `weekend_days`
- `days_in_month`

Demand history is still shifted by the 3-month operating lag. Calendar and already-lagged annual context features are treated as known-ahead. Weather-style features are only exposed as lagged or rolling historical signals.


In [ ]:
if "to_month_start" not in globals():
    def to_month_start(values):
        return pd.to_datetime(values).dt.to_period("M").dt.to_timestamp()


def add_calendar_features(data):
    features = data.copy()
    month = features[DATE_COLUMN].dt.month.astype(float)

    features["calendar_month"] = features[DATE_COLUMN].dt.month
    features["calendar_quarter"] = features[DATE_COLUMN].dt.quarter
    features["calendar_year"] = features[DATE_COLUMN].dt.year
    features["calendar_month_sin"] = np.sin(2 * np.pi * month / 12)
    features["calendar_month_cos"] = np.cos(2 * np.pi * month / 12)
    features["calendar_is_q1"] = features["calendar_quarter"].eq(1).astype(int)
    features["calendar_is_q4"] = features["calendar_quarter"].eq(4).astype(int)
    features["calendar_month_index"] = (
        (features[DATE_COLUMN].dt.year - features[DATE_COLUMN].dt.year.min()) * 12
        + features[DATE_COLUMN].dt.month
    )

    return features


def external_known_ahead_columns(external_numeric_columns):
    known_keywords = [
        "holiday",
        "working",
        "business",
        "calendar",
        "weekend",
        "weekday",
        "days_in_month",
        "lag_1yr",
    ]
    return [
        column
        for column in external_numeric_columns
        if any(keyword in column.lower() for keyword in known_keywords)
    ]


def matches_selected_external_feature(column):
    column_lower = column.lower()
    for feature_match in SELECTED_EXTERNAL_FEATURE_MATCHES:
        match_lower = feature_match.lower()
        if column_lower == match_lower or column_lower.startswith(match_lower):
            return True
    return False


def select_external_source_columns(external_numeric_columns):
    if EXTERNAL_FEATURE_MODE == "off":
        return []
    if EXTERNAL_FEATURE_MODE == "all":
        return list(external_numeric_columns)
    if EXTERNAL_FEATURE_MODE == "calendar_only":
        return external_known_ahead_columns(external_numeric_columns)
    if EXTERNAL_FEATURE_MODE == "selected":
        return [
            column
            for column in external_numeric_columns
            if matches_selected_external_feature(column)
        ]
    raise ValueError(
        "EXTERNAL_FEATURE_MODE must be one of: selected, calendar_only, all, off"
    )


def prepare_external_feature_frame(external, lag_months=FORECAST_LAG_MONTHS):
    external = external.copy()
    external[DATE_COLUMN] = to_month_start(external[DATE_COLUMN])

    numeric_columns = [
        column
        for column in external.select_dtypes(include="number").columns
        if column not in ["year", "month"]
    ]
    selected_numeric_columns = select_external_source_columns(numeric_columns)
    known_ahead_columns = external_known_ahead_columns(selected_numeric_columns)
    historical_only_columns = [
        column for column in selected_numeric_columns
        if column not in known_ahead_columns
    ]

    if not selected_numeric_columns:
        print("No external source columns selected for this run.")
        return external[[DATE_COLUMN]].drop_duplicates(), [], []

    missing_selected_matches = [
        feature_match
        for feature_match in SELECTED_EXTERNAL_FEATURE_MATCHES
        if not any(
            column.lower() == feature_match.lower()
            or column.lower().startswith(feature_match.lower())
            for column in numeric_columns
        )
    ]

    if EXTERNAL_FEATURE_MODE == "selected" and missing_selected_matches:
        print("Selected external matches not found in source table:", missing_selected_matches)

    monthly_external = (
        external[[DATE_COLUMN] + selected_numeric_columns]
        .groupby(DATE_COLUMN, as_index=False)
        .mean(numeric_only=True)
        .sort_values(DATE_COLUMN)
    )

    engineered = monthly_external[[DATE_COLUMN]].copy()
    engineered_feature_columns = []

    for column in historical_only_columns:
        lagged_name = f"external_lag_{lag_months}__{column}"
        engineered[lagged_name] = monthly_external[column].shift(lag_months)
        engineered_feature_columns.append(lagged_name)

        for window in [3, 6, 12]:
            roll_name = f"external_roll_mean_{window}_lag_{lag_months}__{column}"
            engineered[roll_name] = (
                monthly_external[column]
                .shift(lag_months)
                .rolling(window, min_periods=1)
                .mean()
            )
            engineered_feature_columns.append(roll_name)

    for column in known_ahead_columns:
        known_name = f"external_known__{column}"
        engineered[known_name] = monthly_external[column]
        engineered_feature_columns.append(known_name)

    return engineered, engineered_feature_columns, selected_numeric_columns


def build_feature_table(lumpy_sales, external):
    features = add_calendar_features(lumpy_sales)

    external_features, external_feature_columns, selected_external_source_columns = prepare_external_feature_frame(external)

    features = features.merge(
        external_features,
        on=DATE_COLUMN,
        how="left",
    )

    return features, external_feature_columns, selected_external_source_columns


lumpy_model_data, external_engineered_feature_columns, selected_external_source_columns = build_feature_table(
    lumpy_sales=lumpy_sales,
    external=external_raw,
)

external_selection_summary = pd.DataFrame(
    {
        "selected_external_source_column": selected_external_source_columns,
        "known_ahead": [
            column in external_known_ahead_columns(selected_external_source_columns)
            for column in selected_external_source_columns
        ],
    }
)

engineered_feature_summary = pd.DataFrame(
    [
        {
            "rows": len(lumpy_model_data),
            "skus": lumpy_model_data[SKU_COLUMN].nunique(),
            "columns": len(lumpy_model_data.columns),
            "global_demand_lag_features": 0,
            "external_feature_mode": EXTERNAL_FEATURE_MODE,
            "selected_external_source_features": len(selected_external_source_columns),
            "engineered_external_features": len(external_engineered_feature_columns),
            "first_month": lumpy_model_data[DATE_COLUMN].min(),
            "last_month": lumpy_model_data[DATE_COLUMN].max(),
        }
    ]
)

display(engineered_feature_summary)
display(external_selection_summary)
display(lumpy_model_data.head())


## Metrics And Backtest Splits

The required run remains `18` forecast months with a `3` month lag. The notebook now also runs a diagnostic `9` month window so we can compare whether a shorter recommendation horizon is materially cleaner.


In [ ]:
def wmape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denominator = np.sum(np.abs(y_true))
    if denominator == 0:
        return np.nan
    return np.sum(np.abs(y_true - y_pred)) / denominator


def month_offset(timestamp, months):
    return pd.Timestamp(timestamp) + pd.DateOffset(months=months)


def resolve_train_window(data, test_months, lag_months=FORECAST_LAG_MONTHS):
    unique_months = np.sort(data[DATE_COLUMN].dropna().unique())
    available_months = len(unique_months)
    required_preferred = PREFERRED_TRAIN_MONTHS + lag_months + test_months
    required_minimum = MIN_TRAIN_MONTHS + lag_months + test_months

    if available_months >= required_preferred:
        return {
            "train_months": PREFERRED_TRAIN_MONTHS,
            "available_months": available_months,
            "history_mode": "preferred_48_month_train",
            "short_history_warning": "",
        }

    reduced_train_months = available_months - lag_months - test_months

    if reduced_train_months >= MIN_TRAIN_MONTHS:
        print(
            f"Using {reduced_train_months} training months instead of "
            f"{PREFERRED_TRAIN_MONTHS}; available history is {available_months} months "
            f"for the {test_months}-month window."
        )
        return {
            "train_months": int(reduced_train_months),
            "available_months": available_months,
            "history_mode": "reduced_train_window",
            "short_history_warning": "",
        }

    if ALLOW_SHORT_HISTORY_BACKTEST and reduced_train_months >= ABSOLUTE_MIN_TRAIN_MONTHS:
        warning = (
            f"Short-history first pass: only {available_months} months are available. "
            f"Keeping {test_months} test months and {lag_months} lag months leaves "
            f"{reduced_train_months} train months. Treat results as directional only."
        )
        print(warning)
        return {
            "train_months": int(reduced_train_months),
            "available_months": available_months,
            "history_mode": "short_history_first_pass",
            "short_history_warning": warning,
        }

    raise ValueError(
        "Not enough monthly history for the requested backtest. "
        f"Preferred setup needs {required_preferred} months; minimum setup needs {required_minimum} months; "
        f"found {available_months}. With a {test_months}-month test and {lag_months}-month lag, "
        f"only {reduced_train_months} train months are available."
    )


def make_backtest_splits(data, test_months=TEST_MONTHS, lag_months=FORECAST_LAG_MONTHS, window_label=None):
    if window_label is None:
        window_label = f"{test_months}_month_window"

    train_window = resolve_train_window(data, test_months=test_months, lag_months=lag_months)
    train_months = train_window["train_months"]
    min_month = data[DATE_COLUMN].min()
    test_end = data[DATE_COLUMN].max()
    splits = []

    while True:
        test_start = month_offset(test_end, -(test_months - 1))
        if lag_months > 0:
            gap_end = month_offset(test_start, -1)
            gap_start = month_offset(gap_end, -(lag_months - 1))
            train_end = month_offset(gap_start, -1)
        else:
            gap_start = pd.NaT
            gap_end = pd.NaT
            train_end = month_offset(test_start, -1)
        train_start = month_offset(train_end, -(train_months - 1))

        if train_start < min_month:
            break

        splits.append(
            {
                "fold_id": len(splits) + 1,
                "window_label": window_label,
                "window_months": test_months,
                "train_start": train_start,
                "train_end": train_end,
                "gap_start": gap_start,
                "gap_end": gap_end,
                "test_start": test_start,
                "test_end": test_end,
                "train_months": train_months,
                "gap_months": lag_months,
                "forecast_lag_months": lag_months,
                "test_months": test_months,
                "available_months": train_window["available_months"],
                "history_mode": train_window["history_mode"],
                "short_history_warning": train_window["short_history_warning"],
            }
        )

        if MAX_BACKTEST_FOLDS is not None and len(splits) >= MAX_BACKTEST_FOLDS:
            break

        test_end = month_offset(test_end, -BACKTEST_STEP_MONTHS)

    if not splits:
        raise ValueError(f"No valid backtest folds could be created for {window_label}.")

    splits = list(reversed(splits))
    for fold_number, split in enumerate(splits, start=1):
        split["fold_id"] = fold_number

    return pd.DataFrame(splits)


split_frames = []
lag_options = list(dict.fromkeys(FORECAST_LAG_MONTH_OPTIONS or [FORECAST_LAG_MONTHS]))

for test_window in BACKTEST_TEST_WINDOWS:
    if test_window == TEST_MONTHS:
        base_label = f"required_{test_window}_month"
    else:
        base_label = f"diagnostic_{test_window}_month"

    for lag_months in lag_options:
        label = f"{base_label}_lag_{lag_months}m"
        try:
            split_frames.append(
                make_backtest_splits(
                    lumpy_model_data,
                    test_months=test_window,
                    lag_months=lag_months,
                    window_label=label,
                )
            )
        except ValueError as error:
            print(f"Skipping {label}: {error}")

if not split_frames:
    raise ValueError("No valid backtest folds could be created for any configured lag/window combination.")

backtest_splits = pd.concat(split_frames, ignore_index=True)
backtest_splits["global_fold_id"] = np.arange(1, len(backtest_splits) + 1)

display(backtest_splits)


## Feature Columns For Regression

The hurdle regression uses engineered demand history, calendar features, lagged/known external features, and SKU-level statistics calculated inside each training fold only.


In [ ]:
LEAKAGE_COLUMNS = {
    TARGET_COLUMN,
    "total_demand",
    "mean_monthly_demand",
    "max_monthly_demand",
    "months_with_demand",
    "zero_demand_months",
    "zero_month_share",
    "cumulative_demand_share",
    "cumulative_sku_share",
    "average_demand_interval",
    "squared_coefficient_of_variation",
    "REVENUE",
    "COST",
    "UNIT_PRICE",
    "TOTAL_PROFIT",
    "UNIT_COST",
    "STOCK_END_MONTH",
    "value",
}

RF_HISTORY_WINDOWS = [3, 6]

ALLOWED_FEATURE_PREFIXES = (
    "calendar_",
    "external_",
    "rf_",
)


def summarize_known_history(values, default_months_since_positive=FORECAST_LAG_MONTHS):
    values = np.asarray(values, dtype=float)
    values = np.clip(values, 0, None)
    positive_values = values[values > 0]

    if len(values) == 0:
        summary = {
            "rf_known_months": 0,
            "rf_known_total_demand": 0.0,
            "rf_known_mean_demand": 0.0,
            "rf_known_positive_rate": 0.0,
            "rf_known_zero_share": 1.0,
            "rf_known_mean_positive_demand": 0.0,
            "rf_last_known_demand": 0.0,
            "rf_last_positive_demand": 0.0,
            "rf_months_since_positive": default_months_since_positive,
        }
    else:
        positive_positions = np.flatnonzero(values > 0)
        if len(positive_positions):
            months_since_positive = len(values) - 1 - positive_positions[-1]
            last_positive = values[positive_positions[-1]]
        else:
            months_since_positive = len(values)
            last_positive = 0.0

        summary = {
            "rf_known_months": len(values),
            "rf_known_total_demand": float(values.sum()),
            "rf_known_mean_demand": float(values.mean()),
            "rf_known_positive_rate": float((values > 0).mean()),
            "rf_known_zero_share": float((values == 0).mean()),
            "rf_known_mean_positive_demand": float(positive_values.mean()) if len(positive_values) else 0.0,
            "rf_last_known_demand": float(values[-1]),
            "rf_last_positive_demand": float(last_positive),
            "rf_months_since_positive": int(months_since_positive),
        }

    for window in RF_HISTORY_WINDOWS:
        recent = values[-window:] if len(values) else values
        summary[f"rf_recent_{window}m_mean"] = float(recent.mean()) if len(recent) else 0.0
        summary[f"rf_recent_{window}m_total"] = float(recent.sum()) if len(recent) else 0.0
        summary[f"rf_recent_{window}m_positive_months"] = int((recent > 0).sum()) if len(recent) else 0
        summary[f"rf_recent_{window}m_zero_share"] = float((recent == 0).mean()) if len(recent) else 1.0

    return summary


def build_train_history_features(train, lag_months=FORECAST_LAG_MONTHS):
    feature_frames = []

    for sku, sku_train in train.sort_values([SKU_COLUMN, DATE_COLUMN]).groupby(SKU_COLUMN, sort=False):
        values = sku_train[TARGET_COLUMN].astype(float).to_numpy()
        rows = []

        for position, row_index in enumerate(sku_train.index):
            known_end_position = position - lag_months + 1
            known_values = values[:max(known_end_position, 0)]
            summary = summarize_known_history(
                known_values,
                default_months_since_positive=lag_months,
            )
            summary[SKU_COLUMN] = sku
            summary[DATE_COLUMN] = sku_train.loc[row_index, DATE_COLUMN]
            rows.append(summary)

        feature_frames.append(pd.DataFrame(rows, index=sku_train.index))

    if not feature_frames:
        return pd.DataFrame(index=train.index)

    return pd.concat(feature_frames).sort_index()


def build_test_history_features(train, test, lag_months=FORECAST_LAG_MONTHS):
    history_by_sku = {
        sku: sku_train.sort_values(DATE_COLUMN)[TARGET_COLUMN].astype(float).to_numpy()
        for sku, sku_train in train.groupby(SKU_COLUMN, sort=False)
    }
    train_end_by_sku = (
        train.groupby(SKU_COLUMN)[DATE_COLUMN].max()
        if len(train)
        else pd.Series(dtype="datetime64[ns]")
    )
    fallback_history = train.sort_values(DATE_COLUMN)[TARGET_COLUMN].astype(float).to_numpy()
    fallback_train_end = pd.to_datetime(train[DATE_COLUMN]).max() if len(train) else pd.NaT
    rows = []

    for row in test[[SKU_COLUMN, DATE_COLUMN]].itertuples(index=True):
        known_values = history_by_sku.get(row.sku_id, fallback_history)
        summary = summarize_known_history(
            known_values,
            default_months_since_positive=lag_months,
        )
        train_end_date = train_end_by_sku.get(row.sku_id, fallback_train_end)
        if pd.notna(train_end_date) and pd.notna(row.date):
            elapsed_months = max(
                0,
                (pd.Timestamp(row.date).year - pd.Timestamp(train_end_date).year) * 12
                + (pd.Timestamp(row.date).month - pd.Timestamp(train_end_date).month),
            )
            summary["rf_months_since_positive"] = int(
                summary["rf_months_since_positive"] + elapsed_months
            )
        summary[SKU_COLUMN] = row.sku_id
        summary[DATE_COLUMN] = row.date
        rows.append((row.Index, summary))

    if not rows:
        return pd.DataFrame(index=test.index)

    indexes, summaries = zip(*rows)
    return pd.DataFrame(list(summaries), index=indexes).sort_index()


def add_train_only_sku_features(train, test, lag_months=FORECAST_LAG_MONTHS):
    train = train.copy()
    test = test.copy()

    train_history_features = build_train_history_features(train, lag_months=lag_months)
    test_history_features = build_test_history_features(train, test, lag_months=lag_months)

    history_feature_columns = [
        column
        for column in train_history_features.columns
        if column not in [SKU_COLUMN, DATE_COLUMN]
    ]

    train = train.join(train_history_features[history_feature_columns])
    test = test.join(test_history_features[history_feature_columns])

    for frame in [train, test]:
        for column in history_feature_columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce").fillna(0.0)

    return train, test, history_feature_columns


def get_hurdle_feature_columns(data):
    numeric_columns = data.select_dtypes(include="number").columns.tolist()
    feature_columns = []

    for column in numeric_columns:
        if column in LEAKAGE_COLUMNS:
            continue
        if column in ["year", "month"]:
            continue
        if column.endswith("_source"):
            continue
        if column.startswith("demand_lag_") or column.startswith("demand_roll_"):
            continue
        if column.startswith("demand_zero_share_") or column.startswith("demand_positive_months_"):
            continue
        if column.startswith("months_since_positive_"):
            continue
        if any(column.startswith(prefix) for prefix in ALLOWED_FEATURE_PREFIXES):
            feature_columns.append(column)

    return sorted(set(feature_columns))


preview_train = lumpy_model_data.loc[
    (lumpy_model_data[DATE_COLUMN] >= backtest_splits.iloc[-1]["train_start"])
    & (lumpy_model_data[DATE_COLUMN] <= backtest_splits.iloc[-1]["train_end"])
].copy()

preview_test = lumpy_model_data.loc[
    (lumpy_model_data[DATE_COLUMN] >= backtest_splits.iloc[-1]["test_start"])
    & (lumpy_model_data[DATE_COLUMN] <= backtest_splits.iloc[-1]["test_end"])
].copy()

preview_lag_months = int(backtest_splits.iloc[-1].get("gap_months", FORECAST_LAG_MONTHS))
preview_train, preview_test, preview_rf_history_columns = add_train_only_sku_features(
    preview_train,
    preview_test,
    lag_months=preview_lag_months,
)

preview_feature_columns = get_hurdle_feature_columns(preview_train)

feature_inventory = pd.DataFrame(
    {
        "feature": preview_feature_columns,
        "is_external_feature": [
            feature.startswith("external_") for feature in preview_feature_columns
        ],
        "is_sku_train_stat": [
            feature.startswith("sku_train_") for feature in preview_feature_columns
        ],
        "is_rf_internal_history_feature": [
            feature.startswith("rf_") for feature in preview_feature_columns
        ],
        "is_global_demand_lag_feature": [
            feature.startswith("demand_lag_")
            or feature.startswith("demand_roll_")
            or feature.startswith("demand_zero_share_")
            or feature.startswith("demand_positive_months_")
            or feature.startswith("months_since_positive_")
            for feature in preview_feature_columns
        ],
        "non_null_share_preview_train": [
            preview_train[feature].notna().mean() for feature in preview_feature_columns
        ],
    }
)

display(feature_inventory)


## Model 1: SBA/Croston

SBA is a bias-adjusted Croston method. It is a useful baseline for intermittent and lumpy demand because it models demand size and demand interval separately.


In [ ]:
def croston_sba_level(y, alpha=0.1):
    y = np.asarray(y, dtype=float)
    y = np.clip(y, 0, None)
    positive_positions = np.flatnonzero(y > 0)

    if len(positive_positions) == 0:
        return 0.0

    first_positive_position = positive_positions[0]
    demand_size = y[first_positive_position]
    interval = max(first_positive_position + 1, 1)
    periods_since_positive = 1

    for value in y[first_positive_position + 1:]:
        if value > 0:
            demand_size = demand_size + alpha * (value - demand_size)
            interval = interval + alpha * (periods_since_positive - interval)
            periods_since_positive = 1
        else:
            periods_since_positive += 1

    forecast = (1 - alpha / 2) * demand_size / max(interval, 1e-9)
    return float(max(forecast, 0))


## Model 2: TSB

TSB updates the probability of demand every period and the demand size when a non-zero month occurs. This can react better when an item is becoming inactive.


In [ ]:
def tsb_level(y, alpha=0.2, beta=0.1):
    y = np.asarray(y, dtype=float)
    y = np.clip(y, 0, None)
    positive = y[y > 0]

    if len(y) == 0 or len(positive) == 0:
        return 0.0

    probability = max(np.mean(y > 0), 1e-6)
    demand_size = positive[0]

    for value in y:
        occurred = 1.0 if value > 0 else 0.0
        probability = probability + alpha * (occurred - probability)
        if value > 0:
            demand_size = demand_size + beta * (value - demand_size)

    forecast = probability * demand_size
    return float(max(forecast, 0))


## Model 3: Tuned Hurdle Random Forest Regression

The hurdle model treats lumpy demand as two problems:

- Will demand occur this month?
- If demand occurs, how large will it be?

Expected demand is `probability_of_demand * predicted_positive_demand`.

The Random Forest hyperparameters are tuned inside each fold when there is enough training history.


In [ ]:
def make_median_imputer():
    try:
        return SimpleImputer(strategy="median", keep_empty_features=True)
    except TypeError:
        return SimpleImputer(strategy="median")


class HurdleRandomForest:
    def __init__(
        self,
        random_state=RANDOM_STATE,
        n_estimators=300,
        classifier_min_samples_leaf=3,
        regressor_min_samples_leaf=2,
        max_features="sqrt",
        occurrence_threshold=0.0,
    ):
        self.random_state = random_state
        self.n_estimators = n_estimators
        self.classifier_min_samples_leaf = classifier_min_samples_leaf
        self.regressor_min_samples_leaf = regressor_min_samples_leaf
        self.max_features = max_features
        self.occurrence_threshold = occurrence_threshold
        self.imputer = make_median_imputer()
        self.classifier = None
        self.regressor = None
        self.constant_probability = None
        self.constant_amount = None
        self.feature_columns = None
        self.training_auc = np.nan

    def fit(self, train, feature_columns):
        self.feature_columns = feature_columns
        X = train[feature_columns]
        y = train[TARGET_COLUMN].astype(float)
        occurrence = y.gt(0).astype(int)

        X_imputed = self.imputer.fit_transform(X)

        if occurrence.nunique() < 2:
            self.constant_probability = float(occurrence.mean())
        else:
            self.classifier = RandomForestClassifier(
                n_estimators=self.n_estimators,
                min_samples_leaf=self.classifier_min_samples_leaf,
                max_features=self.max_features,
                class_weight="balanced_subsample",
                random_state=self.random_state,
                n_jobs=-1,
            )
            self.classifier.fit(X_imputed, occurrence)
            try:
                self.training_auc = roc_auc_score(
                    occurrence,
                    self.classifier.predict_proba(X_imputed)[:, 1],
                )
            except ValueError:
                self.training_auc = np.nan

        positive_mask = y.gt(0)
        positive_y = y.loc[positive_mask]

        if len(positive_y) < 20:
            self.constant_amount = float(positive_y.median()) if len(positive_y) else 0.0
        else:
            self.regressor = RandomForestRegressor(
                n_estimators=self.n_estimators,
                min_samples_leaf=self.regressor_min_samples_leaf,
                max_features=self.max_features,
                random_state=self.random_state,
                n_jobs=-1,
            )
            self.regressor.fit(
                X_imputed[positive_mask.to_numpy()],
                np.log1p(positive_y),
            )

        if self.constant_probability is None and self.classifier is None:
            self.constant_probability = float(occurrence.mean())

        if self.constant_amount is None and self.regressor is None:
            self.constant_amount = float(positive_y.median()) if len(positive_y) else 0.0

        return self

    def predict_components(self, data):
        X = data[self.feature_columns]
        X_imputed = self.imputer.transform(X)

        if self.classifier is not None:
            probability = self.classifier.predict_proba(X_imputed)[:, 1]
        else:
            probability = np.repeat(self.constant_probability, len(data))

        if self.regressor is not None:
            amount = np.expm1(self.regressor.predict(X_imputed))
        else:
            amount = np.repeat(self.constant_amount, len(data))

        return np.clip(probability, 0, 1), np.clip(amount, 0, None)

    def predict(self, data):
        probability, amount = self.predict_components(data)
        forecast = probability * amount
        if self.occurrence_threshold > 0:
            forecast = np.where(probability >= self.occurrence_threshold, forecast, 0.0)
        return np.clip(forecast, 0, None)


class DirectDemandRandomForest:
    def __init__(
        self,
        random_state=RANDOM_STATE,
        n_estimators=300,
        regressor_min_samples_leaf=3,
        max_features="sqrt",
        forecast_floor=0.0,
    ):
        self.random_state = random_state
        self.n_estimators = n_estimators
        self.regressor_min_samples_leaf = regressor_min_samples_leaf
        self.max_features = max_features
        self.forecast_floor = forecast_floor
        self.imputer = make_median_imputer()
        self.regressor = None
        self.constant_amount = None
        self.feature_columns = None
        self.training_auc = np.nan

    def fit(self, train, feature_columns):
        self.feature_columns = feature_columns
        X = train[feature_columns]
        y = train[TARGET_COLUMN].astype(float).clip(lower=0.0)
        X_imputed = self.imputer.fit_transform(X)

        if len(y) < 20 or y.sum() <= 0:
            self.constant_amount = float(y.mean()) if len(y) else 0.0
        else:
            self.regressor = RandomForestRegressor(
                n_estimators=self.n_estimators,
                min_samples_leaf=self.regressor_min_samples_leaf,
                max_features=self.max_features,
                random_state=self.random_state,
                n_jobs=-1,
            )
            self.regressor.fit(X_imputed, np.log1p(y))

        if self.constant_amount is None and self.regressor is None:
            self.constant_amount = float(y.mean()) if len(y) else 0.0

        return self

    def predict(self, data):
        X = data[self.feature_columns]
        X_imputed = self.imputer.transform(X)

        if self.regressor is not None:
            forecast = np.expm1(self.regressor.predict(X_imputed))
        else:
            forecast = np.repeat(self.constant_amount, len(data))

        forecast = np.clip(forecast, 0, None)
        if self.forecast_floor > 0:
            forecast = np.where(forecast >= self.forecast_floor, forecast, 0.0)
        return forecast


## Tuned Backtest Runner

Each fold tunes SBA, TSB, and the hurdle Random Forest using only the training window.

For very short folds, especially the required 18-month window with only around 7 train months, the notebook uses conservative defaults and marks the tuning status as `short_history_no_validation`.

SKU segments are also assigned from train-window behaviour only, then carried onto the test forecasts for segment-level scoring.


In [ ]:
def train_test_for_split(data, split):
    train = data.loc[
        (data[DATE_COLUMN] >= split["train_start"])
        & (data[DATE_COLUMN] <= split["train_end"])
    ].copy()

    test = data.loc[
        (data[DATE_COLUMN] >= split["test_start"])
        & (data[DATE_COLUMN] <= split["test_end"])
    ].copy()

    return train, test


def create_temporal_tuning_split(train):
    train = train.copy()
    months = np.sort(train[DATE_COLUMN].dropna().unique())

    if len(months) < MIN_TUNING_FIT_MONTHS + TUNING_VALIDATION_MONTHS:
        return train, pd.DataFrame(), "short_history_no_validation"

    validation_months = months[-TUNING_VALIDATION_MONTHS:]
    validation_start = pd.Timestamp(validation_months[0])

    fit_train = train.loc[train[DATE_COLUMN] < validation_start].copy()
    validation = train.loc[train[DATE_COLUMN] >= validation_start].copy()

    if fit_train[DATE_COLUMN].nunique() < MIN_TUNING_FIT_MONTHS:
        return train, pd.DataFrame(), "short_history_no_validation"

    if validation[TARGET_COLUMN].sum() == 0:
        return fit_train, validation, "validation_zero_actual"

    return fit_train, validation, "validation"


def params_to_label(params):
    if not params:
        return "default"

    def format_value(value):
        if isinstance(value, (list, tuple, set)):
            values = list(value)
            preview = ", ".join(map(str, values[:5]))
            suffix = "" if len(values) <= 5 else f", +{len(values) - 5} more"
            return f"{len(values)} items [{preview}{suffix}]"
        return value

    return ", ".join(f"{key}={format_value(value)}" for key, value in params.items())


def zero_level(history):
    return 0.0


def sku_train_mean_level(history):
    history = np.asarray(history, dtype=float)
    if len(history) == 0:
        return 0.0
    return float(max(np.nanmean(history), 0.0))


def recent_mean_level(history, recent_months=6):
    history = np.asarray(history, dtype=float)
    if len(history) == 0:
        return 0.0
    recent = history[-recent_months:]
    return float(max(np.nanmean(recent), 0.0))


def last_nonzero_level(history):
    history = np.asarray(history, dtype=float)
    positives = history[history > 0]
    if len(positives) == 0:
        return 0.0
    return float(max(positives[-1], 0.0))


def forecast_constant_by_sku(train, test, level_function, model_name, level_kwargs=None):
    level_kwargs = level_kwargs or {}
    forecast_rows = []

    for sku, test_sku in test.groupby(SKU_COLUMN, sort=False):
        history = (
            train.loc[train[SKU_COLUMN].eq(sku)]
            .sort_values(DATE_COLUMN)[TARGET_COLUMN]
            .to_numpy()
        )
        level = level_function(history, **level_kwargs)

        sku_forecast = test_sku[[SKU_COLUMN, DATE_COLUMN, TARGET_COLUMN]].copy()
        sku_forecast["forecast"] = level
        sku_forecast["model"] = model_name
        forecast_rows.append(sku_forecast)

    if not forecast_rows:
        return pd.DataFrame(columns=[SKU_COLUMN, DATE_COLUMN, TARGET_COLUMN, "forecast", "model"])

    return pd.concat(forecast_rows, ignore_index=True)


def score_candidate_forecast(candidate_forecast):
    score = wmape(candidate_forecast[TARGET_COLUMN], candidate_forecast["forecast"])
    if pd.isna(score):
        return np.inf
    return score


def tune_sba(train):
    fit_train, validation, tuning_status = create_temporal_tuning_split(train)
    default_alpha = 0.10 if 0.10 in SBA_ALPHA_GRID else SBA_ALPHA_GRID[0]

    if validation.empty or tuning_status != "validation":
        return {
            "model": "SBA Croston",
            "params": {"alpha": default_alpha},
            "tuning_wmape_percent": np.nan,
            "tuning_status": tuning_status,
        }

    results = []
    for alpha in SBA_ALPHA_GRID:
        candidate = forecast_constant_by_sku(
            fit_train,
            validation,
            level_function=croston_sba_level,
            model_name="SBA Croston",
            level_kwargs={"alpha": alpha},
        )
        results.append(
            {
                "alpha": alpha,
                "score": score_candidate_forecast(candidate),
            }
        )

    best = min(results, key=lambda item: item["score"])
    return {
        "model": "SBA Croston",
        "params": {"alpha": best["alpha"]},
        "tuning_wmape_percent": 100 * best["score"],
        "tuning_status": tuning_status,
    }


def tune_tsb(train):
    fit_train, validation, tuning_status = create_temporal_tuning_split(train)
    default_alpha = 0.20 if 0.20 in TSB_ALPHA_GRID else TSB_ALPHA_GRID[0]
    default_beta = 0.10 if 0.10 in TSB_BETA_GRID else TSB_BETA_GRID[0]

    if validation.empty or tuning_status != "validation":
        return {
            "model": "TSB",
            "params": {"alpha": default_alpha, "beta": default_beta},
            "tuning_wmape_percent": np.nan,
            "tuning_status": tuning_status,
        }

    results = []
    for alpha in TSB_ALPHA_GRID:
        for beta in TSB_BETA_GRID:
            candidate = forecast_constant_by_sku(
                fit_train,
                validation,
                level_function=tsb_level,
                model_name="TSB",
                level_kwargs={"alpha": alpha, "beta": beta},
            )
            results.append(
                {
                    "alpha": alpha,
                    "beta": beta,
                    "score": score_candidate_forecast(candidate),
                }
            )

    best = min(results, key=lambda item: item["score"])
    return {
        "model": "TSB",
        "params": {"alpha": best["alpha"], "beta": best["beta"]},
        "tuning_wmape_percent": 100 * best["score"],
        "tuning_status": tuning_status,
    }


HURDLE_RF_MODEL_PARAM_KEYS = {
    "n_estimators",
    "classifier_min_samples_leaf",
    "regressor_min_samples_leaf",
    "max_features",
    "occurrence_threshold",
}

DIRECT_RF_MODEL_PARAM_KEYS = {
    "n_estimators",
    "regressor_min_samples_leaf",
    "max_features",
    "forecast_floor",
}


def split_hurdle_params(params):
    params = params or {}
    model_params = {
        key: value
        for key, value in params.items()
        if key in HURDLE_RF_MODEL_PARAM_KEYS
    }
    feature_profile = params.get("feature_profile", HURDLE_RF_DEFAULT_FEATURE_PROFILE)
    selected_feature_columns = params.get("selected_feature_columns")
    return model_params, feature_profile, selected_feature_columns


def feature_family(feature):
    if feature.startswith("rf_"):
        return "rf_internal_history"
    if feature.startswith("calendar_"):
        return "calendar"
    if feature.startswith("external_"):
        return "external"
    return "other"


def feature_columns_for_profile(all_feature_columns, feature_profile, selected_feature_columns=None):
    all_feature_columns = list(dict.fromkeys(all_feature_columns))
    if selected_feature_columns:
        selected = [feature for feature in selected_feature_columns if feature in all_feature_columns]
        if selected:
            return selected

    rf_features = [feature for feature in all_feature_columns if feature.startswith("rf_")]
    calendar_features = [feature for feature in all_feature_columns if feature.startswith("calendar_")]
    external_features = [feature for feature in all_feature_columns if feature.startswith("external_")]

    if feature_profile == "rf_history_only":
        selected = rf_features
    elif feature_profile == "rf_history_calendar":
        selected = rf_features + calendar_features
    elif feature_profile == "rf_history_calendar_external":
        selected = rf_features + calendar_features + external_features
    elif feature_profile == "positive_impact_auto":
        selected = rf_features + calendar_features
    else:
        selected = all_feature_columns

    if not selected:
        selected = all_feature_columns
    return sorted(set(selected))


def fit_predict_hurdle_random_forest_from_frames(
    train_with_stats,
    test_with_stats,
    feature_columns,
    params,
    feature_profile,
):
    model_params, _, _ = split_hurdle_params(params)
    model = HurdleRandomForest(random_state=RANDOM_STATE, **model_params)
    model.fit(train_with_stats, feature_columns)

    forecast = test_with_stats[[SKU_COLUMN, DATE_COLUMN, TARGET_COLUMN]].copy()
    probability, amount = model.predict_components(test_with_stats)
    forecast["occurrence_probability"] = probability
    forecast["positive_amount_forecast"] = amount
    forecast["occurrence_threshold"] = model.occurrence_threshold
    forecast["forecast"] = model.predict(test_with_stats)
    forecast["model"] = "Hurdle Random Forest"
    forecast["feature_count"] = len(feature_columns)
    forecast["feature_profile"] = feature_profile
    forecast["training_auc"] = model.training_auc

    feature_metadata = pd.DataFrame(
        {
            "feature": feature_columns,
            "model": "Hurdle Random Forest",
            "feature_profile": feature_profile,
        }
    )
    feature_metadata["feature_family"] = feature_metadata["feature"].map(feature_family)
    feature_metadata["is_external_feature"] = feature_metadata["feature"].str.startswith("external_")
    feature_metadata["is_sku_train_stat"] = feature_metadata["feature"].str.startswith("sku_train_")
    feature_metadata["is_rf_internal_history_feature"] = feature_metadata["feature"].str.startswith("rf_")
    feature_metadata["selected_by_feature_gate"] = True

    if model.classifier is not None:
        classifier_importance = pd.Series(model.classifier.feature_importances_, index=feature_columns)
    else:
        classifier_importance = pd.Series(0.0, index=feature_columns)
    if model.regressor is not None:
        regressor_importance = pd.Series(model.regressor.feature_importances_, index=feature_columns)
    else:
        regressor_importance = pd.Series(0.0, index=feature_columns)

    feature_metadata["classifier_importance"] = feature_metadata["feature"].map(classifier_importance)
    feature_metadata["regressor_importance"] = feature_metadata["feature"].map(regressor_importance)
    feature_metadata["combined_importance"] = (
        0.70 * feature_metadata["classifier_importance"].fillna(0.0)
        + 0.30 * feature_metadata["regressor_importance"].fillna(0.0)
    )

    return forecast, feature_metadata


def evaluate_hurdle_feature_set(train_with_stats, validation_with_stats, feature_columns, params):
    if not feature_columns:
        return np.inf
    candidate, _ = fit_predict_hurdle_random_forest_from_frames(
        train_with_stats,
        validation_with_stats,
        feature_columns=feature_columns,
        params=params,
        feature_profile="feature_gate_eval",
    )
    return 100 * score_candidate_forecast(candidate)


def select_positive_impact_feature_columns(train_with_stats, validation_with_stats, all_feature_columns, params):
    rf_features = feature_columns_for_profile(all_feature_columns, "rf_history_only")
    calendar_features = [
        feature for feature in all_feature_columns
        if feature.startswith("calendar_")
    ]
    external_features = [
        feature for feature in all_feature_columns
        if feature.startswith("external_")
    ]

    base_columns = rf_features if rf_features else feature_columns_for_profile(all_feature_columns, "rf_history_calendar")
    base_score = evaluate_hurdle_feature_set(
        train_with_stats,
        validation_with_stats,
        base_columns,
        params,
    )

    gate_rows = []
    candidate_groups = []
    if calendar_features:
        candidate_groups.append(("calendar_group", calendar_features, "calendar"))
    for feature in external_features:
        candidate_groups.append((feature, [feature], "external"))

    for group_name, candidate_features, candidate_family in candidate_groups:
        candidate_columns = sorted(set(base_columns + candidate_features))
        try:
            candidate_score = evaluate_hurdle_feature_set(
                train_with_stats,
                validation_with_stats,
                candidate_columns,
                params,
            )
        except Exception:
            candidate_score = np.inf

        improvement = base_score - candidate_score
        gate_rows.append(
            {
                "feature_gate_group": group_name,
                "feature_family": candidate_family,
                "base_wmape_percent": base_score,
                "candidate_wmape_percent": candidate_score,
                "wmape_improvement_percent": improvement,
                "selected_by_feature_gate": (
                    np.isfinite(candidate_score)
                    and improvement >= FEATURE_GATE_MIN_WMAPE_IMPROVEMENT_PERCENT
                ),
                "feature_count": len(candidate_features),
                "features": ", ".join(candidate_features),
            }
        )

    gate_report = pd.DataFrame(gate_rows)
    if gate_report.empty:
        return sorted(set(base_columns)), gate_report

    selected_rows = gate_report.loc[gate_report["selected_by_feature_gate"]].sort_values(
        "wmape_improvement_percent",
        ascending=False,
    )
    selected_extra_features = []
    for _, row in selected_rows.iterrows():
        for feature in str(row["features"]).split(", "):
            if feature and feature not in selected_extra_features:
                selected_extra_features.append(feature)
            if len(selected_extra_features) >= FEATURE_GATE_MAX_EXTRA_FEATURES:
                break
        if len(selected_extra_features) >= FEATURE_GATE_MAX_EXTRA_FEATURES:
            break

    selected_columns = sorted(set(base_columns + selected_extra_features))
    return selected_columns, gate_report


def resolve_hurdle_feature_columns(train_with_stats, validation_or_test_with_stats, params, for_tuning=False):
    all_feature_columns = get_hurdle_feature_columns(train_with_stats)
    model_params, feature_profile, selected_feature_columns = split_hurdle_params(params)

    gate_report = pd.DataFrame()
    if feature_profile == "positive_impact_auto" and for_tuning:
        feature_columns, gate_report = select_positive_impact_feature_columns(
            train_with_stats,
            validation_or_test_with_stats,
            all_feature_columns,
            model_params,
        )
    else:
        feature_columns = feature_columns_for_profile(
            all_feature_columns,
            feature_profile,
            selected_feature_columns=selected_feature_columns,
        )

    return feature_columns, gate_report


def fit_predict_hurdle_random_forest(train, test, params, lag_months=FORECAST_LAG_MONTHS):
    train_with_stats, test_with_stats, _ = add_train_only_sku_features(train, test, lag_months=lag_months)
    model_params, feature_profile, selected_feature_columns = split_hurdle_params(params)
    all_feature_columns = get_hurdle_feature_columns(train_with_stats)
    feature_columns = feature_columns_for_profile(
        all_feature_columns,
        feature_profile,
        selected_feature_columns=selected_feature_columns,
    )

    return fit_predict_hurdle_random_forest_from_frames(
        train_with_stats,
        test_with_stats,
        feature_columns=feature_columns,
        params=model_params,
        feature_profile=feature_profile,
    )


def tune_hurdle_random_forest(train, lag_months=FORECAST_LAG_MONTHS):
    fit_train, validation, tuning_status = create_temporal_tuning_split(train)
    default_params = {
        **HURDLE_RF_PARAM_GRID[0],
        "feature_profile": HURDLE_RF_DEFAULT_FEATURE_PROFILE,
        "occurrence_threshold": HURDLE_RF_DEFAULT_OCCURRENCE_THRESHOLD,
    }

    if validation.empty or tuning_status != "validation":
        return {
            "model": "Hurdle Random Forest",
            "params": default_params,
            "tuning_wmape_percent": np.nan,
            "tuning_status": tuning_status,
        }

    train_with_stats, validation_with_stats, _ = add_train_only_sku_features(fit_train, validation, lag_months=lag_months)
    all_feature_columns = get_hurdle_feature_columns(train_with_stats)
    results = []

    for base_model_params in HURDLE_RF_PARAM_GRID:
        for occurrence_threshold in HURDLE_RF_OCCURRENCE_THRESHOLD_GRID:
            model_params = {
                **base_model_params,
                "occurrence_threshold": occurrence_threshold,
            }
            for feature_profile in HURDLE_RF_FEATURE_PROFILES:
                if feature_profile == "positive_impact_auto" and (
                    base_model_params != HURDLE_RF_PARAM_GRID[0]
                    or occurrence_threshold != HURDLE_RF_DEFAULT_OCCURRENCE_THRESHOLD
                ):
                    continue
                candidate_params = {
                    **model_params,
                    "feature_profile": feature_profile,
                }
                try:
                    if feature_profile == "positive_impact_auto":
                        feature_columns, gate_report = select_positive_impact_feature_columns(
                            train_with_stats,
                            validation_with_stats,
                            all_feature_columns,
                            model_params,
                        )
                        candidate_params["selected_feature_columns"] = feature_columns
                        candidate_params["feature_gate_selected_count"] = len(feature_columns)
                        candidate_params["feature_gate_external_count"] = len(
                            [feature for feature in feature_columns if feature.startswith("external_")]
                        )
                    else:
                        feature_columns = feature_columns_for_profile(
                            all_feature_columns,
                            feature_profile,
                        )

                    candidate, _ = fit_predict_hurdle_random_forest_from_frames(
                        train_with_stats,
                        validation_with_stats,
                        feature_columns=feature_columns,
                        params=model_params,
                        feature_profile=feature_profile,
                    )
                    score = score_candidate_forecast(candidate)
                except Exception:
                    score = np.inf

                results.append(
                    {
                        "params": candidate_params,
                        "score": score,
                    }
                )

    best = min(results, key=lambda item: item["score"])
    if not np.isfinite(best["score"]):
        return {
            "model": "Hurdle Random Forest",
            "params": default_params,
            "tuning_wmape_percent": np.nan,
            "tuning_status": "tuning_failed_used_default",
        }

    return {
        "model": "Hurdle Random Forest",
        "params": best["params"],
        "tuning_wmape_percent": 100 * best["score"],
        "tuning_status": tuning_status,
    }


def split_direct_rf_params(params):
    params = params or {}
    model_params = {
        key: value
        for key, value in params.items()
        if key in DIRECT_RF_MODEL_PARAM_KEYS
    }
    feature_profile = params.get("feature_profile", DIRECT_RF_DEFAULT_FEATURE_PROFILE)
    selected_feature_columns = params.get("selected_feature_columns")
    return model_params, feature_profile, selected_feature_columns


def fit_predict_direct_demand_random_forest_from_frames(
    train_with_stats,
    test_with_stats,
    feature_columns,
    params,
    feature_profile,
):
    model_params, _, _ = split_direct_rf_params(params)
    model = DirectDemandRandomForest(random_state=RANDOM_STATE, **model_params)
    model.fit(train_with_stats, feature_columns)

    forecast = test_with_stats[[SKU_COLUMN, DATE_COLUMN, TARGET_COLUMN]].copy()
    forecast["forecast"] = model.predict(test_with_stats)
    forecast["model"] = "Direct Demand Random Forest"
    forecast["feature_count"] = len(feature_columns)
    forecast["feature_profile"] = feature_profile
    forecast["regression_forecast_floor"] = model.forecast_floor
    forecast["training_auc"] = np.nan
    return forecast


def fit_predict_direct_demand_random_forest(train, test, params, lag_months=FORECAST_LAG_MONTHS):
    train_with_stats, test_with_stats, _ = add_train_only_sku_features(train, test, lag_months=lag_months)
    model_params, feature_profile, selected_feature_columns = split_direct_rf_params(params)
    all_feature_columns = get_hurdle_feature_columns(train_with_stats)
    feature_columns = feature_columns_for_profile(
        all_feature_columns,
        feature_profile,
        selected_feature_columns=selected_feature_columns,
    )
    return fit_predict_direct_demand_random_forest_from_frames(
        train_with_stats,
        test_with_stats,
        feature_columns=feature_columns,
        params=model_params,
        feature_profile=feature_profile,
    )


def tune_direct_demand_random_forest(train, lag_months=FORECAST_LAG_MONTHS):
    fit_train, validation, tuning_status = create_temporal_tuning_split(train)
    default_params = {
        **DIRECT_RF_PARAM_GRID[0],
        "feature_profile": DIRECT_RF_DEFAULT_FEATURE_PROFILE,
        "forecast_floor": DIRECT_RF_DEFAULT_FORECAST_FLOOR,
    }

    if validation.empty or tuning_status != "validation":
        return {
            "model": "Direct Demand Random Forest",
            "params": default_params,
            "tuning_wmape_percent": np.nan,
            "tuning_status": tuning_status,
        }

    train_with_stats, validation_with_stats, _ = add_train_only_sku_features(fit_train, validation, lag_months=lag_months)
    all_feature_columns = get_hurdle_feature_columns(train_with_stats)
    results = []

    for base_model_params in DIRECT_RF_PARAM_GRID:
        for forecast_floor in DIRECT_RF_FORECAST_FLOOR_GRID:
            model_params = {
                **base_model_params,
                "forecast_floor": forecast_floor,
            }
            for feature_profile in DIRECT_RF_FEATURE_PROFILES:
                candidate_params = {
                    **model_params,
                    "feature_profile": feature_profile,
                }
                try:
                    feature_columns = feature_columns_for_profile(
                        all_feature_columns,
                        feature_profile,
                    )
                    candidate = fit_predict_direct_demand_random_forest_from_frames(
                        train_with_stats,
                        validation_with_stats,
                        feature_columns=feature_columns,
                        params=model_params,
                        feature_profile=feature_profile,
                    )
                    score = score_candidate_forecast(candidate)
                except Exception:
                    score = np.inf

                results.append(
                    {
                        "params": candidate_params,
                        "score": score,
                    }
                )

    best = min(results, key=lambda item: item["score"])
    if not np.isfinite(best["score"]):
        return {
            "model": "Direct Demand Random Forest",
            "params": default_params,
            "tuning_wmape_percent": np.nan,
            "tuning_status": "tuning_failed_used_default",
        }

    return {
        "model": "Direct Demand Random Forest",
        "params": best["params"],
        "tuning_wmape_percent": 100 * best["score"],
        "tuning_status": tuning_status,
    }


def normalize_positive_series(values):
    values = values.fillna(0).clip(lower=0).astype(float)
    total = values.sum()
    if total <= 0:
        return values * 0.0
    return values / total


def available_aggregate_descriptor_columns(data):
    candidates = []
    for column in AGGREGATE_ALLOCATION_DESCRIPTOR_COLUMNS:
        if column not in data.columns:
            continue
        series = data[column].dropna().astype(str)
        unique_values = series.nunique()
        if 1 < unique_values <= AGGREGATE_DESCRIPTOR_MAX_UNIQUE:
            candidates.append(column)
    return candidates


def latest_descriptor_by_sku(data, descriptor_column):
    if descriptor_column not in data.columns:
        return pd.Series(dtype=object)

    descriptor_frame = (
        data[[SKU_COLUMN, DATE_COLUMN, descriptor_column]]
        .dropna(subset=[descriptor_column])
        .sort_values([SKU_COLUMN, DATE_COLUMN])
        .drop_duplicates(SKU_COLUMN, keep="last")
        .set_index(SKU_COLUMN)[descriptor_column]
        .astype(str)
    )
    return descriptor_frame


def recent_train_window(train, months=6):
    if train.empty or not train[DATE_COLUMN].notna().any():
        return train.iloc[0:0].copy()
    recent_start = train[DATE_COLUMN].max() - pd.DateOffset(months=months - 1)
    return train.loc[train[DATE_COLUMN] >= recent_start].copy()


def monthly_total_forecast_from_train(train, test_dates, total_strategy):
    monthly_total = (
        train
        .groupby(DATE_COLUMN)[TARGET_COLUMN]
        .sum()
        .sort_index()
    )
    test_dates = pd.Series(pd.to_datetime(test_dates).unique()).sort_values()

    if monthly_total.empty:
        return pd.Series(0.0, index=test_dates)

    fallback = float(monthly_total.tail(6).mean())
    forecasts = {}

    for date_value in test_dates:
        date_value = pd.Timestamp(date_value)
        if total_strategy == "recent_3m_total":
            forecast = monthly_total.tail(3).mean()
        elif total_strategy == "recent_6m_total":
            forecast = monthly_total.tail(6).mean()
        elif total_strategy == "train_mean_total":
            forecast = monthly_total.mean()
        elif total_strategy == "same_month_mean_total":
            same_month = monthly_total.loc[monthly_total.index.month == date_value.month]
            forecast = same_month.mean() if len(same_month) else fallback
        else:
            raise ValueError(f"Unknown aggregate total strategy: {total_strategy}")

        forecasts[date_value] = float(max(forecast, 0.0))

    return pd.Series(forecasts)


def sku_level_allocation_scores(train, test):
    test_skus = pd.Index(test[SKU_COLUMN].dropna().unique())
    train = train.sort_values([SKU_COLUMN, DATE_COLUMN]).copy()
    recent_train = recent_train_window(train, months=6)

    known_months = train.groupby(SKU_COLUMN)[TARGET_COLUMN].size().reindex(test_skus, fill_value=0)
    total_by_sku = train.groupby(SKU_COLUMN)[TARGET_COLUMN].sum().reindex(test_skus, fill_value=0.0)
    positive_months = (
        train[TARGET_COLUMN].gt(0)
        .groupby(train[SKU_COLUMN])
        .sum()
        .reindex(test_skus, fill_value=0.0)
    )
    mean_positive = (
        train.loc[train[TARGET_COLUMN].gt(0)]
        .groupby(SKU_COLUMN)[TARGET_COLUMN]
        .mean()
        .reindex(test_skus, fill_value=0.0)
    )

    recent_known_months = recent_train.groupby(SKU_COLUMN)[TARGET_COLUMN].size().reindex(test_skus, fill_value=0)
    recent_by_sku = recent_train.groupby(SKU_COLUMN)[TARGET_COLUMN].sum().reindex(test_skus, fill_value=0.0)
    recent_positive_months = (
        recent_train[TARGET_COLUMN].gt(0)
        .groupby(recent_train[SKU_COLUMN])
        .sum()
        .reindex(test_skus, fill_value=0.0)
        if not recent_train.empty
        else pd.Series(0.0, index=test_skus)
    )

    positive_rate = (
        positive_months / known_months.replace(0, np.nan)
    ).fillna(0.0)
    recent_positive_rate = (
        recent_positive_months / recent_known_months.replace(0, np.nan)
    ).fillna(0.0)

    last_positive_date = (
        train.loc[train[TARGET_COLUMN].gt(0)]
        .groupby(SKU_COLUMN)[DATE_COLUMN]
        .max()
        .reindex(test_skus)
    )
    max_train_date = train[DATE_COLUMN].max() if train[DATE_COLUMN].notna().any() else pd.NaT
    if pd.isna(max_train_date):
        recency_score = pd.Series(0.0, index=test_skus)
    else:
        months_since_positive = (
            (pd.Timestamp(max_train_date).year - pd.to_datetime(last_positive_date).dt.year) * 12
            + (pd.Timestamp(max_train_date).month - pd.to_datetime(last_positive_date).dt.month)
        )
        recency_score = (1 / (1 + months_since_positive)).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    demand_share = normalize_positive_series(0.70 * recent_by_sku + 0.30 * total_by_sku)
    frequency_share = normalize_positive_series(0.65 * recent_positive_rate + 0.35 * positive_rate)
    recency_share = normalize_positive_series(recency_score)
    size_share = normalize_positive_series(mean_positive)
    activation_share = normalize_positive_series(
        0.55 * frequency_share
        + 0.30 * recency_share
        + 0.15 * size_share
    )
    probability_weighted_share = normalize_positive_series(
        0.55 * demand_share
        + 0.35 * activation_share
        + 0.10 * normalize_positive_series(total_by_sku)
    )

    return pd.DataFrame(
        {
            "known_months": known_months,
            "train_total": total_by_sku,
            "recent_total": recent_by_sku,
            "positive_rate": positive_rate,
            "recent_positive_rate": recent_positive_rate,
            "recency_score": recency_score,
            "mean_positive": mean_positive,
            "demand_share": demand_share,
            "activation_share": activation_share,
            "probability_weighted_share": probability_weighted_share,
        },
        index=test_skus,
    )


def sku_history_allocation_weights(train, test, allocation_strategy):
    scores = sku_level_allocation_scores(train, test)

    if allocation_strategy == "train_total_share":
        weights = normalize_positive_series(scores["train_total"])
    elif allocation_strategy == "recent_6m_share":
        weights = normalize_positive_series(scores["recent_total"])
    elif allocation_strategy == "hybrid_recent_train_share":
        weights = scores["demand_share"]
    elif allocation_strategy == "probability_weighted_sku_share":
        weights = scores["probability_weighted_share"]
    else:
        raise ValueError(f"Unknown SKU history allocation strategy: {allocation_strategy}")

    return weights


def descriptor_group_share(train, test_descriptors, descriptor_column, group_strategy):
    train = train.copy()
    train["_descriptor"] = train[descriptor_column].fillna("__missing__").astype(str)
    recent_train = recent_train_window(train, months=6)

    train_group_total = train.groupby("_descriptor")[TARGET_COLUMN].sum()
    recent_group_total = recent_train.groupby("_descriptor")[TARGET_COLUMN].sum()
    all_groups = train_group_total.index.union(recent_group_total.index)

    train_group_share = normalize_positive_series(train_group_total.reindex(all_groups, fill_value=0.0))
    recent_group_share = normalize_positive_series(recent_group_total.reindex(all_groups, fill_value=0.0))

    if group_strategy == "descriptor_train_group":
        group_share = train_group_share
    elif group_strategy == "descriptor_recent_group":
        group_share = recent_group_share
    elif group_strategy == "descriptor_hybrid_group":
        group_share = normalize_positive_series(0.70 * recent_group_share + 0.30 * train_group_share)
    else:
        raise ValueError(f"Unknown descriptor group strategy: {group_strategy}")

    test_groups = pd.Index(pd.Series(test_descriptors.dropna().unique()).astype(str))
    if len(test_groups) == 0:
        return pd.Series(dtype=float)

    equal_group_share = pd.Series(1.0 / len(test_groups), index=test_groups)
    group_share = group_share.reindex(test_groups, fill_value=0.0)

    if group_share.sum() <= 0:
        return equal_group_share

    return normalize_positive_series(
        (1 - AGGREGATE_NEW_SKU_EQUAL_WEIGHT) * normalize_positive_series(group_share)
        + AGGREGATE_NEW_SKU_EQUAL_WEIGHT * equal_group_share
    )


def within_group_sku_share(scores, group_skus, within_group_strategy):
    group_skus = pd.Index(group_skus)
    if len(group_skus) == 0:
        return pd.Series(dtype=float)

    equal_share = pd.Series(1.0 / len(group_skus), index=group_skus)
    history_share = normalize_positive_series(scores["demand_share"].reindex(group_skus, fill_value=0.0))
    probability_share = normalize_positive_series(
        scores["probability_weighted_share"].reindex(group_skus, fill_value=0.0)
    )
    activation_share = normalize_positive_series(scores["activation_share"].reindex(group_skus, fill_value=0.0))

    if within_group_strategy == "equal_within_group":
        return equal_share

    if within_group_strategy == "history_or_equal":
        if history_share.sum() <= 0:
            return equal_share
        return normalize_positive_series(
            (1 - AGGREGATE_NEW_SKU_EQUAL_WEIGHT) * history_share
            + AGGREGATE_NEW_SKU_EQUAL_WEIGHT * equal_share
        )

    if within_group_strategy == "probability_weighted_history":
        if probability_share.sum() <= 0:
            return equal_share
        return normalize_positive_series(
            0.70 * probability_share
            + 0.15 * activation_share
            + 0.15 * equal_share
        )

    raise ValueError(f"Unknown within-group allocation strategy: {within_group_strategy}")


def descriptor_allocation_weights(
    train,
    test,
    descriptor_column,
    group_strategy="descriptor_hybrid_group",
    within_group_strategy="probability_weighted_history",
):
    test_skus = pd.Index(test[SKU_COLUMN].dropna().unique())
    if descriptor_column not in train.columns or descriptor_column not in test.columns:
        return pd.Series(0.0, index=test_skus)

    test_descriptors = latest_descriptor_by_sku(test, descriptor_column).reindex(test_skus)
    train_descriptors = latest_descriptor_by_sku(train, descriptor_column).reindex(test_skus)
    test_descriptors = test_descriptors.fillna(train_descriptors).fillna("__missing__").astype(str)

    group_share = descriptor_group_share(
        train,
        test_descriptors,
        descriptor_column,
        group_strategy,
    )
    scores = sku_level_allocation_scores(train, test)
    weights = pd.Series(0.0, index=test_skus, dtype=float)

    for descriptor_value, group_sku_values in test_descriptors.groupby(test_descriptors):
        group_skus = pd.Index(group_sku_values.index)
        group_mass = float(group_share.reindex([descriptor_value], fill_value=0.0).iloc[0])
        if group_mass <= 0 or len(group_skus) == 0:
            continue

        weights.loc[group_skus] = group_mass * within_group_sku_share(
            scores,
            group_skus,
            within_group_strategy,
        )

    return normalize_positive_series(weights)


def build_aggregate_allocation_options(train, test=None):
    option_rows = [
        {"allocation_strategy": "train_total_share"},
        {"allocation_strategy": "recent_6m_share"},
        {"allocation_strategy": "hybrid_recent_train_share"},
        {"allocation_strategy": "probability_weighted_sku_share"},
    ]

    descriptor_source = train if test is None else pd.concat([train, test], ignore_index=True, sort=False)
    for descriptor_column in available_aggregate_descriptor_columns(descriptor_source):
        for group_strategy in [
            "descriptor_train_group",
            "descriptor_recent_group",
            "descriptor_hybrid_group",
        ]:
            for within_group_strategy in [
                "probability_weighted_history",
                "history_or_equal",
                "equal_within_group",
            ]:
                option_rows.append(
                    {
                        "allocation_strategy": group_strategy,
                        "descriptor_column": descriptor_column,
                        "within_group_strategy": within_group_strategy,
                    }
                )

    return option_rows


def sku_allocation_weights(
    train,
    test,
    allocation_strategy,
    descriptor_column=None,
    within_group_strategy="probability_weighted_history",
):
    if allocation_strategy.startswith("descriptor_"):
        return descriptor_allocation_weights(
            train,
            test,
            descriptor_column=descriptor_column,
            group_strategy=allocation_strategy,
            within_group_strategy=within_group_strategy,
        )

    return sku_history_allocation_weights(
        train,
        test,
        allocation_strategy=allocation_strategy,
    )


def forecast_aggregate_allocation(
    train,
    test,
    total_strategy,
    allocation_strategy,
    descriptor_column=None,
    within_group_strategy="probability_weighted_history",
):
    month_totals = monthly_total_forecast_from_train(
        train,
        test[DATE_COLUMN],
        total_strategy=total_strategy,
    )
    weights = sku_allocation_weights(
        train,
        test,
        allocation_strategy=allocation_strategy,
        descriptor_column=descriptor_column,
        within_group_strategy=within_group_strategy,
    )

    forecast = test[[SKU_COLUMN, DATE_COLUMN, TARGET_COLUMN]].copy()
    forecast["forecast_month_total"] = forecast[DATE_COLUMN].map(month_totals).fillna(0.0)
    forecast["allocation_weight"] = forecast[SKU_COLUMN].map(weights).fillna(0.0)
    forecast["forecast"] = forecast["forecast_month_total"] * forecast["allocation_weight"]
    forecast["model"] = "Aggregate Allocation"
    forecast["aggregate_total_strategy"] = total_strategy
    forecast["aggregate_allocation_strategy"] = allocation_strategy
    forecast["aggregate_descriptor_column"] = descriptor_column or ""
    forecast["aggregate_within_group_strategy"] = within_group_strategy
    forecast["aggregate_allocation_layer"] = (
        "descriptor_group_to_probability_weighted_sku"
        if allocation_strategy.startswith("descriptor_")
        else "sku_probability_weighted"
    )
    return forecast



def choose_descriptor_column(train, test=None, preferred=None):
    preferred = preferred or AGGREGATE_ALLOCATION_DESCRIPTOR_COLUMNS
    descriptor_source = train if test is None else pd.concat([train, test], ignore_index=True, sort=False)
    available = available_aggregate_descriptor_columns(descriptor_source)
    for column in preferred:
        if column in available:
            return column
    return available[0] if available else None


def descriptor_by_sku_with_fallback(train, test, descriptor_column):
    test_skus = pd.Index(test[SKU_COLUMN].dropna().unique())
    if descriptor_column is None or descriptor_column not in test.columns:
        return pd.Series("__all__", index=test_skus)

    test_descriptors = latest_descriptor_by_sku(test, descriptor_column).reindex(test_skus)
    train_descriptors = (
        latest_descriptor_by_sku(train, descriptor_column).reindex(test_skus)
        if descriptor_column in train.columns
        else pd.Series(index=test_skus, dtype=object)
    )
    return test_descriptors.fillna(train_descriptors).fillna("__missing__").astype(str)


def empirical_bayes_sku_scores(
    train,
    test,
    descriptor_column=None,
    probability_prior_strength=EMPIRICAL_BAYES_DEFAULT_PROBABILITY_PRIOR,
    size_prior_strength=EMPIRICAL_BAYES_DEFAULT_SIZE_PRIOR,
    seasonality_weight=EMPIRICAL_BAYES_DEFAULT_SEASONALITY_WEIGHT,
):
    test_skus = pd.Index(test[SKU_COLUMN].dropna().unique())
    train = train.copy()
    train[DATE_COLUMN] = pd.to_datetime(train[DATE_COLUMN])
    test = test.copy()
    test[DATE_COLUMN] = pd.to_datetime(test[DATE_COLUMN])

    descriptor_column = descriptor_column or choose_descriptor_column(train, test)
    test_descriptors = descriptor_by_sku_with_fallback(train, test, descriptor_column)
    descriptor_name = descriptor_column or "__all__"

    global_months = max(len(train), 1)
    global_positives = int(train[TARGET_COLUMN].gt(0).sum())
    global_rate = global_positives / global_months
    global_positive = train.loc[train[TARGET_COLUMN].gt(0), TARGET_COLUMN]
    global_positive_mean = float(global_positive.mean()) if len(global_positive) else 0.0

    sku_months = train.groupby(SKU_COLUMN)[TARGET_COLUMN].size().reindex(test_skus, fill_value=0.0)
    sku_positives = (
        train[TARGET_COLUMN].gt(0)
        .groupby(train[SKU_COLUMN])
        .sum()
        .reindex(test_skus, fill_value=0.0)
    )
    sku_positive_mean = (
        train.loc[train[TARGET_COLUMN].gt(0)]
        .groupby(SKU_COLUMN)[TARGET_COLUMN]
        .mean()
        .reindex(test_skus)
    )

    if descriptor_column is not None and descriptor_column in train.columns:
        train["_eb_descriptor"] = train[descriptor_column].fillna("__missing__").astype(str)
    else:
        train["_eb_descriptor"] = "__all__"

    group_months = train.groupby("_eb_descriptor")[TARGET_COLUMN].size()
    group_positives = train[TARGET_COLUMN].gt(0).groupby(train["_eb_descriptor"]).sum()
    group_rate = (group_positives / group_months.replace(0, np.nan)).fillna(global_rate)
    group_positive_mean = (
        train.loc[train[TARGET_COLUMN].gt(0)]
        .groupby("_eb_descriptor")[TARGET_COLUMN]
        .mean()
    )

    train["_eb_month"] = train[DATE_COLUMN].dt.month
    group_month_counts = train.groupby(["_eb_descriptor", "_eb_month"])[TARGET_COLUMN].size()
    group_month_positives = train[TARGET_COLUMN].gt(0).groupby([train["_eb_descriptor"], train["_eb_month"]]).sum()
    group_month_rate = (group_month_positives / group_month_counts.replace(0, np.nan)).fillna(global_rate)

    last_positive_date = (
        train.loc[train[TARGET_COLUMN].gt(0)]
        .groupby(SKU_COLUMN)[DATE_COLUMN]
        .max()
        .reindex(test_skus)
    )
    max_train_date = train[DATE_COLUMN].max() if train[DATE_COLUMN].notna().any() else pd.NaT
    if pd.isna(max_train_date):
        recency_score = pd.Series(0.0, index=test_skus)
    else:
        months_since_positive = (
            (pd.Timestamp(max_train_date).year - pd.to_datetime(last_positive_date).dt.year) * 12
            + (pd.Timestamp(max_train_date).month - pd.to_datetime(last_positive_date).dt.month)
        )
        recency_score = (1 / (1 + months_since_positive)).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    rows = []
    for sku in test_skus:
        descriptor_value = test_descriptors.get(sku, "__missing__")
        descriptor_rate = float(group_rate.reindex([descriptor_value], fill_value=global_rate).iloc[0])
        descriptor_size = float(
            group_positive_mean.reindex([descriptor_value], fill_value=global_positive_mean).iloc[0]
        )
        months = float(sku_months.get(sku, 0.0))
        positives = float(sku_positives.get(sku, 0.0))
        sku_size = sku_positive_mean.get(sku, np.nan)
        if pd.isna(sku_size):
            sku_size = descriptor_size

        occurrence_probability = (
            positives + probability_prior_strength * descriptor_rate
        ) / max(months + probability_prior_strength, 1e-9)
        positive_amount = (
            positives * float(sku_size) + size_prior_strength * descriptor_size
        ) / max(positives + size_prior_strength, 1e-9)

        rows.append(
            {
                SKU_COLUMN: sku,
                "_eb_descriptor": descriptor_value,
                "eb_base_occurrence_probability": float(np.clip(occurrence_probability, 0, 1)),
                "eb_positive_amount": float(max(positive_amount, 0.0)),
                "eb_recency_score": float(recency_score.get(sku, 0.0)),
            }
        )

    sku_scores = pd.DataFrame(rows).set_index(SKU_COLUMN)
    scored = test[[SKU_COLUMN, DATE_COLUMN, TARGET_COLUMN]].copy()
    scored["_eb_descriptor"] = scored[SKU_COLUMN].map(sku_scores["_eb_descriptor"]).fillna("__missing__")
    scored["eb_base_occurrence_probability"] = (
        scored[SKU_COLUMN].map(sku_scores["eb_base_occurrence_probability"]).fillna(global_rate)
    )
    scored["eb_positive_amount"] = (
        scored[SKU_COLUMN].map(sku_scores["eb_positive_amount"]).fillna(global_positive_mean)
    )
    scored["eb_recency_score"] = (
        scored[SKU_COLUMN].map(sku_scores["eb_recency_score"]).fillna(0.0)
    )

    seasonal_rates = []
    for descriptor_value, date_value in scored[["_eb_descriptor", DATE_COLUMN]].itertuples(index=False, name=None):
        key = (descriptor_value, pd.Timestamp(date_value).month)
        seasonal_rates.append(float(group_month_rate.get(key, group_rate.get(descriptor_value, global_rate))))
    scored["eb_seasonal_occurrence_probability"] = seasonal_rates
    scored["eb_occurrence_probability"] = np.clip(
        (1 - seasonality_weight) * scored["eb_base_occurrence_probability"]
        + seasonality_weight * scored["eb_seasonal_occurrence_probability"],
        0,
        1,
    )
    scored["eb_allocation_score"] = (
        scored["eb_occurrence_probability"]
        * scored["eb_positive_amount"]
        * (0.50 + 0.50 * scored["eb_recency_score"])
    ).clip(lower=0.0)
    scored["eb_descriptor_column"] = descriptor_name
    return scored


def allocate_month_totals_by_row_score(
    scored_rows,
    month_totals,
    score_column,
    probability_column=None,
    min_probability=0.0,
    top_share=1.0,
):
    forecast = scored_rows.copy()
    forecast[DATE_COLUMN] = pd.to_datetime(forecast[DATE_COLUMN])
    forecast["forecast_month_total"] = forecast[DATE_COLUMN].map(month_totals).fillna(0.0)
    forecast["allocation_score"] = pd.to_numeric(forecast[score_column], errors="coerce").fillna(0.0).clip(lower=0.0)
    forecast["selected_by_allocation_gate"] = False
    forecast["allocation_weight"] = 0.0
    forecast["gate_min_probability"] = min_probability
    forecast["gate_top_share"] = top_share

    for date_value, group in forecast.groupby(DATE_COLUMN, sort=False):
        group_index = group.index
        scores = forecast.loc[group_index, "allocation_score"].copy()
        selected = scores.gt(0)

        if probability_column is not None and probability_column in forecast.columns:
            probabilities = pd.to_numeric(
                forecast.loc[group_index, probability_column],
                errors="coerce",
            ).fillna(0.0)
            selected = selected & probabilities.ge(min_probability)

        if top_share < 1.0 and len(group_index) > 0:
            top_n = max(1, int(np.ceil(len(group_index) * top_share)))
            candidate_scores = scores.loc[selected]
            if candidate_scores.empty:
                candidate_scores = scores
            top_index = candidate_scores.sort_values(ascending=False).head(top_n).index
            selected = selected & selected.index.isin(top_index)
            if not selected.any():
                selected = pd.Series(scores.index.isin(top_index), index=scores.index)

        if not selected.any() and len(group_index) > 0:
            top_index = scores.sort_values(ascending=False).head(1).index
            selected = pd.Series(scores.index.isin(top_index), index=scores.index)

        selected_index = group_index[selected.to_numpy()]
        selected_scores = scores.loc[selected_index]
        selected_total = selected_scores.sum()
        if selected_total > 0:
            weights = selected_scores / selected_total
        elif len(selected_index) > 0:
            weights = pd.Series(1.0 / len(selected_index), index=selected_index)
        else:
            weights = pd.Series(dtype=float)

        forecast.loc[selected_index, "allocation_weight"] = weights
        forecast.loc[selected_index, "selected_by_allocation_gate"] = True

    forecast["forecast"] = (
        forecast["forecast_month_total"] * forecast["allocation_weight"]
    ).clip(lower=0.0)
    return forecast


def forecast_empirical_bayes_allocation(
    train,
    test,
    total_strategy="recent_6m_total",
    descriptor_column=None,
    probability_prior_strength=EMPIRICAL_BAYES_DEFAULT_PROBABILITY_PRIOR,
    size_prior_strength=EMPIRICAL_BAYES_DEFAULT_SIZE_PRIOR,
    seasonality_weight=EMPIRICAL_BAYES_DEFAULT_SEASONALITY_WEIGHT,
    min_probability=HYBRID_DEFAULT_MIN_PROBABILITY,
    top_share=HYBRID_DEFAULT_TOP_SHARE,
):
    descriptor_column = descriptor_column or choose_descriptor_column(train, test)
    month_totals = monthly_total_forecast_from_train(
        train,
        test[DATE_COLUMN],
        total_strategy=total_strategy,
    )
    scored = empirical_bayes_sku_scores(
        train,
        test,
        descriptor_column=descriptor_column,
        probability_prior_strength=probability_prior_strength,
        size_prior_strength=size_prior_strength,
        seasonality_weight=seasonality_weight,
    )
    forecast = allocate_month_totals_by_row_score(
        scored,
        month_totals,
        score_column="eb_allocation_score",
        probability_column="eb_occurrence_probability",
        min_probability=min_probability,
        top_share=top_share,
    )
    forecast["model"] = "Empirical Bayes Allocation"
    forecast["hybrid_total_strategy"] = total_strategy
    forecast["hybrid_descriptor_column"] = descriptor_column or ""
    forecast["eb_probability_prior_strength"] = probability_prior_strength
    forecast["eb_size_prior_strength"] = size_prior_strength
    forecast["eb_seasonality_weight"] = seasonality_weight
    forecast["hybrid_allocation_layer"] = "aggregate_total_to_empirical_bayes_sku_month_gate"
    return forecast


def default_empirical_bayes_params(train, test=None):
    return {
        "total_strategy": "recent_6m_total",
        "descriptor_column": choose_descriptor_column(train, test),
        "probability_prior_strength": EMPIRICAL_BAYES_DEFAULT_PROBABILITY_PRIOR,
        "size_prior_strength": EMPIRICAL_BAYES_DEFAULT_SIZE_PRIOR,
        "seasonality_weight": EMPIRICAL_BAYES_DEFAULT_SEASONALITY_WEIGHT,
        "min_probability": HYBRID_DEFAULT_MIN_PROBABILITY,
        "top_share": HYBRID_DEFAULT_TOP_SHARE,
    }


def tune_empirical_bayes_allocation(train):
    fit_train, validation, tuning_status = create_temporal_tuning_split(train)
    default_params = default_empirical_bayes_params(train)

    if validation.empty or tuning_status != "validation":
        return {
            "model": "Empirical Bayes Allocation",
            "params": default_params,
            "tuning_wmape_percent": np.nan,
            "tuning_status": tuning_status,
        }

    descriptor_options = available_aggregate_descriptor_columns(
        pd.concat([fit_train, validation], ignore_index=True, sort=False)
    )
    if not descriptor_options:
        descriptor_options = [None]

    results = []
    for total_strategy in HYBRID_TOTAL_STRATEGIES:
        for descriptor_column in descriptor_options:
            for probability_prior_strength in EMPIRICAL_BAYES_PROBABILITY_PRIOR_GRID:
                for size_prior_strength in EMPIRICAL_BAYES_SIZE_PRIOR_GRID:
                    for seasonality_weight in EMPIRICAL_BAYES_SEASONALITY_WEIGHT_GRID:
                        for min_probability in HYBRID_GATE_MIN_PROBABILITY_GRID:
                            for top_share in HYBRID_GATE_TOP_SHARE_GRID:
                                params = {
                                    "total_strategy": total_strategy,
                                    "descriptor_column": descriptor_column,
                                    "probability_prior_strength": probability_prior_strength,
                                    "size_prior_strength": size_prior_strength,
                                    "seasonality_weight": seasonality_weight,
                                    "min_probability": min_probability,
                                    "top_share": top_share,
                                }
                                try:
                                    candidate = forecast_empirical_bayes_allocation(
                                        fit_train,
                                        validation,
                                        **params,
                                    )
                                    score = score_candidate_forecast(candidate)
                                except Exception:
                                    score = np.inf
                                results.append({"params": params, "score": score})

    best = min(results, key=lambda item: item["score"])
    if not np.isfinite(best["score"]):
        return {
            "model": "Empirical Bayes Allocation",
            "params": default_params,
            "tuning_wmape_percent": np.nan,
            "tuning_status": "tuning_failed_used_default",
        }

    return {
        "model": "Empirical Bayes Allocation",
        "params": best["params"],
        "tuning_wmape_percent": 100 * best["score"],
        "tuning_status": tuning_status,
    }


def fit_occurrence_model_components(train, test, model_params, feature_profile, lag_months=FORECAST_LAG_MONTHS):
    train_with_stats, test_with_stats, _ = add_train_only_sku_features(train, test, lag_months=lag_months)
    all_feature_columns = get_hurdle_feature_columns(train_with_stats)
    feature_columns = feature_columns_for_profile(all_feature_columns, feature_profile)
    fit_params = {**model_params, "occurrence_threshold": 0.0}
    model = HurdleRandomForest(random_state=RANDOM_STATE, **fit_params)
    model.fit(train_with_stats, feature_columns)
    probability, amount = model.predict_components(test_with_stats)

    scored = test_with_stats[[SKU_COLUMN, DATE_COLUMN, TARGET_COLUMN]].copy()
    scored["occurrence_probability"] = probability
    scored["positive_amount_forecast"] = amount
    scored["rf_expected_score"] = np.clip(probability * amount, 0, None)
    scored["feature_count"] = len(feature_columns)
    scored["feature_profile"] = feature_profile
    scored["training_auc"] = model.training_auc
    return scored


def score_rows_for_occurrence_gated_allocation(
    train,
    test,
    model_params,
    feature_profile,
    score_mode,
    descriptor_column=None,
    lag_months=FORECAST_LAG_MONTHS,
):
    scored = fit_occurrence_model_components(train, test, model_params, feature_profile, lag_months=lag_months)

    if score_mode == "rf_expected":
        scored["hybrid_allocation_score"] = scored["occurrence_probability"] * scored["positive_amount_forecast"]
    elif score_mode == "rf_probability":
        scored["hybrid_allocation_score"] = scored["occurrence_probability"]
    elif score_mode == "rf_probability_eb_size":
        eb_scored = empirical_bayes_sku_scores(
            train,
            test,
            descriptor_column=descriptor_column,
        )
        scored["eb_positive_amount"] = eb_scored["eb_positive_amount"].to_numpy()
        scored["eb_occurrence_probability"] = eb_scored["eb_occurrence_probability"].to_numpy()
        scored["hybrid_allocation_score"] = scored["occurrence_probability"] * scored["eb_positive_amount"]
    else:
        raise ValueError(f"Unknown occurrence-gated allocation score mode: {score_mode}")

    scored["hybrid_allocation_score"] = scored["hybrid_allocation_score"].fillna(0.0).clip(lower=0.0)
    return scored


def forecast_occurrence_gated_allocation(
    train,
    test,
    total_strategy="recent_6m_total",
    feature_profile=HURDLE_RF_DEFAULT_FEATURE_PROFILE,
    score_mode=OCCURRENCE_GATED_ALLOCATION_DEFAULT_SCORE_MODE,
    min_probability=HYBRID_DEFAULT_MIN_PROBABILITY,
    top_share=HYBRID_DEFAULT_TOP_SHARE,
    descriptor_column=None,
    lag_months=FORECAST_LAG_MONTHS,
    **model_params,
):
    if not model_params:
        model_params = HURDLE_RF_PARAM_GRID[0].copy()
    descriptor_column = descriptor_column or choose_descriptor_column(train, test)
    month_totals = monthly_total_forecast_from_train(
        train,
        test[DATE_COLUMN],
        total_strategy=total_strategy,
    )
    scored = score_rows_for_occurrence_gated_allocation(
        train,
        test,
        model_params=model_params,
        feature_profile=feature_profile,
        score_mode=score_mode,
        descriptor_column=descriptor_column,
        lag_months=lag_months,
    )
    forecast = allocate_month_totals_by_row_score(
        scored,
        month_totals,
        score_column="hybrid_allocation_score",
        probability_column="occurrence_probability",
        min_probability=min_probability,
        top_share=top_share,
    )
    forecast["model"] = "Occurrence Gated Allocation"
    forecast["hybrid_total_strategy"] = total_strategy
    forecast["hybrid_score_mode"] = score_mode
    forecast["hybrid_descriptor_column"] = descriptor_column or ""
    forecast["hybrid_allocation_layer"] = "aggregate_total_to_rf_occurrence_sku_month_gate"
    return forecast


def default_occurrence_gated_allocation_params(train, test=None):
    return {
        **HURDLE_RF_PARAM_GRID[0],
        "total_strategy": "recent_6m_total",
        "feature_profile": HURDLE_RF_DEFAULT_FEATURE_PROFILE,
        "score_mode": OCCURRENCE_GATED_ALLOCATION_DEFAULT_SCORE_MODE,
        "min_probability": HYBRID_DEFAULT_MIN_PROBABILITY,
        "top_share": HYBRID_DEFAULT_TOP_SHARE,
        "descriptor_column": choose_descriptor_column(train, test),
    }


def tune_occurrence_gated_allocation(train, lag_months=FORECAST_LAG_MONTHS):
    fit_train, validation, tuning_status = create_temporal_tuning_split(train)
    default_params = default_occurrence_gated_allocation_params(train)

    if validation.empty or tuning_status != "validation":
        return {
            "model": "Occurrence Gated Allocation",
            "params": default_params,
            "tuning_wmape_percent": np.nan,
            "tuning_status": tuning_status,
        }

    descriptor_column = choose_descriptor_column(fit_train, validation)
    results = []
    model_param_grid = HURDLE_RF_PARAM_GRID[:1]

    for model_params in model_param_grid:
        for feature_profile in OCCURRENCE_GATED_ALLOCATION_FEATURE_PROFILES:
            try:
                base_scored_by_mode = {
                    score_mode: score_rows_for_occurrence_gated_allocation(
                        fit_train,
                        validation,
                        model_params=model_params,
                        feature_profile=feature_profile,
                        score_mode=score_mode,
                        descriptor_column=descriptor_column,
                        lag_months=lag_months,
                    )
                    for score_mode in OCCURRENCE_GATED_ALLOCATION_SCORE_MODES
                }
            except Exception:
                continue

            for total_strategy in HYBRID_TOTAL_STRATEGIES:
                month_totals = monthly_total_forecast_from_train(
                    fit_train,
                    validation[DATE_COLUMN],
                    total_strategy=total_strategy,
                )
                for score_mode, scored in base_scored_by_mode.items():
                    for min_probability in HYBRID_GATE_MIN_PROBABILITY_GRID:
                        for top_share in HYBRID_GATE_TOP_SHARE_GRID:
                            params = {
                                **model_params,
                                "total_strategy": total_strategy,
                                "feature_profile": feature_profile,
                                "score_mode": score_mode,
                                "min_probability": min_probability,
                                "top_share": top_share,
                                "descriptor_column": descriptor_column,
                            }
                            try:
                                candidate = allocate_month_totals_by_row_score(
                                    scored,
                                    month_totals,
                                    score_column="hybrid_allocation_score",
                                    probability_column="occurrence_probability",
                                    min_probability=min_probability,
                                    top_share=top_share,
                                )
                                candidate["model"] = "Occurrence Gated Allocation"
                                score = score_candidate_forecast(candidate)
                            except Exception:
                                score = np.inf
                            results.append({"params": params, "score": score})

    if not results:
        return {
            "model": "Occurrence Gated Allocation",
            "params": default_params,
            "tuning_wmape_percent": np.nan,
            "tuning_status": "tuning_failed_used_default",
        }

    best = min(results, key=lambda item: item["score"])
    if not np.isfinite(best["score"]):
        return {
            "model": "Occurrence Gated Allocation",
            "params": default_params,
            "tuning_wmape_percent": np.nan,
            "tuning_status": "tuning_failed_used_default",
        }

    return {
        "model": "Occurrence Gated Allocation",
        "params": best["params"],
        "tuning_wmape_percent": 100 * best["score"],
        "tuning_status": tuning_status,
    }


def default_aggregate_params(train, test=None):
    allocation_options = build_aggregate_allocation_options(train, test)
    descriptor_options = [
        option for option in allocation_options
        if option["allocation_strategy"].startswith("descriptor_")
    ]
    if descriptor_options:
        preferred = [
            option for option in descriptor_options
            if option["allocation_strategy"] == "descriptor_hybrid_group"
            and option["within_group_strategy"] == "history_or_equal"
        ]
        allocation_params = preferred[0] if preferred else descriptor_options[0]
    else:
        allocation_params = {"allocation_strategy": "probability_weighted_sku_share"}

    return {
        "total_strategy": "recent_6m_total",
        **allocation_params,
    }


def tune_aggregate_allocation(train):
    fit_train, validation, tuning_status = create_temporal_tuning_split(train)
    default_params = default_aggregate_params(train)

    if validation.empty or tuning_status != "validation":
        return {
            "model": "Aggregate Allocation",
            "params": default_params,
            "tuning_wmape_percent": np.nan,
            "tuning_status": tuning_status,
        }

    total_strategies = [
        "recent_3m_total",
        "recent_6m_total",
        "train_mean_total",
        "same_month_mean_total",
    ]
    allocation_options = build_aggregate_allocation_options(fit_train, validation)

    results = []
    for total_strategy in total_strategies:
        for allocation_params in allocation_options:
            params = {
                "total_strategy": total_strategy,
                **allocation_params,
            }
            candidate = forecast_aggregate_allocation(
                fit_train,
                validation,
                **params,
            )
            results.append(
                {
                    "params": params,
                    "score": score_candidate_forecast(candidate),
                }
            )

    best = min(results, key=lambda item: item["score"])
    return {
        "model": "Aggregate Allocation",
        "params": best["params"],
        "tuning_wmape_percent": 100 * best["score"],
        "tuning_status": tuning_status,
    }

def baseline_tuning_result(model_name, params=None):
    return {
        "model": model_name,
        "params": params or {},
        "tuning_wmape_percent": np.nan,
        "tuning_status": "baseline_no_tuning",
    }


def build_baseline_tuning_results():
    return [
        baseline_tuning_result("Zero Forecast", {"strategy": "all_zero"}),
        baseline_tuning_result("SKU Train Mean", {"strategy": "sku_train_mean"}),
        baseline_tuning_result("Recent Mean", {"recent_months": 6}),
        baseline_tuning_result("Last Nonzero", {"strategy": "last_nonzero"}),
    ]


def forecast_model_from_tuning_result(train, test, tuning_result, lag_months=FORECAST_LAG_MONTHS):
    model_name = tuning_result["model"]
    params = tuning_result.get("params") or {}

    if model_name == "SBA Croston":
        return forecast_constant_by_sku(
            train,
            test,
            level_function=croston_sba_level,
            model_name=model_name,
            level_kwargs=params,
        )
    if model_name == "TSB":
        return forecast_constant_by_sku(
            train,
            test,
            level_function=tsb_level,
            model_name=model_name,
            level_kwargs=params,
        )
    if model_name == "Hurdle Random Forest":
        forecast, _ = fit_predict_hurdle_random_forest(train, test, params=params, lag_months=lag_months)
        return forecast
    if model_name == "Direct Demand Random Forest":
        return fit_predict_direct_demand_random_forest(train, test, params=params, lag_months=lag_months)
    if model_name == "Aggregate Allocation":
        return forecast_aggregate_allocation(
            train,
            test,
            total_strategy=params.get("total_strategy", "recent_6m_total"),
            allocation_strategy=params.get("allocation_strategy", "hybrid_recent_train_share"),
            descriptor_column=params.get("descriptor_column"),
            within_group_strategy=params.get("within_group_strategy", "probability_weighted_history"),
        )
    if model_name == "Empirical Bayes Allocation":
        return forecast_empirical_bayes_allocation(
            train,
            test,
            total_strategy=params.get("total_strategy", "recent_6m_total"),
            descriptor_column=params.get("descriptor_column"),
            probability_prior_strength=params.get(
                "probability_prior_strength",
                EMPIRICAL_BAYES_DEFAULT_PROBABILITY_PRIOR,
            ),
            size_prior_strength=params.get(
                "size_prior_strength",
                EMPIRICAL_BAYES_DEFAULT_SIZE_PRIOR,
            ),
            seasonality_weight=params.get(
                "seasonality_weight",
                EMPIRICAL_BAYES_DEFAULT_SEASONALITY_WEIGHT,
            ),
            min_probability=params.get("min_probability", HYBRID_DEFAULT_MIN_PROBABILITY),
            top_share=params.get("top_share", HYBRID_DEFAULT_TOP_SHARE),
        )
    if model_name == "Occurrence Gated Allocation":
        model_params = {
            key: value
            for key, value in params.items()
            if key in HURDLE_RF_MODEL_PARAM_KEYS and key != "occurrence_threshold"
        }
        return forecast_occurrence_gated_allocation(
            train,
            test,
            total_strategy=params.get("total_strategy", "recent_6m_total"),
            feature_profile=params.get("feature_profile", HURDLE_RF_DEFAULT_FEATURE_PROFILE),
            score_mode=params.get(
                "score_mode",
                OCCURRENCE_GATED_ALLOCATION_DEFAULT_SCORE_MODE,
            ),
            min_probability=params.get("min_probability", HYBRID_DEFAULT_MIN_PROBABILITY),
            top_share=params.get("top_share", HYBRID_DEFAULT_TOP_SHARE),
            descriptor_column=params.get("descriptor_column"),
            lag_months=lag_months,
            **model_params,
        )
    if model_name == "Zero Forecast":
        return forecast_constant_by_sku(
            train,
            test,
            level_function=zero_level,
            model_name=model_name,
        )
    if model_name == "SKU Train Mean":
        return forecast_constant_by_sku(
            train,
            test,
            level_function=sku_train_mean_level,
            model_name=model_name,
        )
    if model_name == "Recent Mean":
        return forecast_constant_by_sku(
            train,
            test,
            level_function=recent_mean_level,
            model_name=model_name,
            level_kwargs={"recent_months": params.get("recent_months", 6)},
        )
    if model_name == "Last Nonzero":
        return forecast_constant_by_sku(
            train,
            test,
            level_function=last_nonzero_level,
            model_name=model_name,
        )

    raise ValueError(f"Unknown model name: {model_name}")


def no_calibration_result(status):
    return {
        "calibration_multiplier": 1.0,
        "calibration_wmape_percent": np.nan,
        "calibration_status": status,
    }


def calibrate_forecast_with_validation(train, tuning_result, lag_months=FORECAST_LAG_MONTHS):
    fit_train, validation, tuning_status = create_temporal_tuning_split(train)

    if validation.empty or tuning_status != "validation":
        return no_calibration_result(tuning_status)

    try:
        validation_forecast = forecast_model_from_tuning_result(
            fit_train,
            validation,
            tuning_result,
            lag_months=lag_months,
        )
    except Exception as error:
        return no_calibration_result(f"calibration_failed_{type(error).__name__}")

    results = []
    for multiplier in CALIBRATION_MULTIPLIER_GRID:
        candidate = validation_forecast.copy()
        candidate["forecast"] = candidate["forecast"] * multiplier
        results.append(
            {
                "multiplier": multiplier,
                "score": score_candidate_forecast(candidate),
            }
        )

    best = min(results, key=lambda item: item["score"])
    if not np.isfinite(best["score"]):
        return no_calibration_result("calibration_no_finite_score")

    return {
        "calibration_multiplier": best["multiplier"],
        "calibration_wmape_percent": 100 * best["score"],
        "calibration_status": "validation",
    }


def apply_forecast_calibration(forecast, calibration_result):
    forecast = forecast.copy()
    multiplier = calibration_result["calibration_multiplier"]
    forecast["forecast_before_calibration"] = forecast["forecast"]
    forecast["forecast"] = np.clip(forecast["forecast"] * multiplier, 0, None)
    forecast["calibration_multiplier"] = multiplier
    forecast["calibration_wmape_percent"] = calibration_result["calibration_wmape_percent"]
    forecast["calibration_status"] = calibration_result["calibration_status"]
    return forecast


def create_sku_segments(train):
    def positive_mean(series):
        positive = series[series > 0]
        if positive.empty:
            return 0.0
        return positive.mean()

    stats = (
        train
        .groupby(SKU_COLUMN, as_index=False)
        .agg(
            segment_train_months=(TARGET_COLUMN, "size"),
            segment_train_total_demand=(TARGET_COLUMN, "sum"),
            segment_train_positive_months=(TARGET_COLUMN, lambda series: series.gt(0).sum()),
            segment_train_zero_share=(TARGET_COLUMN, lambda series: series.eq(0).mean()),
            segment_train_mean_positive_demand=(TARGET_COLUMN, positive_mean),
        )
    )

    stats["sku_segment"] = "no_train_demand"
    positive_mask = stats["segment_train_total_demand"] > 0

    if positive_mask.sum() >= 3:
        positive_rank = stats.loc[positive_mask, "segment_train_total_demand"].rank(
            method="first",
            pct=True,
        )
        stats.loc[positive_mask & (positive_rank >= 0.80), "sku_segment"] = "high_volume_lumpy"
        stats.loc[positive_mask & (positive_rank < 0.80), "sku_segment"] = "mid_low_volume_lumpy"
    else:
        stats.loc[positive_mask, "sku_segment"] = "positive_lumpy"

    rare_mask = positive_mask & stats["segment_train_zero_share"].ge(0.85)
    stats.loc[rare_mask, "sku_segment"] = "rare_lumpy"

    return stats


def add_sku_segments(forecast, train):
    segments = create_sku_segments(train)
    forecast = forecast.merge(segments, on=SKU_COLUMN, how="left")
    forecast["sku_segment"] = forecast["sku_segment"].fillna("missing_from_train")
    return forecast


def add_fold_metadata(forecast, split):
    forecast = forecast.copy()
    for key, value in split.items():
        forecast[key] = value
    forecast["demand_type"] = LUMPY_DEMAND_TYPE
    return forecast


def add_tuning_metadata(forecast, tuning_result):
    forecast = forecast.copy()
    forecast["tuning_status"] = tuning_result["tuning_status"]
    forecast["tuning_wmape_percent"] = tuning_result["tuning_wmape_percent"]
    forecast["hyperparameters"] = params_to_label(tuning_result["params"])
    return forecast


def run_backtest(data, splits):
    all_forecasts = []
    all_feature_metadata = []
    all_tuning_results = []

    print(
        f"Running {len(splits)} backtest folds across {splits['window_label'].nunique()} windows "
        "with tuned models, aggregate allocation, internal baselines, and validation calibration...",
        flush=True,
    )

    for _, split in splits.iterrows():
        split_dict = split.to_dict()
        lag_months = int(split_dict.get("gap_months", FORECAST_LAG_MONTHS))
        print(
            "Window",
            split_dict["window_label"],
            "| fold",
            split_dict["fold_id"],
            "| lag",
            lag_months,
            "months",
            "| train",
            split_dict["train_start"].date(),
            "to",
            split_dict["train_end"].date(),
            "| test",
            split_dict["test_start"].date(),
            "to",
            split_dict["test_end"].date(),
            flush=True,
        )

        train, test = train_test_for_split(data, split_dict)

        print("  tuning SBA Croston...", flush=True)
        sba_tuning = tune_sba(train)
        sba_forecast = forecast_model_from_tuning_result(train, test, sba_tuning, lag_months=lag_months)

        print("  tuning TSB...", flush=True)
        tsb_tuning = tune_tsb(train)
        tsb_forecast = forecast_model_from_tuning_result(train, test, tsb_tuning, lag_months=lag_months)

        print("  tuning Hurdle Random Forest...", flush=True)
        hurdle_tuning = tune_hurdle_random_forest(train, lag_months=lag_months)
        hurdle_forecast, hurdle_feature_metadata = fit_predict_hurdle_random_forest(
            train,
            test,
            params=hurdle_tuning["params"],
            lag_months=lag_months,
        )

        print("  tuning Direct Demand Random Forest...", flush=True)
        direct_rf_tuning = tune_direct_demand_random_forest(train, lag_months=lag_months)
        direct_rf_forecast = forecast_model_from_tuning_result(train, test, direct_rf_tuning, lag_months=lag_months)

        print("  tuning Aggregate Allocation...", flush=True)
        aggregate_tuning = tune_aggregate_allocation(train)
        aggregate_forecast = forecast_model_from_tuning_result(train, test, aggregate_tuning, lag_months=lag_months)

        print("  tuning Empirical Bayes Allocation...", flush=True)
        eb_allocation_tuning = tune_empirical_bayes_allocation(train)
        eb_allocation_forecast = forecast_model_from_tuning_result(
            train,
            test,
            eb_allocation_tuning,
            lag_months=lag_months,
        )

        print("  tuning Occurrence Gated Allocation...", flush=True)
        occurrence_allocation_tuning = tune_occurrence_gated_allocation(train, lag_months=lag_months)
        occurrence_allocation_forecast = forecast_model_from_tuning_result(
            train,
            test,
            occurrence_allocation_tuning,
            lag_months=lag_months,
        )

        print("  building internal baselines...", flush=True)
        baseline_outputs = [
            (
                forecast_model_from_tuning_result(train, test, baseline_tuning, lag_months=lag_months),
                baseline_tuning,
            )
            for baseline_tuning in build_baseline_tuning_results()
        ]

        model_outputs = [
            (sba_forecast, sba_tuning),
            (tsb_forecast, tsb_tuning),
            (hurdle_forecast, hurdle_tuning),
            (direct_rf_forecast, direct_rf_tuning),
            (aggregate_forecast, aggregate_tuning),
            (eb_allocation_forecast, eb_allocation_tuning),
            (occurrence_allocation_forecast, occurrence_allocation_tuning),
            *baseline_outputs,
        ]

        hurdle_feature_metadata["fold_id"] = split_dict["fold_id"]
        hurdle_feature_metadata["global_fold_id"] = split_dict["global_fold_id"]
        hurdle_feature_metadata["window_label"] = split_dict["window_label"]
        hurdle_feature_metadata["window_months"] = split_dict["window_months"]
        hurdle_feature_metadata["gap_months"] = split_dict["gap_months"]
        hurdle_feature_metadata["hyperparameters"] = params_to_label(hurdle_tuning["params"])
        all_feature_metadata.append(hurdle_feature_metadata)

        for forecast, tuning_result in model_outputs:
            calibration_result = calibrate_forecast_with_validation(train, tuning_result, lag_months=lag_months)
            forecast = apply_forecast_calibration(forecast, calibration_result)
            forecast = add_sku_segments(forecast, train)
            forecast = add_tuning_metadata(forecast, tuning_result)
            all_forecasts.append(add_fold_metadata(forecast, split_dict))

            tuning_row = {
                "model": tuning_result["model"],
                "tuning_status": tuning_result["tuning_status"],
                "tuning_wmape_percent": tuning_result["tuning_wmape_percent"],
                "hyperparameters": params_to_label(tuning_result["params"]),
                **calibration_result,
            }
            tuning_row.update(split_dict)
            all_tuning_results.append(tuning_row)

        print("  fold complete.", flush=True)

    forecasts = pd.concat(all_forecasts, ignore_index=True)
    feature_metadata = (
        pd.concat(all_feature_metadata, ignore_index=True)
        if all_feature_metadata
        else pd.DataFrame()
    )
    tuning_results = pd.DataFrame(all_tuning_results)

    print("Backtest complete.", flush=True)
    return forecasts, feature_metadata, tuning_results


backtest_forecasts, hurdle_feature_metadata, tuning_results_all = run_backtest(
    data=lumpy_model_data,
    splits=backtest_splits,
)

display(backtest_forecasts.head())
display(tuning_results_all)


## Score The Tuned Backtest

This section scores the tuned models overall and by train-window SKU segment.


In [ ]:
def first_or_mixed(series):
    values = series.dropna().astype(str).unique()
    if len(values) == 0:
        return np.nan
    if len(values) == 1:
        return values[0]
    return "mixed"


PREDICTABLE_FORECAST_POPULATION = "known_sku_with_train_demand"
COLD_START_MISSING_POPULATION = "cold_start_missing_from_train"
KNOWN_NO_TRAIN_DEMAND_POPULATION = "known_sku_no_train_demand"

FORECAST_POPULATION_ORDER = [
    PREDICTABLE_FORECAST_POPULATION,
    COLD_START_MISSING_POPULATION,
    KNOWN_NO_TRAIN_DEMAND_POPULATION,
]


def forecast_population_from_segment(segment):
    if pd.isna(segment):
        return COLD_START_MISSING_POPULATION
    if segment == "missing_from_train":
        return COLD_START_MISSING_POPULATION
    if segment == "no_train_demand":
        return KNOWN_NO_TRAIN_DEMAND_POPULATION
    return PREDICTABLE_FORECAST_POPULATION


def add_forecast_population(forecasts):
    forecasts = forecasts.copy()
    if "sku_segment" not in forecasts.columns:
        forecasts["sku_segment"] = "missing_from_train"
    forecasts["forecast_population"] = forecasts["sku_segment"].map(forecast_population_from_segment)
    forecasts["is_predictable_population"] = forecasts["forecast_population"].eq(
        PREDICTABLE_FORECAST_POPULATION
    )
    return forecasts


def summarize_backtest(forecasts):
    fold_group_columns = ["window_label", "window_months", "fold_id", "model"]

    fold_model_summary = (
        forecasts
        .groupby(fold_group_columns, as_index=False)
        .apply(
            lambda group: pd.Series(
                {
                    "wmape": wmape(group[TARGET_COLUMN], group["forecast"]),
                    "wmape_percent": 100 * wmape(group[TARGET_COLUMN], group["forecast"]),
                    "actual_total": group[TARGET_COLUMN].sum(),
                    "forecast_total": group["forecast"].sum(),
                    "absolute_error": np.abs(group[TARGET_COLUMN] - group["forecast"]).sum(),
                    "rows": len(group),
                    "skus": group[SKU_COLUMN].nunique(),
                    "global_fold_id": group["global_fold_id"].iloc[0],
                    "train_months": group["train_months"].iloc[0],
                    "gap_months": group["gap_months"].iloc[0],
                    "test_months": group["test_months"].iloc[0],
                    "history_mode": group["history_mode"].iloc[0],
                    "tuning_status": first_or_mixed(group["tuning_status"]),
                    "hyperparameters": first_or_mixed(group["hyperparameters"]),
                    "calibration_status": first_or_mixed(group["calibration_status"]),
                    "mean_calibration_multiplier": group["calibration_multiplier"].mean(),
                    "median_calibration_multiplier": group["calibration_multiplier"].median(),
                    "test_start": group["test_start"].iloc[0],
                    "test_end": group["test_end"].iloc[0],
                }
            )
        )
        .reset_index(drop=True)
    )

    model_summary = (
        fold_model_summary
        .groupby(["window_label", "window_months", "model"], as_index=False)
        .agg(
            mean_wmape_percent=("wmape_percent", "mean"),
            median_wmape_percent=("wmape_percent", "median"),
            best_fold_wmape_percent=("wmape_percent", "min"),
            worst_fold_wmape_percent=("wmape_percent", "max"),
            folds=("fold_id", "nunique"),
            train_months=("train_months", "first"),
            history_mode=("history_mode", "first"),
            tuning_status=("tuning_status", first_or_mixed),
            hyperparameters=("hyperparameters", first_or_mixed),
            calibration_status=("calibration_status", first_or_mixed),
            mean_calibration_multiplier=("mean_calibration_multiplier", "mean"),
            median_calibration_multiplier=("median_calibration_multiplier", "median"),
            actual_total=("actual_total", "sum"),
            forecast_total=("forecast_total", "sum"),
            absolute_error=("absolute_error", "sum"),
        )
    )
    model_summary["pooled_wmape_percent"] = (
        100 * model_summary["absolute_error"] / model_summary["actual_total"].replace(0, np.nan)
    )
    model_summary["below_70_percent_target"] = model_summary["pooled_wmape_percent"] < 70
    model_summary["below_50_percent_stretch"] = model_summary["pooled_wmape_percent"] < 50
    model_summary = model_summary.sort_values(["window_months", "pooled_wmape_percent"])

    return fold_model_summary, model_summary


def monthly_backtest_results(forecasts):
    monthly = (
        forecasts
        .groupby(["window_label", "window_months", "fold_id", "model", DATE_COLUMN], as_index=False)
        .agg(
            actual=(TARGET_COLUMN, "sum"),
            forecast=("forecast", "sum"),
            global_fold_id=("global_fold_id", "first"),
        )
        .sort_values(["window_months", "fold_id", "model", DATE_COLUMN])
    )

    monthly["absolute_error"] = np.abs(monthly["actual"] - monthly["forecast"])
    monthly["monthly_wmape_percent"] = (
        100 * monthly["absolute_error"] / monthly["actual"].replace(0, np.nan)
    )

    rolling_group_columns = ["window_label", "window_months", "fold_id", "model"]
    monthly["rolling_3m_abs_error"] = (
        monthly
        .groupby(rolling_group_columns)["absolute_error"]
        .transform(lambda series: series.rolling(3, min_periods=1).sum())
    )
    monthly["rolling_3m_actual"] = (
        monthly
        .groupby(rolling_group_columns)["actual"]
        .transform(lambda series: series.rolling(3, min_periods=1).sum())
    )
    monthly["rolling_3m_wmape_percent"] = (
        100
        * monthly["rolling_3m_abs_error"]
        / monthly["rolling_3m_actual"].replace(0, np.nan)
    )
    monthly["rolling_3m_average_monthly_wmape_percent"] = (
        monthly
        .groupby(rolling_group_columns)["monthly_wmape_percent"]
        .transform(lambda series: series.rolling(3, min_periods=1).mean())
    )

    return monthly



FORECAST_VARIANT_LABELS = {
    "post_calibrated": "forecast",
    "pre_calibration": "forecast_before_calibration",
    "operational_floor": "forecast_operational_floor",
}


def attach_lag_metadata(summary, splits):
    if "gap_months" in summary.columns:
        return summary
    lag_lookup = (
        splits[["window_label", "window_months", "gap_months"]]
        .drop_duplicates()
        .copy()
    )
    return summary.merge(
        lag_lookup,
        on=["window_label", "window_months"],
        how="left",
    )


def add_forecast_variant_columns(forecasts):
    scored = forecasts.copy()
    if "forecast_before_calibration" not in scored.columns:
        scored["forecast_before_calibration"] = scored["forecast"]
    scored["forecast_before_calibration"] = scored["forecast_before_calibration"].fillna(scored["forecast"])

    if "calibration_multiplier" in scored.columns:
        multiplier = scored["calibration_multiplier"].fillna(1.0).astype(float)
    else:
        multiplier = pd.Series(1.0, index=scored.index)

    operational_multiplier = multiplier.clip(lower=OPERATIONAL_CALIBRATION_MIN_MULTIPLIER)
    scored["forecast_operational_floor"] = (
        scored["forecast_before_calibration"] * operational_multiplier
    ).clip(lower=0)

    zero_mask = scored["model"].eq("Zero Forecast")
    scored.loc[zero_mask, "forecast_operational_floor"] = scored.loc[zero_mask, "forecast"]
    return scored


def summarize_monthly_total_backtest(forecasts, extra_group_columns=None):
    extra_group_columns = extra_group_columns or []
    scored = add_forecast_variant_columns(forecasts)

    base_group_columns = [
        "window_label",
        "window_months",
        "fold_id",
        "model",
    ] + extra_group_columns
    monthly_group_columns = base_group_columns + [DATE_COLUMN]

    monthly = (
        scored
        .groupby(monthly_group_columns, as_index=False)
        .agg(
            actual=(TARGET_COLUMN, "sum"),
            post_calibrated_forecast=("forecast", "sum"),
            pre_calibration_forecast=("forecast_before_calibration", "sum"),
            operational_floor_forecast=("forecast_operational_floor", "sum"),
            global_fold_id=("global_fold_id", "first"),
        )
        .sort_values(["window_months", "fold_id", "model", DATE_COLUMN])
    )

    for variant_name in FORECAST_VARIANT_LABELS:
        forecast_column = f"{variant_name}_forecast"
        monthly[f"{variant_name}_absolute_error"] = np.abs(
            monthly["actual"] - monthly[forecast_column]
        )
        monthly[f"{variant_name}_monthly_total_wmape_percent"] = (
            100
            * monthly[f"{variant_name}_absolute_error"]
            / monthly["actual"].replace(0, np.nan)
        )

    fold_rows = []
    for keys, group in monthly.groupby(base_group_columns, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(base_group_columns, keys))
        row["actual_total"] = group["actual"].sum()
        row["months"] = group[DATE_COLUMN].nunique()
        row["global_fold_id"] = group["global_fold_id"].iloc[0]
        for variant_name in FORECAST_VARIANT_LABELS:
            row[f"{variant_name}_forecast_total"] = group[f"{variant_name}_forecast"].sum()
            row[f"{variant_name}_absolute_error"] = group[f"{variant_name}_absolute_error"].sum()
            row[f"{variant_name}_monthly_total_wmape_percent"] = (
                100
                * row[f"{variant_name}_absolute_error"]
                / row["actual_total"]
                if row["actual_total"] > 0
                else np.nan
            )
        fold_rows.append(row)

    fold_summary = pd.DataFrame(fold_rows)
    model_group_columns = [column for column in base_group_columns if column != "fold_id"]
    aggregation = {
        "fold_id": "nunique",
        "actual_total": "sum",
        "months": "sum",
    }
    for variant_name in FORECAST_VARIANT_LABELS:
        aggregation[f"{variant_name}_forecast_total"] = "sum"
        aggregation[f"{variant_name}_absolute_error"] = "sum"

    model_summary = (
        fold_summary
        .groupby(model_group_columns, as_index=False)
        .agg(aggregation)
        .rename(columns={"fold_id": "folds"})
    )
    for variant_name in FORECAST_VARIANT_LABELS:
        model_summary[f"{variant_name}_monthly_total_wmape_percent"] = (
            100
            * model_summary[f"{variant_name}_absolute_error"]
            / model_summary["actual_total"].replace(0, np.nan)
        )

    sort_columns = ["window_months"]
    if extra_group_columns:
        sort_columns += extra_group_columns
    sort_columns += ["post_calibrated_monthly_total_wmape_percent", "model"]

    return (
        monthly,
        fold_summary.sort_values(sort_columns),
        model_summary.sort_values(sort_columns),
    )


def segment_backtest_results(forecasts):
    segment_fold_summary = (
        forecasts
        .groupby(["window_label", "window_months", "fold_id", "model", "sku_segment"], as_index=False)
        .apply(
            lambda group: pd.Series(
                {
                    "forecast_population": group["forecast_population"].iloc[0],
                    "wmape_percent": 100 * wmape(group[TARGET_COLUMN], group["forecast"]),
                    "actual_total": group[TARGET_COLUMN].sum(),
                    "forecast_total": group["forecast"].sum(),
                    "absolute_error": np.abs(group[TARGET_COLUMN] - group["forecast"]).sum(),
                    "skus": group[SKU_COLUMN].nunique(),
                    "rows": len(group),
                }
            )
        )
        .reset_index(drop=True)
    )

    segment_model_summary = (
        segment_fold_summary
        .groupby(["window_label", "window_months", "model", "sku_segment", "forecast_population"], as_index=False)
        .agg(
            folds=("fold_id", "nunique"),
            actual_total=("actual_total", "sum"),
            forecast_total=("forecast_total", "sum"),
            absolute_error=("absolute_error", "sum"),
            skus=("skus", "max"),
            rows=("rows", "sum"),
        )
    )
    segment_model_summary["pooled_wmape_percent"] = (
        100
        * segment_model_summary["absolute_error"]
        / segment_model_summary["actual_total"].replace(0, np.nan)
    )
    segment_model_summary["segment_selection_score"] = np.where(
        segment_model_summary["actual_total"].gt(0),
        segment_model_summary["pooled_wmape_percent"],
        segment_model_summary["absolute_error"],
    )
    segment_model_summary = segment_model_summary.sort_values(
        ["window_months", "sku_segment", "segment_selection_score", "absolute_error"]
    )

    return segment_fold_summary, segment_model_summary


def population_backtest_results(forecasts):
    population_fold_summary = (
        forecasts
        .groupby(["window_label", "window_months", "fold_id", "model", "forecast_population"], as_index=False)
        .apply(
            lambda group: pd.Series(
                {
                    "wmape_percent": 100 * wmape(group[TARGET_COLUMN], group["forecast"]),
                    "actual_total": group[TARGET_COLUMN].sum(),
                    "forecast_total": group["forecast"].sum(),
                    "absolute_error": np.abs(group[TARGET_COLUMN] - group["forecast"]).sum(),
                    "skus": group[SKU_COLUMN].nunique(),
                    "rows": len(group),
                }
            )
        )
        .reset_index(drop=True)
    )

    population_model_summary = (
        population_fold_summary
        .groupby(["window_label", "window_months", "model", "forecast_population"], as_index=False)
        .agg(
            folds=("fold_id", "nunique"),
            actual_total=("actual_total", "sum"),
            forecast_total=("forecast_total", "sum"),
            absolute_error=("absolute_error", "sum"),
            skus=("skus", "max"),
            rows=("rows", "sum"),
        )
    )
    population_model_summary["pooled_wmape_percent"] = (
        100
        * population_model_summary["absolute_error"]
        / population_model_summary["actual_total"].replace(0, np.nan)
    )
    population_model_summary["population_selection_score"] = np.where(
        population_model_summary["actual_total"].gt(0),
        population_model_summary["pooled_wmape_percent"],
        population_model_summary["absolute_error"],
    )
    population_model_summary["population_sort_order"] = population_model_summary[
        "forecast_population"
    ].map({population: index for index, population in enumerate(FORECAST_POPULATION_ORDER)})
    population_model_summary["population_sort_order"] = population_model_summary[
        "population_sort_order"
    ].fillna(99)
    population_model_summary = population_model_summary.sort_values(
        [
            "window_months",
            "population_sort_order",
            "forecast_population",
            "population_selection_score",
            "absolute_error",
        ]
    ).drop(columns=["population_sort_order"])

    return population_fold_summary, population_model_summary


def sku_horizon_variant_results(forecasts, extra_group_columns=None):
    extra_group_columns = extra_group_columns or []
    scored = add_forecast_variant_columns(forecasts)
    base_group_columns = [
        "window_label",
        "window_months",
        "fold_id",
        "model",
    ] + extra_group_columns
    sku_group_columns = base_group_columns + [SKU_COLUMN]

    variant_frames = []
    for variant_name, forecast_column in FORECAST_VARIANT_LABELS.items():
        sku_totals = (
            scored
            .groupby(sku_group_columns, as_index=False, dropna=False)
            .agg(
                actual_total=(TARGET_COLUMN, "sum"),
                forecast_total=(forecast_column, "sum"),
                global_fold_id=("global_fold_id", "first"),
            )
        )
        sku_totals["forecast_variant"] = variant_name
        sku_totals["absolute_error"] = np.abs(
            sku_totals["actual_total"] - sku_totals["forecast_total"]
        )
        sku_totals["positive_actual_sku"] = sku_totals["actual_total"].gt(0)
        sku_totals["forecasted_sku"] = sku_totals["forecast_total"].gt(1e-9)
        sku_totals["sku_horizon_wmape_percent"] = (
            100
            * sku_totals["absolute_error"]
            / sku_totals["actual_total"].replace(0, np.nan)
        )
        variant_frames.append(sku_totals)

    return pd.concat(variant_frames, ignore_index=True)


def summarize_sku_horizon_results(sku_horizon_results, extra_group_columns=None):
    extra_group_columns = extra_group_columns or []
    fold_group_columns = [
        "window_label",
        "window_months",
        "fold_id",
        "model",
        "forecast_variant",
    ] + extra_group_columns

    def summarize_group(group):
        positive_mask = group["actual_total"].gt(0)
        zero_mask = ~positive_mask
        actual_total = group["actual_total"].sum()
        forecast_total = group["forecast_total"].sum()
        absolute_error = group["absolute_error"].sum()
        positive_sku_abs_error = group.loc[positive_mask, "absolute_error"].sum()
        zero_actual_forecast_total = group.loc[zero_mask, "forecast_total"].sum()
        positive_rows = int(positive_mask.sum())
        zero_rows = int(zero_mask.sum())
        forecasted_zero_rows = int((zero_mask & group["forecasted_sku"]).sum())
        below_70 = int((positive_mask & group["sku_horizon_wmape_percent"].lt(70)).sum())
        below_100 = int((positive_mask & group["sku_horizon_wmape_percent"].lt(100)).sum())
        actual_below_70 = group.loc[
            positive_mask & group["sku_horizon_wmape_percent"].lt(70),
            "actual_total",
        ].sum()
        actual_below_100 = group.loc[
            positive_mask & group["sku_horizon_wmape_percent"].lt(100),
            "actual_total",
        ].sum()
        return pd.Series(
            {
                "actual_total": actual_total,
                "forecast_total": forecast_total,
                "absolute_error": absolute_error,
                "sku_fold_rows": len(group),
                "positive_actual_sku_fold_rows": positive_rows,
                "zero_actual_sku_fold_rows": zero_rows,
                "forecasted_zero_actual_sku_fold_rows": forecasted_zero_rows,
                "positive_sku_abs_error": positive_sku_abs_error,
                "zero_actual_sku_forecast_total": zero_actual_forecast_total,
                "positive_actual_sku_fold_rows_below_70": below_70,
                "positive_actual_sku_fold_rows_below_100": below_100,
                "positive_actual_total_below_70": actual_below_70,
                "positive_actual_total_below_100": actual_below_100,
                "global_fold_id": group["global_fold_id"].iloc[0],
            }
        )

    fold_summary = (
        sku_horizon_results
        .groupby(fold_group_columns, as_index=False, dropna=False)
        .apply(summarize_group)
        .reset_index(drop=True)
    )

    fold_summary["horizon_sku_wmape_percent"] = (
        100
        * fold_summary["absolute_error"]
        / fold_summary["actual_total"].replace(0, np.nan)
    )
    fold_summary["positive_sku_only_wmape_percent"] = (
        100
        * fold_summary["positive_sku_abs_error"]
        / fold_summary["actual_total"].replace(0, np.nan)
    )
    fold_summary["zero_actual_sku_forecast_share_percent"] = (
        100
        * fold_summary["zero_actual_sku_forecast_total"]
        / fold_summary["forecast_total"].replace(0, np.nan)
    )
    fold_summary["positive_actual_sku_below_70_share_percent"] = (
        100
        * fold_summary["positive_actual_sku_fold_rows_below_70"]
        / fold_summary["positive_actual_sku_fold_rows"].replace(0, np.nan)
    )
    fold_summary["positive_actual_sku_below_100_share_percent"] = (
        100
        * fold_summary["positive_actual_sku_fold_rows_below_100"]
        / fold_summary["positive_actual_sku_fold_rows"].replace(0, np.nan)
    )
    fold_summary["positive_actual_below_70_share_percent"] = (
        100
        * fold_summary["positive_actual_total_below_70"]
        / fold_summary["actual_total"].replace(0, np.nan)
    )
    fold_summary["positive_actual_below_100_share_percent"] = (
        100
        * fold_summary["positive_actual_total_below_100"]
        / fold_summary["actual_total"].replace(0, np.nan)
    )

    model_group_columns = [
        column for column in fold_group_columns if column != "fold_id"
    ]
    aggregation = {
        "fold_id": "nunique",
        "actual_total": "sum",
        "forecast_total": "sum",
        "absolute_error": "sum",
        "sku_fold_rows": "sum",
        "positive_actual_sku_fold_rows": "sum",
        "zero_actual_sku_fold_rows": "sum",
        "forecasted_zero_actual_sku_fold_rows": "sum",
        "positive_sku_abs_error": "sum",
        "zero_actual_sku_forecast_total": "sum",
        "positive_actual_sku_fold_rows_below_70": "sum",
        "positive_actual_sku_fold_rows_below_100": "sum",
        "positive_actual_total_below_70": "sum",
        "positive_actual_total_below_100": "sum",
    }
    model_summary = (
        fold_summary
        .groupby(model_group_columns, as_index=False, dropna=False)
        .agg(aggregation)
        .rename(columns={"fold_id": "folds"})
    )
    model_summary["horizon_sku_wmape_percent"] = (
        100
        * model_summary["absolute_error"]
        / model_summary["actual_total"].replace(0, np.nan)
    )
    model_summary["positive_sku_only_wmape_percent"] = (
        100
        * model_summary["positive_sku_abs_error"]
        / model_summary["actual_total"].replace(0, np.nan)
    )
    model_summary["zero_actual_sku_forecast_share_percent"] = (
        100
        * model_summary["zero_actual_sku_forecast_total"]
        / model_summary["forecast_total"].replace(0, np.nan)
    )
    model_summary["positive_actual_sku_below_70_share_percent"] = (
        100
        * model_summary["positive_actual_sku_fold_rows_below_70"]
        / model_summary["positive_actual_sku_fold_rows"].replace(0, np.nan)
    )
    model_summary["positive_actual_sku_below_100_share_percent"] = (
        100
        * model_summary["positive_actual_sku_fold_rows_below_100"]
        / model_summary["positive_actual_sku_fold_rows"].replace(0, np.nan)
    )
    model_summary["positive_actual_below_70_share_percent"] = (
        100
        * model_summary["positive_actual_total_below_70"]
        / model_summary["actual_total"].replace(0, np.nan)
    )
    model_summary["positive_actual_below_100_share_percent"] = (
        100
        * model_summary["positive_actual_total_below_100"]
        / model_summary["actual_total"].replace(0, np.nan)
    )
    model_summary["below_70_percent_target"] = model_summary["horizon_sku_wmape_percent"] < 70
    model_summary["below_50_percent_stretch"] = model_summary["horizon_sku_wmape_percent"] < 50

    sort_columns = ["window_months"]
    if extra_group_columns:
        sort_columns += extra_group_columns
    sort_columns += ["horizon_sku_wmape_percent", "model", "forecast_variant"]

    return (
        fold_summary.sort_values(sort_columns),
        model_summary.sort_values(sort_columns),
    )


def build_mase_scale_lookup(data, splits, lag_months=MASE_NAIVE_LAG_MONTHS):
    rows = []

    for _, split in splits.iterrows():
        split_dict = split.to_dict()
        train, test = train_test_for_split(data, split_dict)
        sku_scales = {}
        portfolio_diffs = []

        for sku, sku_train in train.sort_values(DATE_COLUMN).groupby(SKU_COLUMN, sort=False):
            values = sku_train[TARGET_COLUMN].astype(float).to_numpy()
            if len(values) > lag_months:
                diffs = np.abs(values[lag_months:] - values[:-lag_months])
                diffs = diffs[np.isfinite(diffs)]
            else:
                diffs = np.array([], dtype=float)

            if len(diffs):
                portfolio_diffs.extend(diffs.tolist())
                scale = float(diffs.mean())
            else:
                scale = np.nan

            sku_scales[sku] = scale

        positive_train = train.loc[train[TARGET_COLUMN].gt(0), TARGET_COLUMN].astype(float)
        fallback_scale = (
            float(np.mean(portfolio_diffs))
            if len(portfolio_diffs) and np.mean(portfolio_diffs) > 0
            else float(positive_train.mean())
            if len(positive_train)
            else float(train[TARGET_COLUMN].abs().mean())
            if len(train)
            else 1.0
        )
        if not np.isfinite(fallback_scale) or fallback_scale <= 0:
            fallback_scale = 1.0

        test_skus = pd.Index(test[SKU_COLUMN].dropna().unique())
        for sku in test_skus:
            scale = sku_scales.get(sku, np.nan)
            if pd.isna(scale) or scale <= 0:
                scale = fallback_scale
                source = "portfolio_fallback"
            else:
                source = "sku_naive_lag"
            rows.append(
                {
                    "global_fold_id": split_dict["global_fold_id"],
                    SKU_COLUMN: sku,
                    "mase_scale": float(scale),
                    "mase_scale_source": source,
                }
            )

    return pd.DataFrame(rows)


def lumpy_metric_variant_results(forecasts, data, splits):
    scored = add_forecast_variant_columns(forecasts)
    scale_lookup = build_mase_scale_lookup(data, splits)
    scored = scored.merge(
        scale_lookup,
        on=["global_fold_id", SKU_COLUMN],
        how="left",
    )
    fallback_scale = scored["mase_scale"].dropna()
    fallback_scale = float(fallback_scale.median()) if len(fallback_scale) else 1.0
    if not np.isfinite(fallback_scale) or fallback_scale <= 0:
        fallback_scale = 1.0
    scored["mase_scale"] = scored["mase_scale"].fillna(fallback_scale).clip(lower=1e-9)
    scored["mase_scale_source"] = scored["mase_scale_source"].fillna("global_fallback")

    variant_frames = []
    base_columns = [
        SKU_COLUMN,
        DATE_COLUMN,
        TARGET_COLUMN,
        "model",
        "window_label",
        "window_months",
        "fold_id",
        "global_fold_id",
        "forecast_population",
        "sku_segment",
        "mase_scale",
        "mase_scale_source",
    ]

    for variant_name, forecast_column in FORECAST_VARIANT_LABELS.items():
        frame = scored[base_columns].copy()
        frame["forecast_variant"] = variant_name
        frame["variant_forecast"] = scored[forecast_column].fillna(scored["forecast"]).clip(lower=0)
        frame["absolute_error"] = np.abs(frame[TARGET_COLUMN] - frame["variant_forecast"])
        frame["under_forecast_quantity"] = (frame[TARGET_COLUMN] - frame["variant_forecast"]).clip(lower=0)
        frame["over_forecast_quantity"] = (frame["variant_forecast"] - frame[TARGET_COLUMN]).clip(lower=0)
        frame["scaled_absolute_error"] = frame["absolute_error"] / frame["mase_scale"]
        frame["inventory_utility_cost"] = (
            INVENTORY_HOLDING_COST_WEIGHT * frame["over_forecast_quantity"]
            + INVENTORY_STOCKOUT_COST_WEIGHT * frame["under_forecast_quantity"]
        )
        frame["actual_positive"] = frame[TARGET_COLUMN].gt(0)
        frame["forecast_positive"] = frame["variant_forecast"].gt(1e-9)
        frame["positive_service_hit"] = frame["actual_positive"] & (
            frame["variant_forecast"] >= frame[TARGET_COLUMN]
        )
        variant_frames.append(frame)

    return pd.concat(variant_frames, ignore_index=True)


def summarize_lumpy_metric_suite(metric_results, extra_group_columns=None):
    extra_group_columns = extra_group_columns or []
    fold_group_columns = [
        "window_label",
        "window_months",
        "fold_id",
        "model",
        "forecast_variant",
    ] + extra_group_columns

    def summarize_group(group):
        actual_total = group[TARGET_COLUMN].sum()
        forecast_total = group["variant_forecast"].sum()
        absolute_error = group["absolute_error"].sum()
        under_forecast = group["under_forecast_quantity"].sum()
        over_forecast = group["over_forecast_quantity"].sum()
        positive_actual_rows = int(group["actual_positive"].sum())
        forecast_positive_rows = int(group["forecast_positive"].sum())
        true_positive_forecast_rows = int((group["actual_positive"] & group["forecast_positive"]).sum())
        positive_service_hits = int(group["positive_service_hit"].sum())
        scaled_errors = group["scaled_absolute_error"].replace([np.inf, -np.inf], np.nan).dropna()
        return pd.Series(
            {
                "actual_total": actual_total,
                "forecast_total": forecast_total,
                "absolute_error": absolute_error,
                "under_forecast_quantity": under_forecast,
                "over_forecast_quantity": over_forecast,
                "inventory_utility_cost": group["inventory_utility_cost"].sum(),
                "scaled_absolute_error_sum": scaled_errors.sum(),
                "scaled_absolute_error_rows": len(scaled_errors),
                "rows": len(group),
                "positive_actual_rows": positive_actual_rows,
                "forecast_positive_rows": forecast_positive_rows,
                "true_positive_forecast_rows": true_positive_forecast_rows,
                "positive_service_hits": positive_service_hits,
                "global_fold_id": group["global_fold_id"].iloc[0],
            }
        )

    fold_summary = (
        metric_results
        .groupby(fold_group_columns, as_index=False, dropna=False)
        .apply(summarize_group)
        .reset_index(drop=True)
    )

    model_group_columns = [
        column for column in fold_group_columns if column != "fold_id"
    ]
    model_summary = (
        fold_summary
        .groupby(model_group_columns, as_index=False, dropna=False)
        .agg(
            folds=("fold_id", "nunique"),
            actual_total=("actual_total", "sum"),
            forecast_total=("forecast_total", "sum"),
            absolute_error=("absolute_error", "sum"),
            under_forecast_quantity=("under_forecast_quantity", "sum"),
            over_forecast_quantity=("over_forecast_quantity", "sum"),
            inventory_utility_cost=("inventory_utility_cost", "sum"),
            scaled_absolute_error_sum=("scaled_absolute_error_sum", "sum"),
            scaled_absolute_error_rows=("scaled_absolute_error_rows", "sum"),
            rows=("rows", "sum"),
            positive_actual_rows=("positive_actual_rows", "sum"),
            forecast_positive_rows=("forecast_positive_rows", "sum"),
            true_positive_forecast_rows=("true_positive_forecast_rows", "sum"),
            positive_service_hits=("positive_service_hits", "sum"),
        )
    )

    for frame in [fold_summary, model_summary]:
        frame["wmape_percent"] = (
            100 * frame["absolute_error"] / frame["actual_total"].replace(0, np.nan)
        )
        frame["mase"] = (
            frame["scaled_absolute_error_sum"]
            / frame["scaled_absolute_error_rows"].replace(0, np.nan)
        )
        frame["fill_rate_proxy_percent"] = (
            100
            * (1 - frame["under_forecast_quantity"] / frame["actual_total"].replace(0, np.nan))
        )
        frame["over_forecast_to_actual_percent"] = (
            100 * frame["over_forecast_quantity"] / frame["actual_total"].replace(0, np.nan)
        )
        frame["inventory_utility_cost_per_actual"] = (
            frame["inventory_utility_cost"] / frame["actual_total"].replace(0, np.nan)
        )
        frame["positive_row_recall_percent"] = (
            100
            * frame["true_positive_forecast_rows"]
            / frame["positive_actual_rows"].replace(0, np.nan)
        )
        frame["positive_forecast_precision_percent"] = (
            100
            * frame["true_positive_forecast_rows"]
            / frame["forecast_positive_rows"].replace(0, np.nan)
        )
        frame["positive_row_service_level_proxy_percent"] = (
            100
            * frame["positive_service_hits"]
            / frame["positive_actual_rows"].replace(0, np.nan)
        )

    sort_columns = ["window_months"]
    if extra_group_columns:
        sort_columns += extra_group_columns
    sort_columns += ["inventory_utility_cost_per_actual", "mase", "wmape_percent", "model", "forecast_variant"]

    return (
        fold_summary.sort_values(sort_columns),
        model_summary.sort_values(sort_columns),
    )


def build_segment_winner_forecasts(forecasts, best_by_segment):
    keys = ["window_label", "window_months", "sku_segment", "model"]
    winners = best_by_segment[keys].copy()
    selected = forecasts.merge(winners, on=keys, how="inner")
    selected = selected.copy()
    selected["source_model"] = selected["model"]
    selected["model"] = "Segment Winner Blend"
    return selected


def build_population_winner_forecasts(forecasts, best_by_population):
    keys = ["window_label", "window_months", "forecast_population", "model"]
    winners = best_by_population[keys].copy()
    selected = forecasts.merge(winners, on=keys, how="inner")
    selected = selected.copy()
    selected["source_model"] = selected["model"]
    selected["model"] = "Population Winner Blend"
    return selected


backtest_forecasts = add_forecast_population(backtest_forecasts)

segment_fold_summary, segment_model_summary = segment_backtest_results(backtest_forecasts)

best_model_by_segment = (
    segment_model_summary
    .sort_values(["window_months", "sku_segment", "segment_selection_score", "absolute_error"])
    .groupby(["window_label", "window_months", "sku_segment"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

segment_winner_forecasts = build_segment_winner_forecasts(
    backtest_forecasts,
    best_model_by_segment,
)
segment_winner_fold_summary, segment_winner_summary = summarize_backtest(segment_winner_forecasts)
segment_winner_summary["selection_method"] = "best_backtest_model_by_sku_segment"

population_fold_summary, population_model_summary = population_backtest_results(backtest_forecasts)

best_model_by_population = (
    population_model_summary
    .sort_values(
        [
            "window_months",
            "forecast_population",
            "population_selection_score",
            "absolute_error",
        ]
    )
    .groupby(["window_label", "window_months", "forecast_population"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

population_winner_forecasts = build_population_winner_forecasts(
    backtest_forecasts,
    best_model_by_population,
)
population_winner_fold_summary, population_winner_summary = summarize_backtest(
    population_winner_forecasts
)
population_winner_summary["selection_method"] = "best_backtest_model_by_forecast_population"

evaluation_forecasts = pd.concat(
    [backtest_forecasts, segment_winner_forecasts, population_winner_forecasts],
    ignore_index=True,
    sort=False,
)

fold_model_summary, model_summary = summarize_backtest(evaluation_forecasts)
monthly_results = monthly_backtest_results(evaluation_forecasts)
monthly_total_results, monthly_total_fold_summary, monthly_total_model_summary = summarize_monthly_total_backtest(
    evaluation_forecasts
)

sku_horizon_results = sku_horizon_variant_results(evaluation_forecasts)
sku_horizon_fold_summary, sku_horizon_model_summary = summarize_sku_horizon_results(
    sku_horizon_results
)
(
    sku_horizon_population_fold_summary,
    sku_horizon_population_summary,
) = summarize_sku_horizon_results(
    sku_horizon_variant_results(
        evaluation_forecasts,
        extra_group_columns=["forecast_population"],
    ),
    extra_group_columns=["forecast_population"],
)
(
    sku_horizon_segment_fold_summary,
    sku_horizon_segment_summary,
) = summarize_sku_horizon_results(
    sku_horizon_variant_results(
        evaluation_forecasts,
        extra_group_columns=["sku_segment", "forecast_population"],
    ),
    extra_group_columns=["sku_segment", "forecast_population"],
)

best_sku_horizon_model_by_window = (
    sku_horizon_model_summary
    .sort_values(["window_months", "horizon_sku_wmape_percent", "absolute_error"])
    .groupby(["window_label", "window_months"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)


metric_suite_results = lumpy_metric_variant_results(
    evaluation_forecasts,
    lumpy_model_data,
    backtest_splits,
)
metric_suite_fold_summary, metric_suite_model_summary = summarize_lumpy_metric_suite(
    metric_suite_results
)
(
    metric_suite_population_fold_summary,
    metric_suite_population_summary,
) = summarize_lumpy_metric_suite(
    metric_suite_results,
    extra_group_columns=["forecast_population"],
)

for _summary_name in [
    "sku_horizon_fold_summary",
    "sku_horizon_model_summary",
    "sku_horizon_population_summary",
    "sku_horizon_segment_summary",
    "metric_suite_fold_summary",
    "metric_suite_model_summary",
    "metric_suite_population_summary",
]:
    globals()[_summary_name] = attach_lag_metadata(
        globals()[_summary_name],
        backtest_splits,
    )

(
    monthly_total_population_results,
    monthly_total_population_fold_summary,
    monthly_total_population_summary,
) = summarize_monthly_total_backtest(
    evaluation_forecasts,
    extra_group_columns=["forecast_population"],
)
evaluation_population_fold_summary, evaluation_population_model_summary = population_backtest_results(
    evaluation_forecasts
)

known_history_model_summary = population_model_summary.loc[
    population_model_summary["forecast_population"].eq(PREDICTABLE_FORECAST_POPULATION)
].copy()

best_known_history_by_window = (
    known_history_model_summary
    .sort_values(["window_months", "population_selection_score", "absolute_error"])
    .groupby(["window_label", "window_months"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

best_model_by_window = (
    model_summary
    .sort_values(["window_months", "pooled_wmape_percent"])
    .groupby(["window_label", "window_months"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

display(model_summary)
display(best_model_by_window)
display(known_history_model_summary)
display(best_known_history_by_window)
display(population_model_summary)
display(best_model_by_population)
display(population_winner_summary)
display(monthly_total_model_summary)
display(sku_horizon_model_summary)
display(best_sku_horizon_model_by_window)
display(metric_suite_model_summary)
display(metric_suite_population_summary)
display(sku_horizon_population_summary)
display(sku_horizon_segment_summary)
display(monthly_total_population_summary)
display(segment_model_summary)
display(best_model_by_segment)
display(segment_winner_summary)
display(fold_model_summary)
display(monthly_results.head(30))


## Forecast Agents


In [ ]:
def safe_percent(numerator, denominator):
    if denominator == 0 or pd.isna(denominator):
        return np.nan
    return 100 * numerator / denominator


def wmape_percent_for_column(data, forecast_column):
    if forecast_column not in data.columns:
        return np.nan
    return 100 * wmape(data[TARGET_COLUMN], data[forecast_column])


def run_data_quality_agent(data, splits):
    summary_rows = []
    segment_rows = []

    for _, split in splits.iterrows():
        split_dict = split.to_dict()
        train, test = train_test_for_split(data, split_dict)

        train_sku_demand = train.groupby(SKU_COLUMN)[TARGET_COLUMN].sum()
        test_sku_demand = test.groupby(SKU_COLUMN)[TARGET_COLUMN].sum()

        train_skus = set(train_sku_demand.index)
        train_positive_skus = set(train_sku_demand.loc[train_sku_demand > 0].index)
        test_positive_skus = set(test_sku_demand.loc[test_sku_demand > 0].index)

        actual_total = test[TARGET_COLUMN].sum()
        missing_from_train_actual = test_sku_demand.loc[
            [sku for sku in test_sku_demand.index if sku not in train_skus]
        ].sum() if len(test_sku_demand) else 0.0
        no_train_demand_actual = test_sku_demand.loc[
            [sku for sku in test_sku_demand.index if sku in train_skus and sku not in train_positive_skus]
        ].sum() if len(test_sku_demand) else 0.0

        summary_rows.append(
            {
                "agent": "Data Quality Agent",
                "window_label": split_dict["window_label"],
                "window_months": split_dict["window_months"],
                "fold_id": split_dict["fold_id"],
                "train_months": split_dict["train_months"],
                "test_months": split_dict["test_months"],
                "history_mode": split_dict["history_mode"],
                "train_skus": len(train_skus),
                "test_skus": test[SKU_COLUMN].nunique(),
                "train_positive_skus": len(train_positive_skus),
                "test_positive_skus": len(test_positive_skus),
                "test_new_positive_skus": len([sku for sku in test_positive_skus if sku not in train_skus]),
                "test_positive_skus_without_train_demand": len(
                    [sku for sku in test_positive_skus if sku in train_skus and sku not in train_positive_skus]
                ),
                "test_actual_total": actual_total,
                "test_actual_from_missing_train_skus": missing_from_train_actual,
                "test_actual_from_no_train_demand_skus": no_train_demand_actual,
                "missing_train_actual_share_percent": safe_percent(missing_from_train_actual, actual_total),
                "no_train_demand_actual_share_percent": safe_percent(no_train_demand_actual, actual_total),
                "train_zero_month_share": train[TARGET_COLUMN].eq(0).mean(),
                "test_zero_month_share": test[TARGET_COLUMN].eq(0).mean(),
            }
        )

        segment_frame = test[[SKU_COLUMN, DATE_COLUMN, TARGET_COLUMN]].merge(
            create_sku_segments(train),
            on=SKU_COLUMN,
            how="left",
        )
        segment_frame["sku_segment"] = segment_frame["sku_segment"].fillna("missing_from_train")
        segment_frame["forecast_population"] = segment_frame["sku_segment"].map(
            forecast_population_from_segment
        )
        segment_summary = (
            segment_frame
            .groupby(["forecast_population", "sku_segment"], as_index=False)
            .agg(
                actual_total=(TARGET_COLUMN, "sum"),
                skus=(SKU_COLUMN, "nunique"),
                rows=(SKU_COLUMN, "size"),
                positive_rows=(TARGET_COLUMN, lambda series: series.gt(0).sum()),
            )
        )
        segment_summary["agent"] = "Data Quality Agent"
        segment_summary["window_label"] = split_dict["window_label"]
        segment_summary["window_months"] = split_dict["window_months"]
        segment_summary["fold_id"] = split_dict["fold_id"]
        segment_summary["actual_share_percent"] = (
            100
            * segment_summary["actual_total"]
            / segment_summary["actual_total"].sum()
        )
        segment_rows.append(segment_summary)

    return pd.DataFrame(summary_rows), pd.concat(segment_rows, ignore_index=True)


def run_benchmark_agent(forecasts):
    rows = []
    forecast_variants = {
        "post_calibrated": "forecast",
        "pre_calibration": "forecast_before_calibration",
    }

    for keys, group in forecasts.groupby(["window_label", "window_months", "model"], dropna=False):
        base_row = {
            "agent": "Benchmark Agent",
            "window_label": keys[0],
            "window_months": keys[1],
            "model": keys[2],
            "actual_total": group[TARGET_COLUMN].sum(),
            "calibration_multiplier": group["calibration_multiplier"].iloc[0]
            if "calibration_multiplier" in group.columns
            else np.nan,
            "calibration_status": first_or_mixed(group["calibration_status"])
            if "calibration_status" in group.columns
            else np.nan,
        }

        for variant_name, forecast_column in forecast_variants.items():
            if forecast_column in group.columns:
                base_row[f"{variant_name}_forecast_total"] = group[forecast_column].sum()
                base_row[f"{variant_name}_wmape_percent"] = wmape_percent_for_column(group, forecast_column)

        if "forecast_before_calibration" in group.columns:
            multiplier = base_row["calibration_multiplier"]
            if pd.isna(multiplier):
                multiplier = 1.0
            operational_multiplier = max(float(multiplier), OPERATIONAL_CALIBRATION_MIN_MULTIPLIER)
            if keys[2] == "Zero Forecast":
                operational_forecast = group["forecast"]
                operational_multiplier = 1.0
            else:
                operational_forecast = group["forecast_before_calibration"] * operational_multiplier
            base_row["operational_floor_multiplier"] = operational_multiplier
            base_row["operational_floor_forecast_total"] = operational_forecast.sum()
            base_row["operational_floor_wmape_percent"] = (
                100 * wmape(group[TARGET_COLUMN], operational_forecast)
            )

        rows.append(base_row)

    summary = pd.DataFrame(rows)

    zero_columns = [
        "window_label",
        "window_months",
        "post_calibrated_wmape_percent",
        "pre_calibration_wmape_percent",
        "operational_floor_wmape_percent",
    ]
    zero_summary = summary.loc[summary["model"].eq("Zero Forecast"), zero_columns].rename(
        columns={
            "post_calibrated_wmape_percent": "zero_post_calibrated_wmape_percent",
            "pre_calibration_wmape_percent": "zero_pre_calibration_wmape_percent",
            "operational_floor_wmape_percent": "zero_operational_floor_wmape_percent",
        }
    )
    summary = summary.merge(zero_summary, on=["window_label", "window_months"], how="left")

    for variant_name in ["post_calibrated", "pre_calibration", "operational_floor"]:
        summary[f"{variant_name}_delta_vs_zero_wmape"] = (
            summary[f"{variant_name}_wmape_percent"]
            - summary[f"zero_{variant_name}_wmape_percent"]
        )

    return summary.sort_values(["window_months", "post_calibrated_wmape_percent", "model"])


def run_segment_strategy_agent(best_by_segment, data_quality_segments):
    demand_mix = (
        data_quality_segments
        .groupby(["window_label", "window_months", "sku_segment"], as_index=False)
        .agg(
            actual_total=("actual_total", "sum"),
            skus=("skus", "max"),
            actual_share_percent=("actual_share_percent", "sum"),
        )
    )
    recommendations = best_by_segment.merge(
        demand_mix,
        on=["window_label", "window_months", "sku_segment"],
        how="left",
        suffixes=("", "_quality"),
    )

    def choose_recommendation(row):
        wmape_value = row["pooled_wmape_percent"]
        model_name = row["model"]
        segment_name = row["sku_segment"]

        if segment_name in ["missing_from_train", "no_train_demand"]:
            return "Needs new-SKU or supersession signal; history-only SKU forecast should not be trusted."
        if wmape_value < 70:
            return f"Candidate forecasting segment with {model_name}; target met in this backtest."
        if wmape_value < 100:
            return f"Watchlist segment; {model_name} beats zero but has not met the 70 percent target."
        if model_name == "Zero Forecast" or wmape_value >= 100:
            return "SKU-month model is not beating zero; use aggregate demand planning or stocking policy until stronger signals exist."
        return "Needs further review."

    recommendations["agent"] = "Segment Strategy Agent"
    recommendations["recommendation"] = recommendations.apply(choose_recommendation, axis=1)
    return recommendations.sort_values(["window_months", "sku_segment"])


def run_signal_inventory_agent(data):
    rows = []
    excluded_columns = {
        TARGET_COLUMN,
        DATE_COLUMN,
        "Date",
        "year",
        "month",
        "demand_type_clean",
        DEMAND_TYPE_COLUMN,
        COLLISION_COLUMN,
        "collision_flag",
        "collision_flag_clean",
        "CURRENCY",
        "COUNTRY_BRAND_CHANNEL",
        "Brand",
        "Channel",
        "Country",
        "REGION",
        "ts_id",
    }
    eligible_descriptor_columns = set(available_aggregate_descriptor_columns(data))
    configured_descriptor_columns = set(AGGREGATE_ALLOCATION_DESCRIPTOR_COLUMNS)
    total_skus = data[SKU_COLUMN].nunique() if SKU_COLUMN in data.columns else 0

    for column in data.columns:
        if column in {TARGET_COLUMN, DATE_COLUMN, "year", "month", "demand_type_clean"}:
            continue
        series = data[column]
        non_null_share = series.notna().mean()
        nunique = series.nunique(dropna=True)
        dtype_name = str(series.dtype)

        if column == SKU_COLUMN or column == "ts_id":
            role = "identifier"
        elif column.startswith("calendar_"):
            role = "known_calendar"
        elif column.startswith("external_"):
            role = "external_feature"
        elif column in configured_descriptor_columns:
            role = "candidate_descriptor"
        elif (
            pd.api.types.is_object_dtype(series)
            or pd.api.types.is_string_dtype(series)
            or pd.api.types.is_categorical_dtype(series)
        ):
            role = "context_descriptor"
        elif pd.api.types.is_numeric_dtype(series):
            role = "candidate_numeric_signal"
        else:
            role = "other"

        if role in ["candidate_descriptor", "context_descriptor"] and total_skus > 0:
            sku_non_null_share = (
                data.loc[series.notna(), SKU_COLUMN].nunique() / total_skus
            )
        else:
            sku_non_null_share = np.nan

        descriptor_coverage = max(
            non_null_share,
            0 if pd.isna(sku_non_null_share) else sku_non_null_share,
        )
        descriptor_allocation_candidate = (
            role == "candidate_descriptor"
            and column in eligible_descriptor_columns
            and descriptor_coverage >= 0.50
        )

        if column in LEAKAGE_COLUMNS:
            signal_action = "exclude_possible_leakage"
        elif descriptor_allocation_candidate:
            signal_action = "test_as_descriptor_allocation_group"
        elif role == "candidate_descriptor" and nunique <= 1:
            signal_action = "not_enough_variation_for_group_allocation"
        elif role == "candidate_descriptor" and nunique > AGGREGATE_DESCRIPTOR_MAX_UNIQUE:
            signal_action = "too_high_cardinality_for_group_allocation"
        elif column in excluded_columns or role == "context_descriptor":
            signal_action = "context_only_not_allocation_group"
        elif non_null_share < 0.50 and (pd.isna(sku_non_null_share) or sku_non_null_share < 0.50):
            signal_action = "too_sparse_for_now"
        elif role in ["candidate_numeric_signal", "external_feature"]:
            signal_action = "review_known_ahead_status_before_model_use"
        elif role == "known_calendar":
            signal_action = "safe_known_ahead"
        else:
            signal_action = "low_priority"

        rows.append(
            {
                "agent": "Signal Discovery Agent",
                "column": column,
                "dtype": dtype_name,
                "role": role,
                "non_null_share": non_null_share,
                "sku_non_null_share": sku_non_null_share,
                "unique_values": nunique,
                "configured_allocation_descriptor": column in configured_descriptor_columns,
                "usable_for_future_model": role in [
                    "known_calendar",
                    "external_feature",
                    "candidate_descriptor",
                    "candidate_numeric_signal",
                ],
                "descriptor_allocation_candidate": descriptor_allocation_candidate,
                "signal_action": signal_action,
            }
        )

    return pd.DataFrame(rows).sort_values(
        [
            "descriptor_allocation_candidate",
            "configured_allocation_descriptor",
            "usable_for_future_model",
            "role",
            "column",
        ],
        ascending=[False, False, False, True, True],
    )


def lifecycle_signal_role(column):
    normalized = str(column).lower()
    matches = [pattern for pattern in LIFECYCLE_SIGNAL_NAME_MATCHES if pattern in normalized]
    if not matches:
        return "", "not_lifecycle_signal"
    if any(pattern in normalized for pattern in ["stockout", "stock_out", "out_of_stock", "availability", "available"]):
        return ", ".join(matches), "availability_or_stockout"
    if any(pattern in normalized for pattern in ["supersession", "superseded", "replacement"]):
        return ", ".join(matches), "supersession_or_replacement"
    if any(pattern in normalized for pattern in ["launch", "introduced", "introduction"]):
        return ", ".join(matches), "launch_or_introduction"
    if any(pattern in normalized for pattern in ["discontinued", "inactive", "active", "lifecycle"]):
        return ", ".join(matches), "lifecycle_status"
    return ", ".join(matches), "candidate_lifecycle_signal"


def run_lifecycle_signal_agent(data):
    rows = []
    for column in data.columns:
        matches, role = lifecycle_signal_role(column)
        if role == "not_lifecycle_signal":
            continue
        series = data[column]
        non_null_share = series.notna().mean()
        sku_non_null_share = (
            data.loc[series.notna(), SKU_COLUMN].nunique() / data[SKU_COLUMN].nunique()
            if SKU_COLUMN in data.columns and data[SKU_COLUMN].nunique() > 0
            else np.nan
        )
        if role == "availability_or_stockout":
            recommendation = "High value if known at forecast time; use to separate true zero demand from unavailable-stock zeroes."
        elif role == "supersession_or_replacement":
            recommendation = "High value for collision parts; use to transfer demand between old and replacement SKUs."
        elif role == "launch_or_introduction":
            recommendation = "Useful for cold-start and part-age features."
        elif role == "lifecycle_status":
            recommendation = "Useful for active/inactive filtering and obsolescence decay."
        else:
            recommendation = "Review as a possible lifecycle signal."

        rows.append(
            {
                "agent": "Lifecycle Signal Agent",
                "column": column,
                "matched_terms": matches,
                "lifecycle_role": role,
                "dtype": str(series.dtype),
                "non_null_share": non_null_share,
                "sku_non_null_share": sku_non_null_share,
                "unique_values": series.nunique(dropna=True),
                "recommendation": recommendation,
            }
        )

    if rows:
        return pd.DataFrame(rows).sort_values(
            ["lifecycle_role", "sku_non_null_share", "non_null_share"],
            ascending=[True, False, False],
        )

    return pd.DataFrame(
        [
            {
                "agent": "Lifecycle Signal Agent",
                "column": "",
                "matched_terms": "",
                "lifecycle_role": "no_lifecycle_columns_found",
                "dtype": "",
                "non_null_share": np.nan,
                "sku_non_null_share": np.nan,
                "unique_values": np.nan,
                "recommendation": "No obvious stockout, availability, supersession, launch, or lifecycle columns were found. These are likely the highest-value extra data sources to add next.",
            }
        ]
    )


def run_hybrid_impact_agent(model_summary, known_history_model_summary, occurrence_gate_report):
    target_models = [
        "Empirical Bayes Allocation",
        "Occurrence Gated Allocation",
        "Aggregate Allocation",
        "Hurdle Random Forest",
        "Direct Demand Random Forest",
        "Zero Forecast",
    ]
    rows = []

    for (window_label, window_months), window_models in model_summary.groupby(["window_label", "window_months"]):
        zero_rows = window_models.loc[window_models["model"].eq("Zero Forecast")]
        aggregate_rows = window_models.loc[window_models["model"].eq("Aggregate Allocation")]
        zero_wmape = zero_rows["pooled_wmape_percent"].iloc[0] if not zero_rows.empty else np.nan
        aggregate_wmape = aggregate_rows["pooled_wmape_percent"].iloc[0] if not aggregate_rows.empty else np.nan

        known_window = known_history_model_summary.loc[
            known_history_model_summary["window_label"].eq(window_label)
            & known_history_model_summary["window_months"].eq(window_months)
        ]
        known_zero_rows = known_window.loc[known_window["model"].eq("Zero Forecast")]
        known_zero_wmape = (
            known_zero_rows["pooled_wmape_percent"].iloc[0]
            if not known_zero_rows.empty
            else np.nan
        )

        for model_name in target_models:
            model_rows = window_models.loc[window_models["model"].eq(model_name)]
            if model_rows.empty:
                continue
            model_row = model_rows.iloc[0]
            occurrence_rows = occurrence_gate_report.loc[
                occurrence_gate_report["window_label"].eq(window_label)
                & occurrence_gate_report["window_months"].eq(window_months)
                & occurrence_gate_report["model"].eq(model_name)
            ]
            occurrence_row = occurrence_rows.iloc[0] if not occurrence_rows.empty else {}

            known_rows = known_window.loc[known_window["model"].eq(model_name)]
            known_wmape = (
                known_rows["pooled_wmape_percent"].iloc[0]
                if not known_rows.empty
                else np.nan
            )

            delta_vs_zero = model_row["pooled_wmape_percent"] - zero_wmape if pd.notna(zero_wmape) else np.nan
            delta_vs_aggregate = (
                model_row["pooled_wmape_percent"] - aggregate_wmape
                if pd.notna(aggregate_wmape)
                else np.nan
            )
            known_delta_vs_zero = known_wmape - known_zero_wmape if pd.notna(known_zero_wmape) and pd.notna(known_wmape) else np.nan

            if model_row["pooled_wmape_percent"] < 70:
                recommendation = "Target met at SKU level; prioritize validation stability."
            elif pd.notna(delta_vs_zero) and delta_vs_zero < 0:
                recommendation = "Beats zero at SKU level; worth deeper tuning."
            elif model_name in ["Empirical Bayes Allocation", "Occurrence Gated Allocation"] and pd.notna(delta_vs_aggregate) and delta_vs_aggregate < 0:
                recommendation = "Improves on static aggregate allocation; keep exploring the hybrid path."
            elif model_name in ["Empirical Bayes Allocation", "Occurrence Gated Allocation"]:
                recommendation = "Hybrid did not beat the benchmark in this run; inspect occurrence precision/recall before expanding."
            else:
                recommendation = "Benchmark/reference model."

            rows.append(
                {
                    "agent": "Hybrid Impact Agent",
                    "window_label": window_label,
                    "window_months": window_months,
                    "model": model_name,
                    "sku_wmape_percent": model_row["pooled_wmape_percent"],
                    "delta_vs_zero_wmape_percent": delta_vs_zero,
                    "delta_vs_aggregate_allocation_wmape_percent": delta_vs_aggregate,
                    "known_history_wmape_percent": known_wmape,
                    "known_history_delta_vs_zero_wmape_percent": known_delta_vs_zero,
                    "forecast_total": model_row["forecast_total"],
                    "actual_total": model_row["actual_total"],
                    "positive_row_recall_percent": occurrence_row.get("positive_row_recall_percent", np.nan)
                    if isinstance(occurrence_row, dict)
                    else occurrence_row.get("positive_row_recall_percent", np.nan),
                    "positive_forecast_precision_percent": occurrence_row.get("positive_forecast_precision_percent", np.nan)
                    if isinstance(occurrence_row, dict)
                    else occurrence_row.get("positive_forecast_precision_percent", np.nan),
                    "zero_actual_error_share_percent": occurrence_row.get("zero_actual_error_share_percent", np.nan)
                    if isinstance(occurrence_row, dict)
                    else occurrence_row.get("zero_actual_error_share_percent", np.nan),
                    "recommendation": recommendation,
                }
            )

    return pd.DataFrame(rows).sort_values(["window_months", "sku_wmape_percent", "model"])


def run_results_critic_agent(
    model_summary,
    benchmark_summary,
    data_quality_summary,
    population_model_summary,
):
    rows = []

    for (window_label, window_months), window_models in model_summary.groupby(["window_label", "window_months"]):
        best_post = window_models.sort_values("pooled_wmape_percent").iloc[0]
        window_bench = benchmark_summary.loc[
            benchmark_summary["window_label"].eq(window_label)
            & benchmark_summary["window_months"].eq(window_months)
        ]
        best_pre = window_bench.sort_values("pre_calibration_wmape_percent").iloc[0]
        best_operational = window_bench.sort_values("operational_floor_wmape_percent").iloc[0]
        quality = data_quality_summary.loc[
            data_quality_summary["window_label"].eq(window_label)
            & data_quality_summary["window_months"].eq(window_months)
        ].iloc[0]

        window_population = population_model_summary.loc[
            population_model_summary["window_label"].eq(window_label)
            & population_model_summary["window_months"].eq(window_months)
        ]
        known_population = window_population.loc[
            window_population["forecast_population"].eq(PREDICTABLE_FORECAST_POPULATION)
        ]
        cold_population = window_population.loc[
            window_population["forecast_population"].isin(
                [COLD_START_MISSING_POPULATION, KNOWN_NO_TRAIN_DEMAND_POPULATION]
            )
        ]

        if known_population.empty:
            best_known_model = np.nan
            best_known_wmape = np.nan
            known_actual_total = 0.0
        else:
            best_known = known_population.sort_values(
                ["population_selection_score", "absolute_error"]
            ).iloc[0]
            best_known_model = best_known["model"]
            best_known_wmape = best_known["pooled_wmape_percent"]
            known_actual_total = best_known["actual_total"]

        cold_actual_total = (
            cold_population
            .drop_duplicates(["forecast_population"])
            ["actual_total"]
            .sum()
        )
        cold_start_actual_share_percent = safe_percent(
            cold_actual_total,
            quality["test_actual_total"],
        )

        if pd.notna(best_known_wmape) and best_known_wmape < 70:
            verdict = "Known-SKU target met; judge cold-start separately before portfolio rollout."
        elif pd.notna(best_known_wmape) and best_known_wmape < 100 and best_post["pooled_wmape_percent"] >= 100:
            verdict = "Known-SKU signal is improving, but cold-start/no-history demand is dragging the portfolio result."
        elif best_post["pooled_wmape_percent"] < 70:
            verdict = "Portfolio target met; validate stability with more folds/history."
        elif best_pre["pre_calibration_wmape_percent"] < 100:
            verdict = "A non-zero model beats zero before calibration; inspect operational floor and population mix."
        elif best_post["pooled_wmape_percent"] <= 100:
            verdict = "Zero-style forecast is still winning; separate known-SKU and cold-start decisions."
        else:
            verdict = "All candidate models are worse than zero; move away from SKU-month history-only modelling."

        rows.append(
            {
                "agent": "Results Critic Agent",
                "window_label": window_label,
                "window_months": window_months,
                "best_post_model": best_post["model"],
                "best_post_wmape_percent": best_post["pooled_wmape_percent"],
                "best_pre_calibration_model": best_pre["model"],
                "best_pre_calibration_wmape_percent": best_pre["pre_calibration_wmape_percent"],
                "best_operational_floor_model": best_operational["model"],
                "best_operational_floor_wmape_percent": best_operational["operational_floor_wmape_percent"],
                "best_known_history_model": best_known_model,
                "best_known_history_wmape_percent": best_known_wmape,
                "known_history_actual_total": known_actual_total,
                "cold_start_actual_share_percent": cold_start_actual_share_percent,
                "test_actual_total": quality["test_actual_total"],
                "missing_train_actual_share_percent": quality["missing_train_actual_share_percent"],
                "no_train_demand_actual_share_percent": quality["no_train_demand_actual_share_percent"],
                "verdict": verdict,
            }
        )

    return pd.DataFrame(rows)


def run_aggregate_allocation_agent(
    benchmark_summary,
    tuning_results,
    signal_inventory,
    monthly_total_model_summary,
):
    aggregate_rows = benchmark_summary.loc[
        benchmark_summary["model"].eq("Aggregate Allocation")
    ].copy()
    tuning_rows = tuning_results.loc[
        tuning_results["model"].eq("Aggregate Allocation"),
        ["window_label", "window_months", "hyperparameters", "tuning_status", "tuning_wmape_percent"],
    ].copy()
    monthly_rows = monthly_total_model_summary.loc[
        monthly_total_model_summary["model"].eq("Aggregate Allocation"),
        [
            "window_label",
            "window_months",
            "post_calibrated_monthly_total_wmape_percent",
            "pre_calibration_monthly_total_wmape_percent",
            "operational_floor_monthly_total_wmape_percent",
        ],
    ].copy()

    descriptor_candidates = signal_inventory.loc[
        signal_inventory.get("descriptor_allocation_candidate", pd.Series(False, index=signal_inventory.index)).fillna(False)
    ].copy()
    descriptor_candidate_list = ", ".join(descriptor_candidates["column"].head(10).tolist())

    report = aggregate_rows.merge(
        tuning_rows,
        on=["window_label", "window_months"],
        how="left",
    ).merge(
        monthly_rows,
        on=["window_label", "window_months"],
        how="left",
    )

    def recommendation(row):
        monthly_total_wmape = row.get("post_calibrated_monthly_total_wmape_percent", np.nan)
        if pd.notna(monthly_total_wmape) and monthly_total_wmape < 70 and row["post_calibrated_wmape_percent"] >= 100:
            return "Monthly total is forecastable; SKU allocation is the bottleneck. Focus on subfamily-to-SKU probability allocation."
        if row["pre_calibration_wmape_percent"] < row["zero_pre_calibration_wmape_percent"]:
            return "Aggregate allocation beats zero before calibration; make this the lead candidate."
        if row["operational_floor_delta_vs_zero_wmape"] <= 5:
            return "Aggregate allocation is close to zero while producing a usable non-zero plan; improve descriptor allocation and monthly total forecast."
        return "Aggregate allocation is the best non-zero direction if it beats other non-zero models, but still needs stronger allocation signals."

    report["agent"] = "Aggregate Allocation Agent"
    report["descriptor_candidates_available"] = descriptor_candidate_list
    report["recommendation"] = report.apply(recommendation, axis=1)
    return report.sort_values(["window_months"])










def run_feature_selection_agent(feature_inventory, hurdle_feature_metadata, tuning_results):
    rows = []
    inventory = feature_inventory.copy()
    selected_features = set(hurdle_feature_metadata["feature"].dropna().astype(str))

    if inventory.empty:
        return pd.DataFrame(
            [{
                "agent": "Feature Selection Agent",
                "feature_family": "none",
                "candidate_features": 0,
                "selected_features": 0,
                "selected_share_percent": np.nan,
                "recommendation": "No candidate features were available.",
            }]
        )

    inventory["feature_family"] = inventory["feature"].map(feature_family)
    inventory["selected_in_hurdle_rf"] = inventory["feature"].isin(selected_features)

    for family, group in inventory.groupby("feature_family", dropna=False):
        candidate_count = len(group)
        selected_count = int(group["selected_in_hurdle_rf"].sum())
        selected_share = safe_percent(selected_count, candidate_count)
        if family == "external" and selected_count == 0:
            recommendation = "External features were available but not selected by the validation gate; keep them out for SKU-level RF."
        elif family == "external":
            recommendation = "Only validation-positive external features are being used; monitor test WMAPE for overfit."
        elif family == "rf_internal_history":
            recommendation = "Core SKU-history features; keep unless overfit agent flags a large validation/test gap."
        elif family == "calendar":
            recommendation = "Calendar features are allowed when validation supports them."
        else:
            recommendation = "Review feature family manually."

        rows.append(
            {
                "agent": "Feature Selection Agent",
                "feature_family": family,
                "candidate_features": candidate_count,
                "selected_features": selected_count,
                "selected_share_percent": selected_share,
                "recommendation": recommendation,
            }
        )

    hurdle_tuning = tuning_results.loc[tuning_results["model"].eq("Hurdle Random Forest")].copy()
    if not hurdle_tuning.empty:
        for _, row in hurdle_tuning.iterrows():
            rows.append(
                {
                    "agent": "Feature Selection Agent",
                    "feature_family": "selected_profile",
                    "candidate_features": np.nan,
                    "selected_features": np.nan,
                    "selected_share_percent": np.nan,
                    "window_label": row.get("window_label"),
                    "window_months": row.get("window_months"),
                    "hyperparameters": row.get("hyperparameters"),
                    "tuning_wmape_percent": row.get("tuning_wmape_percent"),
                    "recommendation": "Chosen Hurdle RF feature profile and selected feature set from validation tuning.",
                }
            )

    return pd.DataFrame(rows)


def run_feature_overfit_agent(tuning_results, model_summary, hurdle_feature_metadata):
    rows = []
    hurdle_tuning = tuning_results.loc[tuning_results["model"].eq("Hurdle Random Forest")].copy()
    hurdle_summary = model_summary.loc[model_summary["model"].eq("Hurdle Random Forest")].copy()
    selected_external_count = (
        hurdle_feature_metadata
        .loc[hurdle_feature_metadata["feature"].astype(str).str.startswith("external_"), "feature"]
        .nunique()
    )

    for _, tuning_row in hurdle_tuning.iterrows():
        summary_match = hurdle_summary.loc[
            hurdle_summary["window_label"].eq(tuning_row["window_label"])
            & hurdle_summary["window_months"].eq(tuning_row["window_months"])
        ]
        if summary_match.empty:
            continue
        summary_row = summary_match.iloc[0]
        tuning_wmape = tuning_row.get("tuning_wmape_percent")
        test_wmape = summary_row.get("pooled_wmape_percent")
        overfit_gap = test_wmape - tuning_wmape if pd.notna(tuning_wmape) else np.nan

        if pd.isna(overfit_gap):
            recommendation = "No validation comparison available; avoid external-heavy feature profiles in this fold."
        elif overfit_gap > FEATURE_OVERFIT_WARNING_GAP_PERCENT:
            recommendation = "Validation was much better than test; treat selected features as overfit and prefer simpler RF profiles."
        elif test_wmape >= 100:
            recommendation = "Feature set is not beating zero at SKU level; do not add more features until allocation/target design improves."
        else:
            recommendation = "No major overfit gap detected; keep monitoring across more folds."

        rows.append(
            {
                "agent": "Feature Overfit Agent",
                "window_label": tuning_row["window_label"],
                "window_months": tuning_row["window_months"],
                "tuning_wmape_percent": tuning_wmape,
                "test_wmape_percent": test_wmape,
                "overfit_gap_percent": overfit_gap,
                "selected_external_feature_count": selected_external_count,
                "hyperparameters": tuning_row.get("hyperparameters"),
                "recommendation": recommendation,
            }
        )

    return pd.DataFrame(rows)


def run_occurrence_gate_agent(forecasts):
    rows = []

    for keys, group in forecasts.groupby(["window_label", "window_months", "model"], dropna=False):
        actual_positive = group[TARGET_COLUMN].gt(0)
        forecast_positive = group["forecast"].gt(1e-9)
        true_positive = int((actual_positive & forecast_positive).sum())
        false_positive = int((~actual_positive & forecast_positive).sum())
        false_negative = int((actual_positive & ~forecast_positive).sum())
        absolute_error = (group["forecast"] - group[TARGET_COLUMN]).abs()
        total_abs_error = absolute_error.sum()
        zero_actual_error = absolute_error.loc[~actual_positive].sum()
        positive_actual_error = absolute_error.loc[actual_positive].sum()
        actual_total = group[TARGET_COLUMN].sum()
        forecast_total = group["forecast"].sum()
        precision = safe_percent(true_positive, true_positive + false_positive)
        recall = safe_percent(true_positive, true_positive + false_negative)
        wmape_value = safe_percent(total_abs_error, actual_total)

        if forecast_positive.sum() == 0:
            recommendation = "Zero-style output; useful benchmark, but it misses all positive demand months."
        elif pd.notna(precision) and precision < 50 and pd.notna(recall) and recall >= 80:
            recommendation = "Too many false positive SKU-months; raise occurrence threshold or regression floor."
        elif pd.notna(recall) and recall < 50:
            recommendation = "Occurrence gate is too strict; it is missing too many positive demand months."
        elif pd.notna(wmape_value) and wmape_value < 70:
            recommendation = "Occurrence gate is usable in this window; validate stability across more history."
        else:
            recommendation = "Monitor precision and recall; SKU-month occurrence remains the bottleneck."

        rows.append(
            {
                "agent": "Occurrence Gate Agent",
                "window_label": keys[0],
                "window_months": keys[1],
                "model": keys[2],
                "actual_total": actual_total,
                "forecast_total": forecast_total,
                "wmape_percent": wmape_value,
                "positive_actual_rows": int(actual_positive.sum()),
                "forecast_positive_rows": int(forecast_positive.sum()),
                "positive_row_recall_percent": recall,
                "positive_forecast_precision_percent": precision,
                "false_positive_rows": false_positive,
                "false_negative_rows": false_negative,
                "zero_actual_error_share_percent": safe_percent(zero_actual_error, total_abs_error),
                "positive_actual_error_share_percent": safe_percent(positive_actual_error, total_abs_error),
                "forecast_on_zero_actual": group.loc[~actual_positive, "forecast"].sum(),
                "missed_positive_actual_if_zero_forecast": group.loc[
                    actual_positive & ~forecast_positive,
                    TARGET_COLUMN,
                ].sum(),
                "recommendation": recommendation,
            }
        )

    return pd.DataFrame(rows).sort_values(["window_months", "wmape_percent", "model"])


def run_monthly_total_agent(monthly_total_model_summary, model_summary):
    rows = []

    for (window_label, window_months), window_totals in monthly_total_model_summary.groupby(["window_label", "window_months"]):
        best_post = window_totals.sort_values("post_calibrated_monthly_total_wmape_percent").iloc[0]
        best_pre = window_totals.sort_values("pre_calibration_monthly_total_wmape_percent").iloc[0]
        best_operational = window_totals.sort_values("operational_floor_monthly_total_wmape_percent").iloc[0]
        aggregate_rows = window_totals.loc[window_totals["model"].eq("Aggregate Allocation")]
        aggregate_row = aggregate_rows.iloc[0] if not aggregate_rows.empty else None

        window_sku = model_summary.loc[
            model_summary["window_label"].eq(window_label)
            & model_summary["window_months"].eq(window_months)
        ]
        best_sku = window_sku.sort_values("pooled_wmape_percent").iloc[0]

        aggregate_monthly_wmape = (
            aggregate_row["post_calibrated_monthly_total_wmape_percent"]
            if aggregate_row is not None
            else np.nan
        )

        if pd.notna(aggregate_monthly_wmape) and aggregate_monthly_wmape < 50:
            recommendation = "Aggregate monthly demand is strong; preserve this model and improve SKU allocation separately."
        elif best_post["post_calibrated_monthly_total_wmape_percent"] < 70:
            recommendation = "Monthly total forecast is usable; evaluate planning at aggregate/category level while improving SKU allocation."
        elif best_sku["pooled_wmape_percent"] >= 100:
            recommendation = "Neither SKU-month nor monthly total target is met; prioritize total-demand drivers before SKU allocation."
        else:
            recommendation = "Monthly total forecast needs review alongside SKU-month performance."

        rows.append(
            {
                "agent": "Monthly Total Agent",
                "window_label": window_label,
                "window_months": window_months,
                "best_monthly_total_model": best_post["model"],
                "best_monthly_total_wmape_percent": best_post["post_calibrated_monthly_total_wmape_percent"],
                "best_pre_calibration_monthly_total_model": best_pre["model"],
                "best_pre_calibration_monthly_total_wmape_percent": best_pre["pre_calibration_monthly_total_wmape_percent"],
                "best_operational_monthly_total_model": best_operational["model"],
                "best_operational_monthly_total_wmape_percent": best_operational["operational_floor_monthly_total_wmape_percent"],
                "aggregate_allocation_monthly_total_wmape_percent": aggregate_monthly_wmape,
                "best_sku_month_model": best_sku["model"],
                "best_sku_month_wmape_percent": best_sku["pooled_wmape_percent"],
                "recommendation": recommendation,
            }
        )

    return pd.DataFrame(rows)


def run_sku_horizon_agent(sku_horizon_model_summary, model_summary):
    rows = []

    for (window_label, window_months), window_horizon in sku_horizon_model_summary.groupby(["window_label", "window_months"]):
        best_horizon = window_horizon.sort_values(
            ["horizon_sku_wmape_percent", "absolute_error"]
        ).iloc[0]
        best_positive_only = window_horizon.sort_values(
            ["positive_sku_only_wmape_percent", "absolute_error"]
        ).iloc[0]

        window_monthly = model_summary.loc[
            model_summary["window_label"].eq(window_label)
            & model_summary["window_months"].eq(window_months)
        ]
        best_monthly = window_monthly.sort_values("pooled_wmape_percent").iloc[0]
        same_model_monthly = window_monthly.loc[
            window_monthly["model"].eq(best_horizon["model"])
        ]
        same_model_monthly_wmape = (
            same_model_monthly["pooled_wmape_percent"].iloc[0]
            if not same_model_monthly.empty
            else np.nan
        )

        horizon_wmape = best_horizon["horizon_sku_wmape_percent"]
        positive_only_wmape = best_horizon["positive_sku_only_wmape_percent"]
        zero_share = best_horizon["zero_actual_sku_forecast_share_percent"]

        if horizon_wmape < 70:
            recommendation = "SKU horizon target is met; treat this as a strong candidate for 9/18-month quantity planning."
        elif positive_only_wmape < 70 and pd.notna(zero_share) and zero_share > 5:
            recommendation = "Positive-demand SKUs are near target; reduce zero-actual SKU leakage with stricter candidate filters."
        elif horizon_wmape < 90:
            recommendation = "Promising for SKU-level horizon quantity, but not target-ready; focus on SKU allocation precision."
        elif best_monthly["pooled_wmape_percent"] >= 100 and horizon_wmape < 100:
            recommendation = "Horizon quantity is better than monthly phasing; decide whether planning needs SKU totals or SKU-month timing."
        else:
            recommendation = "Still close to zero forecast; lifecycle, stockout, supersession, and active-SKU signals are likely needed."

        rows.append(
            {
                "agent": "SKU Horizon Agent",
                "window_label": window_label,
                "window_months": window_months,
                "best_horizon_model": best_horizon["model"],
                "best_horizon_forecast_variant": best_horizon["forecast_variant"],
                "best_horizon_sku_wmape_percent": horizon_wmape,
                "best_horizon_positive_sku_only_wmape_percent": positive_only_wmape,
                "best_horizon_zero_actual_sku_forecast_share_percent": zero_share,
                "best_horizon_positive_actual_below_70_share_percent": best_horizon["positive_actual_below_70_share_percent"],
                "best_horizon_positive_actual_below_100_share_percent": best_horizon["positive_actual_below_100_share_percent"],
                "best_positive_only_model": best_positive_only["model"],
                "best_positive_only_forecast_variant": best_positive_only["forecast_variant"],
                "best_positive_only_wmape_percent": best_positive_only["positive_sku_only_wmape_percent"],
                "best_sku_month_model": best_monthly["model"],
                "best_sku_month_wmape_percent": best_monthly["pooled_wmape_percent"],
                "best_horizon_model_sku_month_wmape_percent": same_model_monthly_wmape,
                "recommendation": recommendation,
            }
        )

    return pd.DataFrame(rows).sort_values(["window_months", "best_horizon_sku_wmape_percent"])


def run_metric_decision_agent(metric_suite_model_summary, sku_horizon_model_summary):
    rows = []

    for (window_label, window_months), window_metrics in metric_suite_model_summary.groupby(["window_label", "window_months"]):
        best_cost = window_metrics.sort_values(
            ["inventory_utility_cost_per_actual", "mase", "wmape_percent"]
        ).iloc[0]
        best_mase = window_metrics.sort_values(
            ["mase", "inventory_utility_cost_per_actual", "wmape_percent"]
        ).iloc[0]
        best_wmape = window_metrics.sort_values(
            ["wmape_percent", "mase", "inventory_utility_cost_per_actual"]
        ).iloc[0]

        horizon_window = sku_horizon_model_summary.loc[
            sku_horizon_model_summary["window_label"].eq(window_label)
            & sku_horizon_model_summary["window_months"].eq(window_months)
        ]
        best_horizon = horizon_window.sort_values(
            ["horizon_sku_wmape_percent", "absolute_error"]
        ).iloc[0] if not horizon_window.empty else None

        if best_horizon is not None and best_horizon["horizon_sku_wmape_percent"] < best_wmape["wmape_percent"] - 15:
            recommendation = "Use SKU-horizon and inventory-utility metrics for planning decisions; SKU-month WMAPE is mostly measuring timing noise."
        elif best_cost["model"] != best_wmape["model"]:
            recommendation = "Model ranking changes under cost-weighted metrics; do not optimize on WMAPE alone."
        elif best_cost["fill_rate_proxy_percent"] < 70:
            recommendation = "All models are under-serving demand under this cost view; prioritize occurrence recall and safety-stock policy."
        else:
            recommendation = "Cost and statistical metrics broadly agree; keep WMAPE as a diagnostic and use inventory utility as the decision metric."

        rows.append(
            {
                "agent": "Metric Decision Agent",
                "window_label": window_label,
                "window_months": window_months,
                "holding_cost_weight": INVENTORY_HOLDING_COST_WEIGHT,
                "stockout_cost_weight": INVENTORY_STOCKOUT_COST_WEIGHT,
                "best_inventory_utility_model": best_cost["model"],
                "best_inventory_utility_variant": best_cost["forecast_variant"],
                "best_inventory_utility_cost_per_actual": best_cost["inventory_utility_cost_per_actual"],
                "best_inventory_utility_fill_rate_proxy_percent": best_cost["fill_rate_proxy_percent"],
                "best_inventory_utility_over_forecast_to_actual_percent": best_cost["over_forecast_to_actual_percent"],
                "best_mase_model": best_mase["model"],
                "best_mase_variant": best_mase["forecast_variant"],
                "best_mase": best_mase["mase"],
                "best_wmape_model": best_wmape["model"],
                "best_wmape_variant": best_wmape["forecast_variant"],
                "best_wmape_percent": best_wmape["wmape_percent"],
                "best_horizon_model": best_horizon["model"] if best_horizon is not None else np.nan,
                "best_horizon_variant": best_horizon["forecast_variant"] if best_horizon is not None else np.nan,
                "best_horizon_sku_wmape_percent": best_horizon["horizon_sku_wmape_percent"] if best_horizon is not None else np.nan,
                "recommendation": recommendation,
            }
        )

    return pd.DataFrame(rows).sort_values(["window_months", "best_inventory_utility_cost_per_actual"])


def run_lag_comparison_agent(metric_suite_model_summary, sku_horizon_model_summary):
    rows = []

    for window_months, window_metrics in metric_suite_model_summary.groupby("window_months"):
        window_metrics = window_metrics.copy()
        requested_rows = window_metrics.loc[
            window_metrics["gap_months"].eq(FORECAST_LAG_MONTHS)
        ]
        best_cost = window_metrics.sort_values(
            ["inventory_utility_cost_per_actual", "mase", "wmape_percent"]
        ).iloc[0]
        requested_cost = (
            requested_rows
            .sort_values(["inventory_utility_cost_per_actual", "mase", "wmape_percent"])
            .iloc[0]
            if not requested_rows.empty
            else None
        )

        horizon_window = sku_horizon_model_summary.loc[
            sku_horizon_model_summary["window_months"].eq(window_months)
        ].copy()
        best_horizon = horizon_window.sort_values(
            ["horizon_sku_wmape_percent", "absolute_error"]
        ).iloc[0] if not horizon_window.empty else None
        requested_horizon_rows = horizon_window.loc[
            horizon_window["gap_months"].eq(FORECAST_LAG_MONTHS)
        ]
        requested_horizon = (
            requested_horizon_rows
            .sort_values(["horizon_sku_wmape_percent", "absolute_error"])
            .iloc[0]
            if not requested_horizon_rows.empty
            else None
        )

        requested_cost_value = (
            requested_cost["inventory_utility_cost_per_actual"]
            if requested_cost is not None
            else np.nan
        )
        requested_horizon_value = (
            requested_horizon["horizon_sku_wmape_percent"]
            if requested_horizon is not None
            else np.nan
        )
        cost_delta = best_cost["inventory_utility_cost_per_actual"] - requested_cost_value
        horizon_delta = (
            best_horizon["horizon_sku_wmape_percent"] - requested_horizon_value
            if best_horizon is not None
            else np.nan
        )

        if pd.notna(cost_delta) and cost_delta < -0.05:
            recommendation = "Alternative lag improves inventory-utility score; present 3-month as benchmark and the better lag as the recommended planning setup."
        elif pd.notna(horizon_delta) and horizon_delta < -5:
            recommendation = "Alternative lag improves SKU-horizon accuracy; useful if lead-time policy can move."
        elif best_cost["gap_months"] == FORECAST_LAG_MONTHS:
            recommendation = "The requested 3-month lag is competitive under the current metrics."
        else:
            recommendation = "Lag differences are modest; choose based on operational lead-time feasibility."

        rows.append(
            {
                "agent": "Lag Comparison Agent",
                "window_months": window_months,
                "requested_lag_months": FORECAST_LAG_MONTHS,
                "best_inventory_utility_lag_months": best_cost["gap_months"],
                "best_inventory_utility_window_label": best_cost["window_label"],
                "best_inventory_utility_model": best_cost["model"],
                "best_inventory_utility_variant": best_cost["forecast_variant"],
                "best_inventory_utility_cost_per_actual": best_cost["inventory_utility_cost_per_actual"],
                "requested_lag_best_inventory_utility_cost_per_actual": requested_cost_value,
                "inventory_utility_delta_vs_requested_lag": cost_delta,
                "best_sku_horizon_lag_months": best_horizon["gap_months"] if best_horizon is not None else np.nan,
                "best_sku_horizon_window_label": best_horizon["window_label"] if best_horizon is not None else np.nan,
                "best_sku_horizon_model": best_horizon["model"] if best_horizon is not None else np.nan,
                "best_sku_horizon_variant": best_horizon["forecast_variant"] if best_horizon is not None else np.nan,
                "best_sku_horizon_wmape_percent": best_horizon["horizon_sku_wmape_percent"] if best_horizon is not None else np.nan,
                "requested_lag_best_sku_horizon_wmape_percent": requested_horizon_value,
                "sku_horizon_delta_vs_requested_lag": horizon_delta,
                "recommendation": recommendation,
            }
        )

    return pd.DataFrame(rows).sort_values(["window_months"])


def run_population_strategy_agent(best_by_population, population_model_summary):
    recommendations = best_by_population.copy()
    total_actual = (
        recommendations
        .groupby(["window_label", "window_months"])["actual_total"]
        .transform("sum")
    )
    recommendations["actual_share_percent"] = (
        100 * recommendations["actual_total"] / total_actual.replace(0, np.nan)
    )

    def recommendation(row):
        population = row["forecast_population"]
        model_name = row["model"]
        wmape_value = row["pooled_wmape_percent"]

        if population == PREDICTABLE_FORECAST_POPULATION:
            if wmape_value < 70:
                return f"Use this as the primary modelable population; {model_name} meets the 70 percent target."
            if wmape_value < 100:
                return f"Keep improving this population; {model_name} beats zero but is above target."
            return "Known-history SKUs are not yet beating zero; investigate item lifecycle, supersession, and stockout/availability signals."

        if population == COLD_START_MISSING_POPULATION:
            if model_name == "Aggregate Allocation":
                return "Cold-start should be handled by descriptor/category allocation; enrich with supersession, launch date, and family/subfamily mappings."
            return "Do not judge this as a SKU-history forecast; use a separate cold-start policy or descriptor allocation."

        if population == KNOWN_NO_TRAIN_DEMAND_POPULATION:
            return "Treat as activation risk: default to zero unless descriptor/category allocation or lifecycle signals justify demand."

        return "Review separately from the combined portfolio score."

    recommendations["agent"] = "Population Strategy Agent"
    recommendations["recommendation"] = recommendations.apply(recommendation, axis=1)
    return recommendations.sort_values(["window_months", "forecast_population"])

data_quality_agent_summary, data_quality_agent_segments = run_data_quality_agent(
    lumpy_model_data,
    backtest_splits,
)
benchmark_agent_summary = run_benchmark_agent(backtest_forecasts)
segment_strategy_agent_recommendations = run_segment_strategy_agent(
    best_model_by_segment,
    data_quality_agent_segments,
)
signal_inventory_agent_report = run_signal_inventory_agent(lumpy_model_data)
lifecycle_signal_agent_report = run_lifecycle_signal_agent(lumpy_model_data)
results_critic_agent_report = run_results_critic_agent(
    model_summary,
    benchmark_agent_summary,
    data_quality_agent_summary,
    population_model_summary,
)
aggregate_allocation_agent_report = run_aggregate_allocation_agent(
    benchmark_agent_summary,
    tuning_results_all,
    signal_inventory_agent_report,
    monthly_total_model_summary,
)
monthly_total_agent_report = run_monthly_total_agent(
    monthly_total_model_summary,
    model_summary,
)
population_strategy_agent_report = run_population_strategy_agent(
    best_model_by_population,
    population_model_summary,
)
sku_horizon_agent_report = run_sku_horizon_agent(
    sku_horizon_model_summary,
    model_summary,
)
metric_decision_agent_report = run_metric_decision_agent(
    metric_suite_model_summary,
    sku_horizon_model_summary,
)
lag_comparison_agent_report = run_lag_comparison_agent(
    metric_suite_model_summary,
    sku_horizon_model_summary,
)
feature_selection_agent_report = run_feature_selection_agent(
    feature_inventory,
    hurdle_feature_metadata,
    tuning_results_all,
)
feature_overfit_agent_report = run_feature_overfit_agent(
    tuning_results_all,
    model_summary,
    hurdle_feature_metadata,
)
occurrence_gate_agent_report = run_occurrence_gate_agent(backtest_forecasts)
hybrid_impact_agent_report = run_hybrid_impact_agent(
    model_summary,
    known_history_model_summary,
    occurrence_gate_agent_report,
)

display(data_quality_agent_summary)
display(benchmark_agent_summary)
display(monthly_total_agent_report)
display(population_strategy_agent_report)
display(sku_horizon_agent_report)
display(metric_decision_agent_report)
display(lag_comparison_agent_report)
display(feature_selection_agent_report)
display(feature_overfit_agent_report)
display(occurrence_gate_agent_report)
display(hybrid_impact_agent_report)
display(lifecycle_signal_agent_report)
display(segment_strategy_agent_recommendations)
display(results_critic_agent_report)
display(aggregate_allocation_agent_report)
display(signal_inventory_agent_report.head(50))


## Feature And Segment Views


In [ ]:
feature_fold_column = "global_fold_id" if "global_fold_id" in hurdle_feature_metadata.columns else "fold_id"

hurdle_feature_group_columns = [
    "window_label",
    "window_months",
    "feature",
    "is_external_feature",
    "is_sku_train_stat",
]

if "is_rf_internal_history_feature" in hurdle_feature_metadata.columns:
    hurdle_feature_group_columns.append("is_rf_internal_history_feature")
if "feature_profile" in hurdle_feature_metadata.columns:
    hurdle_feature_group_columns.append("feature_profile")
if "feature_family" in hurdle_feature_metadata.columns:
    hurdle_feature_group_columns.append("feature_family")

hurdle_feature_summary = (
    hurdle_feature_metadata
    .groupby(hurdle_feature_group_columns, as_index=False)
    .agg(folds_used=(feature_fold_column, "nunique"))
    .sort_values(
        ["window_months", "is_external_feature", "is_sku_train_stat", "feature"],
        ascending=[True, False, False, True],
    )
)

display(hurdle_feature_summary)


## Visuals


In [ ]:
for window_label, summary in model_summary.groupby("window_label", sort=False):
    summary = summary.sort_values("pooled_wmape_percent")
    actual_total = summary["actual_total"].max()

    plt.figure(figsize=(10, 5))
    plt.bar(summary["model"], summary["pooled_wmape_percent"])
    plt.axhline(70, color="orange", linestyle="--", linewidth=2, label="70 percent target")
    plt.axhline(50, color="green", linestyle="--", linewidth=2, label="50 percent stretch")
    plt.title(f"Lumpy Demand Backtest WMAPE - {window_label} | actual total {actual_total:,.0f}")
    plt.ylabel("Pooled WMAPE percent")
    plt.xticks(rotation=20, ha="right")
    plt.legend()
    plt.grid(axis="y", alpha=0.3)
    plt.show()


def monthly_actual_series(monthly_frame):
    actual = (
        monthly_frame
        .groupby(DATE_COLUMN, as_index=False)
        .agg(actual=("actual", "first"))
        .sort_values(DATE_COLUMN)
    )
    actual[DATE_COLUMN] = pd.to_datetime(actual[DATE_COLUMN])
    return actual


def priority_forecast_models(window_label, latest_monthly):
    available_models = latest_monthly["model"].dropna().unique().tolist()
    model_candidates = []

    if "best_model_by_window" in globals():
        best_rows = best_model_by_window.loc[
            best_model_by_window["window_label"].eq(window_label),
            "model",
        ].dropna().tolist()
        model_candidates.extend(best_rows)

    model_candidates.extend(
        [
            "Segment Winner Blend",
            "Empirical Bayes Allocation",
            "Occurrence Gated Allocation",
            "Aggregate Allocation",
            "Hurdle Random Forest",
            "Direct Demand Random Forest",
            "TSB",
            "Zero Forecast",
        ]
    )

    priority_models = []
    for model_name in model_candidates:
        if model_name in available_models and model_name not in priority_models:
            priority_models.append(model_name)

    return priority_models or available_models


def plot_forecasts_with_actual(monthly_frame, title, model_names=None):
    plot_data = monthly_frame.copy()
    plot_data[DATE_COLUMN] = pd.to_datetime(plot_data[DATE_COLUMN])

    if model_names is not None:
        plot_data = plot_data.loc[plot_data["model"].isin(model_names)].copy()

    if plot_data.empty:
        print(f"No forecast rows available for: {title}")
        return

    actual = monthly_actual_series(plot_data)

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(
        actual[DATE_COLUMN],
        actual["actual"],
        width=20,
        color="black",
        alpha=0.12,
        label="Actual demand",
    )
    ax.plot(
        actual[DATE_COLUMN],
        actual["actual"],
        label="Actual",
        linewidth=3,
        color="black",
        marker="o",
    )

    for model_name, model_data in plot_data.groupby("model", sort=False):
        model_data = model_data.sort_values(DATE_COLUMN)
        ax.plot(
            model_data[DATE_COLUMN],
            model_data["forecast"],
            label=f"{model_name} forecast",
            linestyle="--",
            linewidth=2,
        )

    ax.set_title(title)
    ax.set_xlabel("Month")
    ax.set_ylabel("Demand")
    ax.legend(ncol=2)
    ax.grid(alpha=0.3)
    plt.show()


def plot_rolling_metric_with_actual(monthly_frame, metric_column, title, ylabel):
    plot_data = monthly_frame.copy()
    plot_data[DATE_COLUMN] = pd.to_datetime(plot_data[DATE_COLUMN])
    actual = monthly_actual_series(plot_data)

    fig, ax = plt.subplots(figsize=(14, 6))
    ax2 = ax.twinx()
    ax2.bar(
        actual[DATE_COLUMN],
        actual["actual"],
        width=20,
        color="black",
        alpha=0.08,
        label="Actual demand",
    )
    ax2.set_ylabel("Actual demand")

    for model_name, model_data in plot_data.groupby("model", sort=False):
        model_data = model_data.sort_values(DATE_COLUMN)
        ax.plot(
            model_data[DATE_COLUMN],
            model_data[metric_column],
            label=model_name,
            linewidth=2,
        )

    ax.axhline(70, color="orange", linestyle="--", linewidth=2, label="70 percent target")
    ax.axhline(50, color="green", linestyle="--", linewidth=2, label="50 percent stretch")
    ax.set_title(title)
    ax.set_xlabel("Month")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)

    handles, labels = ax.get_legend_handles_labels()
    actual_handles, actual_labels = ax2.get_legend_handles_labels()
    ax.legend(handles + actual_handles, labels + actual_labels, ncol=2)
    plt.show()


for window_label, window_monthly in monthly_results.groupby("window_label", sort=False):
    latest_fold_id = window_monthly["fold_id"].max()
    latest_monthly = window_monthly.loc[window_monthly["fold_id"].eq(latest_fold_id)].copy()
    focused_models = priority_forecast_models(window_label, latest_monthly)

    plot_forecasts_with_actual(
        latest_monthly,
        title=f"Latest Fold Lumpy Demand Forecasts vs Actual - {window_label}, fold {latest_fold_id}",
        model_names=focused_models,
    )

    plot_rolling_metric_with_actual(
        latest_monthly,
        metric_column="rolling_3m_wmape_percent",
        title=f"Latest Fold 3-Month Rolling Pooled WMAPE With Actual Demand - {window_label}, fold {latest_fold_id}",
        ylabel="sum(abs error) / sum(actual), trailing 3 months",
    )

    plot_rolling_metric_with_actual(
        latest_monthly,
        metric_column="rolling_3m_average_monthly_wmape_percent",
        title=f"Latest Fold 3-Month Rolling Average Monthly WMAPE With Actual Demand - {window_label}, fold {latest_fold_id}",
        ylabel="average monthly WMAPE, trailing 3 months",
    )



for window_label, window_population in population_model_summary.groupby("window_label", sort=False):
    plot_frame = window_population.copy()
    plot_frame["population_model"] = (
        plot_frame["forecast_population"] + " | " + plot_frame["model"]
    )
    plot_frame = plot_frame.sort_values(
        ["forecast_population", "population_selection_score", "absolute_error"]
    )

    plt.figure(figsize=(14, 6))
    plt.bar(plot_frame["population_model"], plot_frame["pooled_wmape_percent"])
    plt.axhline(70, color="orange", linestyle="--", linewidth=2, label="70 percent target")
    plt.axhline(50, color="green", linestyle="--", linewidth=2, label="50 percent stretch")
    plt.title(f"WMAPE By Forecast Population - {window_label}")
    plt.ylabel("Pooled WMAPE percent")
    plt.xticks(rotation=35, ha="right")
    plt.legend()
    plt.grid(axis="y", alpha=0.3)
    plt.show()



for window_label, window_total_summary in monthly_total_model_summary.groupby("window_label", sort=False):
    plot_frame = window_total_summary.sort_values("post_calibrated_monthly_total_wmape_percent")

    plt.figure(figsize=(12, 5))
    plt.bar(plot_frame["model"], plot_frame["post_calibrated_monthly_total_wmape_percent"])
    plt.axhline(70, color="orange", linestyle="--", linewidth=2, label="70 percent target")
    plt.axhline(50, color="green", linestyle="--", linewidth=2, label="50 percent stretch")
    plt.title(f"Monthly Total Forecast WMAPE - {window_label}")
    plt.ylabel("Monthly total WMAPE percent")
    plt.xticks(rotation=25, ha="right")
    plt.legend()
    plt.grid(axis="y", alpha=0.3)
    plt.show()


## Export Results


In [ ]:
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLE_DIRECTORY.mkdir(parents=True, exist_ok=True)

output_paths = {
    "backtest_forecasts": OUTPUT_TABLE_DIRECTORY / "lumpy_backtest_forecasts.csv",
    "evaluation_forecasts": OUTPUT_TABLE_DIRECTORY / "lumpy_evaluation_forecasts_with_blends.csv",
    "segment_winner_forecasts": OUTPUT_TABLE_DIRECTORY / "lumpy_segment_winner_forecasts.csv",
    "population_winner_forecasts": OUTPUT_TABLE_DIRECTORY / "lumpy_population_winner_forecasts.csv",
    "fold_model_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_fold_model_summary.csv",
    "model_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_model_summary.csv",
    "best_model_by_window": OUTPUT_TABLE_DIRECTORY / "lumpy_best_model_by_window.csv",
    "monthly_results": OUTPUT_TABLE_DIRECTORY / "lumpy_monthly_rolling_wmape.csv",
    "monthly_total_results": OUTPUT_TABLE_DIRECTORY / "lumpy_monthly_total_results.csv",
    "monthly_total_fold_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_monthly_total_fold_summary.csv",
    "monthly_total_model_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_monthly_total_model_summary.csv",
    "sku_horizon_results": OUTPUT_TABLE_DIRECTORY / "lumpy_sku_horizon_results.csv",
    "sku_horizon_fold_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_sku_horizon_fold_summary.csv",
    "sku_horizon_model_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_sku_horizon_model_summary.csv",
    "sku_horizon_population_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_sku_horizon_population_summary.csv",
    "sku_horizon_segment_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_sku_horizon_segment_summary.csv",
    "best_sku_horizon_model_by_window": OUTPUT_TABLE_DIRECTORY / "lumpy_best_sku_horizon_model_by_window.csv",
    "metric_suite_fold_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_metric_suite_fold_summary.csv",
    "metric_suite_model_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_metric_suite_model_summary.csv",
    "metric_suite_population_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_metric_suite_population_summary.csv",
    "monthly_total_population_results": OUTPUT_TABLE_DIRECTORY / "lumpy_monthly_total_population_results.csv",
    "monthly_total_population_fold_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_monthly_total_population_fold_summary.csv",
    "monthly_total_population_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_monthly_total_population_summary.csv",
    "segment_fold_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_segment_fold_summary.csv",
    "segment_model_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_segment_model_summary.csv",
    "best_model_by_segment": OUTPUT_TABLE_DIRECTORY / "lumpy_best_model_by_segment.csv",
    "segment_winner_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_segment_winner_summary.csv",
    "population_fold_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_population_fold_summary.csv",
    "population_model_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_population_model_summary.csv",
    "best_model_by_population": OUTPUT_TABLE_DIRECTORY / "lumpy_best_model_by_population.csv",
    "population_winner_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_population_winner_summary.csv",
    "evaluation_population_fold_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_evaluation_population_fold_summary.csv",
    "evaluation_population_model_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_evaluation_population_model_summary.csv",
    "known_history_model_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_known_history_model_summary.csv",
    "best_known_history_by_window": OUTPUT_TABLE_DIRECTORY / "lumpy_best_known_history_by_window.csv",
    "tuning_results": OUTPUT_TABLE_DIRECTORY / "lumpy_tuning_results.csv",
    "hurdle_feature_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_hurdle_feature_summary.csv",
    "feature_inventory": OUTPUT_TABLE_DIRECTORY / "lumpy_feature_inventory.csv",
    "external_selection_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_selected_external_features.csv",
    "data_quality_agent_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_agent_data_quality_summary.csv",
    "data_quality_agent_segments": OUTPUT_TABLE_DIRECTORY / "lumpy_agent_data_quality_segments.csv",
    "benchmark_agent_summary": OUTPUT_TABLE_DIRECTORY / "lumpy_agent_benchmark_summary.csv",
    "monthly_total_agent_report": OUTPUT_TABLE_DIRECTORY / "lumpy_agent_monthly_total_report.csv",
    "sku_horizon_agent_report": OUTPUT_TABLE_DIRECTORY / "lumpy_agent_sku_horizon_report.csv",
    "metric_decision_agent_report": OUTPUT_TABLE_DIRECTORY / "lumpy_agent_metric_decision_report.csv",
    "lag_comparison_agent_report": OUTPUT_TABLE_DIRECTORY / "lumpy_agent_lag_comparison_report.csv",
    "segment_strategy_agent_recommendations": OUTPUT_TABLE_DIRECTORY / "lumpy_agent_segment_strategy_recommendations.csv",
    "population_strategy_agent_report": OUTPUT_TABLE_DIRECTORY / "lumpy_agent_population_strategy_report.csv",
    "feature_selection_agent_report": OUTPUT_TABLE_DIRECTORY / "lumpy_agent_feature_selection_report.csv",
    "feature_overfit_agent_report": OUTPUT_TABLE_DIRECTORY / "lumpy_agent_feature_overfit_report.csv",
    "occurrence_gate_agent_report": OUTPUT_TABLE_DIRECTORY / "lumpy_agent_occurrence_gate_report.csv",
    "hybrid_impact_agent_report": OUTPUT_TABLE_DIRECTORY / "lumpy_agent_hybrid_impact_report.csv",
    "lifecycle_signal_agent_report": OUTPUT_TABLE_DIRECTORY / "lumpy_agent_lifecycle_signal_report.csv",
    "results_critic_agent_report": OUTPUT_TABLE_DIRECTORY / "lumpy_agent_results_critic_report.csv",
    "signal_inventory_agent_report": OUTPUT_TABLE_DIRECTORY / "lumpy_agent_signal_inventory_report.csv",
    "aggregate_allocation_agent_report": OUTPUT_TABLE_DIRECTORY / "lumpy_agent_aggregate_allocation_report.csv",
}

backtest_forecasts.to_csv(output_paths["backtest_forecasts"], index=False)
evaluation_forecasts.to_csv(output_paths["evaluation_forecasts"], index=False)
segment_winner_forecasts.to_csv(output_paths["segment_winner_forecasts"], index=False)
population_winner_forecasts.to_csv(output_paths["population_winner_forecasts"], index=False)
fold_model_summary.to_csv(output_paths["fold_model_summary"], index=False)
model_summary.to_csv(output_paths["model_summary"], index=False)
best_model_by_window.to_csv(output_paths["best_model_by_window"], index=False)
monthly_results.to_csv(output_paths["monthly_results"], index=False)
monthly_total_results.to_csv(output_paths["monthly_total_results"], index=False)
monthly_total_fold_summary.to_csv(output_paths["monthly_total_fold_summary"], index=False)
monthly_total_model_summary.to_csv(output_paths["monthly_total_model_summary"], index=False)
sku_horizon_results.to_csv(output_paths["sku_horizon_results"], index=False)
sku_horizon_fold_summary.to_csv(output_paths["sku_horizon_fold_summary"], index=False)
sku_horizon_model_summary.to_csv(output_paths["sku_horizon_model_summary"], index=False)
sku_horizon_population_summary.to_csv(output_paths["sku_horizon_population_summary"], index=False)
sku_horizon_segment_summary.to_csv(output_paths["sku_horizon_segment_summary"], index=False)
best_sku_horizon_model_by_window.to_csv(output_paths["best_sku_horizon_model_by_window"], index=False)
metric_suite_fold_summary.to_csv(output_paths["metric_suite_fold_summary"], index=False)
metric_suite_model_summary.to_csv(output_paths["metric_suite_model_summary"], index=False)
metric_suite_population_summary.to_csv(output_paths["metric_suite_population_summary"], index=False)
monthly_total_population_results.to_csv(output_paths["monthly_total_population_results"], index=False)
monthly_total_population_fold_summary.to_csv(output_paths["monthly_total_population_fold_summary"], index=False)
monthly_total_population_summary.to_csv(output_paths["monthly_total_population_summary"], index=False)
segment_fold_summary.to_csv(output_paths["segment_fold_summary"], index=False)
segment_model_summary.to_csv(output_paths["segment_model_summary"], index=False)
best_model_by_segment.to_csv(output_paths["best_model_by_segment"], index=False)
segment_winner_summary.to_csv(output_paths["segment_winner_summary"], index=False)
population_fold_summary.to_csv(output_paths["population_fold_summary"], index=False)
population_model_summary.to_csv(output_paths["population_model_summary"], index=False)
best_model_by_population.to_csv(output_paths["best_model_by_population"], index=False)
population_winner_summary.to_csv(output_paths["population_winner_summary"], index=False)
evaluation_population_fold_summary.to_csv(output_paths["evaluation_population_fold_summary"], index=False)
evaluation_population_model_summary.to_csv(output_paths["evaluation_population_model_summary"], index=False)
known_history_model_summary.to_csv(output_paths["known_history_model_summary"], index=False)
best_known_history_by_window.to_csv(output_paths["best_known_history_by_window"], index=False)
tuning_results_all.to_csv(output_paths["tuning_results"], index=False)
hurdle_feature_summary.to_csv(output_paths["hurdle_feature_summary"], index=False)
feature_inventory.to_csv(output_paths["feature_inventory"], index=False)
external_selection_summary.to_csv(output_paths["external_selection_summary"], index=False)
data_quality_agent_summary.to_csv(output_paths["data_quality_agent_summary"], index=False)
data_quality_agent_segments.to_csv(output_paths["data_quality_agent_segments"], index=False)
benchmark_agent_summary.to_csv(output_paths["benchmark_agent_summary"], index=False)
monthly_total_agent_report.to_csv(output_paths["monthly_total_agent_report"], index=False)
sku_horizon_agent_report.to_csv(output_paths["sku_horizon_agent_report"], index=False)
metric_decision_agent_report.to_csv(output_paths["metric_decision_agent_report"], index=False)
lag_comparison_agent_report.to_csv(output_paths["lag_comparison_agent_report"], index=False)
segment_strategy_agent_recommendations.to_csv(output_paths["segment_strategy_agent_recommendations"], index=False)
population_strategy_agent_report.to_csv(output_paths["population_strategy_agent_report"], index=False)
feature_selection_agent_report.to_csv(output_paths["feature_selection_agent_report"], index=False)
feature_overfit_agent_report.to_csv(output_paths["feature_overfit_agent_report"], index=False)
occurrence_gate_agent_report.to_csv(output_paths["occurrence_gate_agent_report"], index=False)
hybrid_impact_agent_report.to_csv(output_paths["hybrid_impact_agent_report"], index=False)
lifecycle_signal_agent_report.to_csv(output_paths["lifecycle_signal_agent_report"], index=False)
results_critic_agent_report.to_csv(output_paths["results_critic_agent_report"], index=False)
signal_inventory_agent_report.to_csv(output_paths["signal_inventory_agent_report"], index=False)
aggregate_allocation_agent_report.to_csv(output_paths["aggregate_allocation_agent_report"], index=False)

display(pd.DataFrame(
    [{"artifact": key, "path": str(path)} for key, path in output_paths.items()]
))


## Reading The Result

Start with `model_summary`, then `best_model_by_window`. For individual SKU quantity over the whole forecast horizon, use `sku_horizon_model_summary`, `best_sku_horizon_model_by_window`, and `sku_horizon_agent_report`; these separate the horizon-level SKU planning question from SKU-month timing accuracy. For lumpy-demand decision scoring, use `metric_suite_model_summary` and `metric_decision_agent_report` to compare WMAPE, MASE-style scaled error, fill-rate proxy, under/over forecast quantity, and cost-weighted inventory utility. `FORECAST_LAG_MONTH_OPTIONS` compares alternative lead-time gaps against the requested 3-month benchmark; use `lag_comparison_agent_report` to see whether another lag materially improves results. For the cleanest read on the modelable demand, use `known_history_model_summary` and `best_known_history_by_window` before judging the combined portfolio.

This run makes selected external features available: `EXTERNAL_FEATURE_MODE = "selected"`. The Hurdle RF feature gate only uses external features when they improve validation WMAPE inside the training window.

The required view is `required_18_month`: this is the original 18-month forecast window with a 3-month lag and 3-month rolling WMAPE. Use `monthly_total_model_summary` to judge whether total demand is forecastable before judging SKU allocation.

The recommendation view is `diagnostic_9_month`. This shorter lead-time experiment checks whether a more operationally realistic 9-month recommendation horizon is materially cleaner.

The model table now includes the intermittent models, `Aggregate Allocation`, the occurrence-gated `Hurdle Random Forest`, the full-regression challenger `Direct Demand Random Forest`, the aggregate-plus-gate hybrids `Empirical Bayes Allocation` and `Occurrence Gated Allocation`, plus four internal baselines: `Zero Forecast`, `SKU Train Mean`, `Recent Mean`, and `Last Nonzero`.

Croston/SBA and TSB use their demand history internally. The Hurdle Random Forest is still a regression model, but it now has a tuned occurrence threshold so low-probability SKU-months can be set to zero before scoring. The `Direct Demand Random Forest` tests the pure regression framing by predicting SKU-month demand directly with a tuned forecast floor. Both models use compact model-owned `rf_*` history features; the shared dataset no longer creates global `demand_lag_*` or `demand_roll_*` columns.

Use `feature_inventory` to confirm this: `is_global_demand_lag_feature` should be `False` for every RF feature, while `is_rf_internal_history_feature` marks the model-owned history fields.

Use `tuning_results_all` to see which hyperparameters and calibration multipliers were selected for each fold. If a row says `short_history_no_validation`, there was not enough training history to tune or calibrate honestly, so the notebook used conservative defaults.

Use `population_model_summary`, `best_model_by_population`, `sku_horizon_population_summary`, and `sku_horizon_segment_summary` to separate known SKUs with demand history, cold-start/missing SKUs, and known SKUs with no train demand. Then use `segment_model_summary` and `best_model_by_segment` to see whether high-volume, rare-lumpy, and no-train-demand SKUs behave differently. `Segment Winner Blend` is a diagnostic portfolio that uses the best backtested model for each SKU segment, so it is useful for recommendation design but should not be treated as a separately trained forecasting model.


## Forecast Agents

The notebook now runs local diagnostic agents after scoring:

- `data_quality_agent_summary` and `data_quality_agent_segments` explain whether train/test history is usable.
- `benchmark_agent_summary` compares post-calibrated, pre-calibration, and operational-floor forecasts against zero.
- `monthly_total_agent_report` checks whether monthly total demand is forecastable separately from SKU-month allocation.
- `population_strategy_agent_report` gives the recommended action by forecast population, so cold-start SKUs do not hide known-SKU performance.
- `segment_strategy_agent_recommendations` gives a recommended action per SKU segment.
- `results_critic_agent_report` writes the short readout after each run.
- `signal_inventory_agent_report` inventories available non-target fields for future signal discovery.
- `feature_selection_agent_report` explains which feature families were selected or rejected.
- `feature_overfit_agent_report` compares Hurdle RF validation WMAPE with test WMAPE to flag overfit.
- `occurrence_gate_agent_report` checks whether each model is missing positive SKU-months or creating too many false positive zero-demand SKU-month forecasts.
- `hybrid_impact_agent_report` compares the new hybrid allocation models against zero, Aggregate Allocation, Hurdle RF, and Direct RF at SKU level and known-history SKU level.
- `lifecycle_signal_agent_report` looks for stockout, availability, supersession, launch, active/inactive, and lifecycle columns that could explain zero demand months.

The candidate model `Aggregate Allocation` forecasts total lumpy monthly demand first, then allocates it back to descriptor groups and SKUs. It now tests a two-stage descriptor allocator: monthly total -> family/subfamily group -> SKU probability/recency share. The new hybrid models go one step further by allocating the monthly total only to selected SKU-months: `Empirical Bayes Allocation` uses pooled family/subfamily occurrence and positive-size estimates, while `Occurrence Gated Allocation` uses RF occurrence probabilities and size scores. Use `hybrid_impact_agent_report` to see which idea had the largest SKU-level impact.
